In [3]:
# =============================================================================
# build_rj_sample.ipynb
#
# Purpose: Build the Rajasthan NFHS-4 cluster-level treatment probability file
#          and unmatched GP priority list from scratch.
#
# Inputs (read-only):
#   data/rajasthan gp reservations/sp_2005_2010_manually_reviewed.csv
#   data/All India Village to GP LGD codes.csv
#   outputs/monte carlo simulation rajasthan/rajasthan_mc_nfhs4_constrained_valid.csv
#
# Outputs (written to outputs/final_rj_sample/):
#   cluster_treatment_probs_rj.csv
#   unmatched_gps_for_manual_review.xlsx
#   cluster_gp_res_long.csv  (intermediate, saved for reproducibility)
#   gp_reservation_history.csv (intermediate, saved for reproducibility)
# =============================================================================

import pandas as pd
import numpy as np
import ast
import re
import json
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT   = Path("..")
DATA_DIR    = REPO_ROOT / "data"
MC_DIR      = REPO_ROOT / "outputs" / "monte carlo simulation rajasthan"
OUTPUT_DIR  = REPO_ROOT / "outputs" / "final_rj_sample"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESERVATION_PATH = DATA_DIR / "rajasthan gp reservations" / "sp_2005_2010_manually_reviewed.csv"
LGD_PATH         = DATA_DIR / "All India Village to GP LGD codes.csv"
MC_PATH          = MC_DIR  / "rajasthan_mc_nfhs4_constrained_valid.csv"

print("Output directory:", OUTPUT_DIR.resolve())
print("Reservation file exists:", RESERVATION_PATH.exists())
print("LGD file exists:        ", LGD_PATH.exists())
print("MC file exists:         ", MC_PATH.exists())

Output directory: /Users/sonalideliwala/Documents/GitHub/exemplar/outputs/final_rj_sample
Reservation file exists: True
LGD file exists:         True
MC file exists:          True


In [4]:
# =============================================================================
# CELL 2 — Helper functions
# =============================================================================

def normalize_name(x):
    """Lowercase, strip punctuation and whitespace for fuzzy matching."""
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", "", x)
    return x

def is_women_reserved(x):
    """Return 1 if reservation code ends in W (women's quota), else 0."""
    if pd.isna(x):
        return 0
    x = str(x).upper().strip()
    x = re.sub(r"[^A-Z0-9]", "", x)
    return int(x.endswith("W"))

def parse_gp_dist(val):
    """Parse gp_distribution column from MC file into a dict."""
    if pd.isna(val):
        return {}
    if isinstance(val, dict):
        return val
    s = str(val).strip()
    if s == "":
        return {}
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return {}

def normalize_lgd_code(x):
    """Strip .0 suffix and whitespace from LGD codes."""
    return str(x).replace(".0", "").strip()

# District name aliases: LGD current name -> historical reservation file name
DISTRICT_ALIASES = {
    "chittaurgarh":   "chittorgarh",
    "dhaulpur":       "dholpur",
    "jalor":          "jalore",
    "jhunjhunun":     "jhunjhunu",
    "anupgarh":       "ganganagar",
    "balotra":        "barmer",
    "beawar":         "ajmer",
    "deeg":           "bharatpur",
    "didwanakuchaman":"nagaur",
    "dudu":           "jaipur",
    "gangapurcity":   "sawaimadhopur",
    "jaipurgramin":   "jaipur",
    "jodhpurgramin":  "jodhpur",
    "kekri":          "ajmer",
    "khairthaltijara":"alwar",
    "kotputlibehror": "jaipur",
    "neemkathana":    "sikar",
    "phalodi":        "jodhpur",
    "salumbar":       "udaipur",
    "sanchore":       "jalore",
    "shahpura":       "bhilwara",
}

def normalize_district_historical(x):
    """Normalize district name and apply historical aliases."""
    x_norm = normalize_name(x)
    return DISTRICT_ALIASES.get(x_norm, x_norm)

DOSE_LABEL_MAP = {0: "never", 1: "once", 2: "twice"}

print("Helper functions defined.")
print(f"District aliases: {len(DISTRICT_ALIASES)}")
print("\nQuick tests:")
print(f"  normalize_name('Gram Panchayat!'): '{normalize_name('Gram Panchayat!')}'")
print(f"  is_women_reserved('GENW'): {is_women_reserved('GENW')}")
print(f"  is_women_reserved('GEN'):  {is_women_reserved('GEN')}")
print(f"  normalize_district_historical('Jhunjhunun'): "
      f"'{normalize_district_historical('Jhunjhunun')}'")
print(f"  normalize_district_historical('Beawar'): "
      f"'{normalize_district_historical('Beawar')}'")

Helper functions defined.
District aliases: 21

Quick tests:
  normalize_name('Gram Panchayat!'): 'grampanchayat'
  is_women_reserved('GENW'): 1
  is_women_reserved('GEN'):  0
  normalize_district_historical('Jhunjhunun'): 'jhunjhunu'
  normalize_district_historical('Beawar'): 'ajmer'


In [5]:
# =============================================================================
# CELL 3 — Load and clean reservation data
# =============================================================================

reservations = pd.read_csv(RESERVATION_PATH)
print(f"Raw reservation rows: {len(reservations)}")
print(f"Columns: {reservations.columns.tolist()}")

# ── Filter nuked rows ─────────────────────────────────────────────────────────
if "nuke" in reservations.columns:
    nuke_norm = reservations["nuke"].astype(str).str.strip().str.lower()
    res_clean = reservations[
        ~nuke_norm.isin(["1", "true", "yes", "y", "drop", "nuke"])
    ].copy()
else:
    res_clean = reservations.copy()

print(f"Rows after nuke filter: {len(res_clean)}")

# ── Compute treatment variables ───────────────────────────────────────────────
res_clean["reserved_women_2005"] = res_clean["reservation_2005"].apply(is_women_reserved)
res_clean["reserved_women_2010"] = res_clean["reservation_2010"].apply(is_women_reserved)
res_clean["reservation_dose_n"]  = (
    res_clean["reserved_women_2005"] + res_clean["reserved_women_2010"]
)
res_clean["reservation_dose"] = res_clean["reservation_dose_n"].map(DOSE_LABEL_MAP)

# ── Build normalized match keys ───────────────────────────────────────────────
# Use 2010 names where available, fall back to 2005
res_clean["gp_norm"] = (
    res_clean["gp_new_2010"]
    .fillna(res_clean["gp_new_2005"])
    .apply(normalize_name)
)
res_clean["district_norm"] = (
    res_clean["dist_name_new_2010"]
    .fillna(res_clean["dist_name_new_2005"])
    .apply(normalize_name)
)
res_clean["samiti_norm"] = (
    res_clean["samiti_name_new_2010"]
    .fillna(res_clean["samiti_name_new_2005"])
    .apply(normalize_name)
)

# ── Diagnostics ───────────────────────────────────────────────────────────────
print(f"\nReservation dose distribution:")
print(res_clean["reservation_dose"].value_counts(dropna=False))

print(f"\nYear-specific breakdown:")
print(f"  Reserved in 2005 only (1,0): "
      f"{((res_clean['reserved_women_2005']==1) & (res_clean['reserved_women_2010']==0)).sum()}")
print(f"  Reserved in 2010 only (0,1): "
      f"{((res_clean['reserved_women_2005']==0) & (res_clean['reserved_women_2010']==1)).sum()}")
print(f"  Reserved both (1,1):         "
      f"{((res_clean['reserved_women_2005']==1) & (res_clean['reserved_women_2010']==1)).sum()}")
print(f"  Reserved neither (0,0):      "
      f"{((res_clean['reserved_women_2005']==0) & (res_clean['reserved_women_2010']==0)).sum()}")

print(f"\nUnique normalized GP names:      {res_clean['gp_norm'].nunique()}")
print(f"Unique normalized districts:     {res_clean['district_norm'].nunique()}")
print(f"Unique normalized samitis:       {res_clean['samiti_norm'].nunique()}")

print(f"\nSample cleaned rows:")
print(res_clean[[
    "dist_name_new_2010", "samiti_name_new_2010", "gp_new_2010",
    "reservation_2005", "reservation_2010",
    "reserved_women_2005", "reserved_women_2010", "reservation_dose"
]].head(5).to_string(index=False))

Raw reservation rows: 9860
Columns: ['sl_no_2005', 'dist_name_2005', 'samiti_name_2005', 'gp_2005', 'reservation_2005', 'name_2005', 'sex_2005', 'category_2005', 'gp_new_2005', 'dist_name_new_2005', 'samiti_name_new_2005', 'dist_2010_2005', 'key_2005', 'sl_no_2010', 'dist_name_2010', 'samiti_name_2010', 'gp_2010', 'reservation_2010', 'name_2010', 'sex_2010', 'category_2010', 'gp_new_2010', 'dist_name_new_2010', 'samiti_name_new_2010', 'key_2010', 'dist', 'nuke']
Rows after nuke filter: 9860

Reservation dose distribution:
reservation_dose
once     4857
never    3459
twice    1544
Name: count, dtype: int64

Year-specific breakdown:
  Reserved in 2005 only (1,0): 1694
  Reserved in 2010 only (0,1): 3163
  Reserved both (1,1):         1544
  Reserved neither (0,0):      3459

Unique normalized GP names:      7504
Unique normalized districts:     33
Unique normalized samitis:       230

Sample cleaned rows:
dist_name_new_2010 samiti_name_new_2010 gp_new_2010 reservation_2005 reservation_20

In [6]:
# =============================================================================
# CELL 4 — Load LGD GP lookup and build normalized match keys
# =============================================================================

lgd_raw = pd.read_csv(
    LGD_PATH,
    encoding="latin-1",
    dtype={"Gram Panchayat LGD Code": str}
)
print(f"Raw LGD rows: {len(lgd_raw)}")
print(f"Columns: {lgd_raw.columns.tolist()}")

# ── Filter to Rajasthan ───────────────────────────────────────────────────────
lgd_rj = lgd_raw[
    lgd_raw["State Code"].astype(str).str.strip().isin(["8", "08"])
].copy()
print(f"\nRajasthan LGD rows: {len(lgd_rj)}")

# ── Build one row per unique GP ───────────────────────────────────────────────
lgd_rj["gp_lgd_code"] = lgd_rj["Gram Panchayat LGD Code"].apply(normalize_lgd_code)

gp_lookup = (
    lgd_rj
    .dropna(subset=["gp_lgd_code", "Gram Panchayat Name"])
    .groupby("gp_lgd_code", as_index=False)
    .agg(
        gp_name_lgd    = ("Gram Panchayat Name",  "first"),
        district_lgd   = ("District Name",         "first"),
        subdistrict_lgd= ("Subdistrict Name",      "first"),
        n_villages     = ("Village Census 2011 Code", "nunique")
    )
)

# ── Build normalized match keys ───────────────────────────────────────────────
gp_lookup["gp_norm"]         = gp_lookup["gp_name_lgd"].apply(normalize_name)
gp_lookup["district_norm"]   = gp_lookup["district_lgd"].apply(normalize_name)
gp_lookup["subdistrict_norm"]= gp_lookup["subdistrict_lgd"].apply(normalize_name)
gp_lookup["district_norm_hist"] = gp_lookup["district_lgd"].apply(
    normalize_district_historical
)

print(f"\nUnique Rajasthan GPs in LGD: {len(gp_lookup)}")
print(f"Unique districts:            {gp_lookup['district_norm'].nunique()}")
print(f"Unique subdistricts:         {gp_lookup['subdistrict_norm'].nunique()}")

# ── Check GP name uniqueness within district ──────────────────────────────────
gp_name_district_counts = (
    gp_lookup.groupby(["district_norm_hist", "gp_norm"])
    .size()
    .reset_index(name="n_gps")
)
duplicate_gp_district = gp_name_district_counts[
    gp_name_district_counts["n_gps"] > 1
]
print(f"\nGP names unique within historical district: "
      f"{len(gp_name_district_counts) - len(duplicate_gp_district)} "
      f"({(1 - len(duplicate_gp_district)/len(gp_name_district_counts)):.1%})")
print(f"GP names duplicated within historical district: "
      f"{len(duplicate_gp_district)} "
      f"({len(duplicate_gp_district)/len(gp_name_district_counts):.1%})")

# ── Check GP name uniqueness within district+samiti ───────────────────────────
gp_name_district_samiti_counts = (
    gp_lookup.groupby(["district_norm_hist", "subdistrict_norm", "gp_norm"])
    .size()
    .reset_index(name="n_gps")
)
duplicate_gp_district_samiti = gp_name_district_samiti_counts[
    gp_name_district_samiti_counts["n_gps"] > 1
]
print(f"\nGP names unique within district+samiti: "
      f"{len(gp_name_district_samiti_counts) - len(duplicate_gp_district_samiti)} "
      f"({(1 - len(duplicate_gp_district_samiti)/len(gp_name_district_samiti_counts)):.1%})")

print(f"\nSample GP lookup rows:")
print(gp_lookup[[
    "gp_lgd_code", "gp_name_lgd", "district_lgd",
    "subdistrict_lgd", "gp_norm", "district_norm_hist"
]].head(5).to_string(index=False))

Raw LGD rows: 638847
Columns: ['State Name', 'State Code', 'District Name', 'District Census 2011 Code', 'Subdistrict Name', 'Subdistrict Census 2011 Code', 'Village Name', 'Village Census 2011 Code', 'Village Census 2001 Code', 'Gram Panchayat LGD Code', 'Gram Panchayat Name']

Rajasthan LGD rows: 47971

Unique Rajasthan GPs in LGD: 11199
Unique districts:            50
Unique subdistricts:         417

GP names unique within historical district: 10815 (98.3%)
GP names duplicated within historical district: 183 (1.7%)

GP names unique within district+samiti: 11197 (100.0%)

Sample GP lookup rows:
gp_lgd_code  gp_name_lgd  district_lgd subdistrict_lgd      gp_norm district_norm_hist
     236070         Hoda      Bhilwara      Mandalgarh         hoda           bhilwara
     236071       Genoli      Bhilwara      Mandalgarh       genoli           bhilwara
     236072   Baldarkhan      Bhilwara      Mandalgarh   baldarkhan           bhilwara
     236073 Gajsukhdesar       Bikaner        J

In [7]:
# =============================================================================
# CELL 5 — Match reservation rows to GP LGD codes (staged matching)
#
# Stage A: exact GP name + district + samiti  (unique GP in LGD)
# Stage B: exact GP name + district           (unique GP in LGD)
# Stage C: exact GP name statewide            (unique GP name in all of RJ)
# Stage D: exact GP name + district           (unique dose in reservation data,
#           even if GP name not unique in LGD — relaxed for coverage)
# =============================================================================

res = res_clean.copy()
res["gp_lgd_code"]    = pd.NA
res["gp_name_lgd"]    = pd.NA
res["district_lgd"]   = pd.NA
res["subdistrict_lgd"]= pd.NA
res["match_stage"]    = pd.NA

# ── Prepare LGD candidate sets for each stage ─────────────────────────────────

# Stage A: GPs unique within district_norm_hist + subdistrict_norm
stage_a_candidates = (
    gp_lookup
    .groupby(["district_norm_hist", "subdistrict_norm", "gp_norm"])
    .filter(lambda x: len(x) == 1)
    [["gp_norm", "district_norm_hist", "subdistrict_norm",
      "gp_lgd_code", "gp_name_lgd", "district_lgd", "subdistrict_lgd"]]
    .copy()
)

# Stage B: GPs unique within district_norm_hist
stage_b_candidates = (
    gp_lookup
    .groupby(["district_norm_hist", "gp_norm"])
    .filter(lambda x: len(x) == 1)
    [["gp_norm", "district_norm_hist",
      "gp_lgd_code", "gp_name_lgd", "district_lgd", "subdistrict_lgd"]]
    .copy()
)

# Stage C: GPs unique statewide
stage_c_candidates = (
    gp_lookup
    .groupby(["gp_norm"])
    .filter(lambda x: len(x) == 1)
    [["gp_norm", "gp_lgd_code", "gp_name_lgd",
      "district_lgd", "subdistrict_lgd"]]
    .copy()
)

print(f"Stage A candidates (unique GP+district+samiti): {len(stage_a_candidates)}")
print(f"Stage B candidates (unique GP+district):        {len(stage_b_candidates)}")
print(f"Stage C candidates (unique GP statewide):       {len(stage_c_candidates)}")

# Also add historical district norm to reservation data
res["district_norm_hist"] = res["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)

# ── Stage A ───────────────────────────────────────────────────────────────────
m_a = res.merge(
    stage_a_candidates,
    left_on=["gp_norm", "district_norm_hist", "samiti_norm"],
    right_on=["gp_norm", "district_norm_hist", "subdistrict_norm"],
    how="left",
    suffixes=("", "_a")
)
mask_a = m_a["gp_lgd_code_a"].notna().to_numpy()
res.loc[mask_a, "gp_lgd_code"]    = m_a.loc[mask_a, "gp_lgd_code_a"].values
res.loc[mask_a, "gp_name_lgd"]    = m_a.loc[mask_a, "gp_name_lgd_a"].values
res.loc[mask_a, "district_lgd"]   = m_a.loc[mask_a, "district_lgd_a"].values
res.loc[mask_a, "subdistrict_lgd"]= m_a.loc[mask_a, "subdistrict_lgd_a"].values
res.loc[mask_a, "match_stage"]    = "A_exact_gp_district_samiti"
print(f"\nStage A matched: {mask_a.sum()}")

# ── Stage B ───────────────────────────────────────────────────────────────────
unmatched = res.index[res["gp_lgd_code"].isna()]
m_b = res.loc[unmatched].merge(
    stage_b_candidates,
    left_on=["gp_norm", "district_norm_hist"],
    right_on=["gp_norm", "district_norm_hist"],
    how="left",
    suffixes=("", "_b")
)
mask_b = m_b["gp_lgd_code_b"].notna().to_numpy()
orig_idx_b = unmatched[mask_b]
res.loc[orig_idx_b, "gp_lgd_code"]    = m_b.loc[mask_b, "gp_lgd_code_b"].values
res.loc[orig_idx_b, "gp_name_lgd"]    = m_b.loc[mask_b, "gp_name_lgd_b"].values
res.loc[orig_idx_b, "district_lgd"]   = m_b.loc[mask_b, "district_lgd_b"].values
res.loc[orig_idx_b, "subdistrict_lgd"]= m_b.loc[mask_b, "subdistrict_lgd_b"].values
res.loc[orig_idx_b, "match_stage"]    = "B_exact_gp_district"
print(f"Stage B matched: {mask_b.sum()}")

# ── Stage C ───────────────────────────────────────────────────────────────────
unmatched = res.index[res["gp_lgd_code"].isna()]
m_c = res.loc[unmatched].merge(
    stage_c_candidates,
    on="gp_norm",
    how="left",
    suffixes=("", "_c")
)
mask_c = m_c["gp_lgd_code_c"].notna().to_numpy()
orig_idx_c = unmatched[mask_c]
res.loc[orig_idx_c, "gp_lgd_code"]    = m_c.loc[mask_c, "gp_lgd_code_c"].values
res.loc[orig_idx_c, "gp_name_lgd"]    = m_c.loc[mask_c, "gp_name_lgd_c"].values
res.loc[orig_idx_c, "district_lgd"]   = m_c.loc[mask_c, "district_lgd_c"].values
res.loc[orig_idx_c, "subdistrict_lgd"]= m_c.loc[mask_c, "subdistrict_lgd_c"].values
res.loc[orig_idx_c, "match_stage"]    = "C_unique_gp_name_statewide"
print(f"Stage C matched: {mask_c.sum()}")

print(f"\nMatch stage counts:")
print(res["match_stage"].value_counts(dropna=False))
print(f"\nTotal matched: {res['gp_lgd_code'].notna().sum()} "
      f"({res['gp_lgd_code'].notna().mean():.1%})")
print(f"Total unmatched: {res['gp_lgd_code'].isna().sum()}")

Stage A candidates (unique GP+district+samiti): 11197
Stage B candidates (unique GP+district):        10815
Stage C candidates (unique GP statewide):       9026

Stage A matched: 3138
Stage B matched: 2069
Stage C matched: 388

Match stage counts:
match_stage
<NA>                          4265
A_exact_gp_district_samiti    3138
B_exact_gp_district           2069
C_unique_gp_name_statewide     388
Name: count, dtype: int64

Total matched: 5595 (56.7%)
Total unmatched: 4265


In [8]:
# =============================================================================
# CELL 6 — Build GP-level reservation history, handle conflicts
# =============================================================================

# ── Work only with matched rows ───────────────────────────────────────────────
matched_res = res[res["gp_lgd_code"].notna()].copy()
matched_res["gp_lgd_code"] = matched_res["gp_lgd_code"].apply(normalize_lgd_code)

print(f"Matched reservation rows: {len(matched_res)}")
print(f"Unique GP LGD codes:      {matched_res['gp_lgd_code'].nunique()}")

# ── Check for conflicting dose values per GP LGD code ────────────────────────
conflict_check = (
    matched_res
    .groupby("gp_lgd_code")
    .agg(
        n_rows         = ("gp_lgd_code", "size"),
        n_dose_values  = ("reservation_dose_n", "nunique"),
        dose_values    = ("reservation_dose", lambda x: " | ".join(sorted(set(x.dropna())))),
        gp_name_lgd    = ("gp_name_lgd", "first"),
        district_lgd   = ("district_lgd", "first"),
        subdistrict_lgd= ("subdistrict_lgd", "first"),
        match_stage    = ("match_stage", "first")
    )
    .reset_index()
)

conflict_gp_codes = set(
    conflict_check.loc[conflict_check["n_dose_values"] > 1, "gp_lgd_code"]
)
print(f"\nGP LGD codes with conflicting dose: {len(conflict_gp_codes)}")
print(f"Sample conflicts:")
print(conflict_check[conflict_check["gp_lgd_code"].isin(conflict_gp_codes)][
    ["gp_lgd_code","gp_name_lgd","district_lgd","dose_values","n_rows"]
].head(10).to_string(index=False))

# ── Save conflicts for review ─────────────────────────────────────────────────
conflict_review = matched_res[
    matched_res["gp_lgd_code"].isin(conflict_gp_codes)
].copy()
conflict_review.to_csv(
    OUTPUT_DIR / "gp_reservation_conflicts_review.csv", index=False
)
print(f"\nSaved {len(conflict_review)} conflict rows for review.")

# ── Build GP-level history excluding conflicts ────────────────────────────────
matched_res_noconflict = matched_res[
    ~matched_res["gp_lgd_code"].isin(conflict_gp_codes)
].copy()

gp_res_history = (
    matched_res_noconflict
    .sort_values(["gp_lgd_code", "match_stage"])
    .groupby("gp_lgd_code", as_index=False)
    .agg(
        reserved_women_2005 = ("reserved_women_2005", "first"),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        reservation_dose_n  = ("reservation_dose_n",  "first"),
        reservation_dose    = ("reservation_dose",    "first"),
        gp_name_lgd         = ("gp_name_lgd",         "first"),
        district_lgd        = ("district_lgd",         "first"),
        subdistrict_lgd     = ("subdistrict_lgd",      "first"),
        match_stage         = ("match_stage",          "first")
    )
)

print(f"\nGP reservation history (non-conflicted): {len(gp_res_history)} GPs")
print(f"\nDose distribution:")
print(gp_res_history["reservation_dose"].value_counts(dropna=False))
print(f"\nMatch stage breakdown:")
print(gp_res_history["match_stage"].value_counts())

# ── Stage D: district-only matching ──────────────────────────────────────────
already_matched_codes = set(gp_res_history["gp_lgd_code"])

# Add historical district norm to res_clean (needed for Stage D groupby)
res_clean["district_norm_hist"] = res_clean["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)

# Reservation-side: unique dose per GP name + historical district
res_district_only = (
    res_clean
    .groupby(["gp_norm", "district_norm_hist"])
    .agg(
        n_dose_values       = ("reservation_dose",    "nunique"),
        reservation_dose    = ("reservation_dose",    "first"),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        reservation_dose_n  = ("reservation_dose_n",  "first")
    )
    .reset_index()
)
# Only keep unambiguous matches
res_district_only = res_district_only[
    res_district_only["n_dose_values"] == 1
].drop(columns="n_dose_values")

print(f"\nStage D: unique GP+district pairs in reservation data: "
      f"{len(res_district_only)}")

# LGD-side: GPs not yet matched
gp_lookup_unmatched = gp_lookup[
    ~gp_lookup["gp_lgd_code"].isin(already_matched_codes)
].copy()

print(f"LGD GPs not yet matched: {len(gp_lookup_unmatched)}")

# Match
stage_d_matches = gp_lookup_unmatched.merge(
    res_district_only,
    on=["gp_norm", "district_norm_hist"],
    how="inner"
)[["gp_lgd_code", "gp_name_lgd", "district_lgd", "subdistrict_lgd",
   "reservation_dose", "reservation_dose_n",
   "reserved_women_2005", "reserved_women_2010"]].copy()

stage_d_matches["match_stage"] = "D_district_only_unique_dose"

print(f"Stage D new matches:     {len(stage_d_matches)}")
print(f"Stage D dose distribution:")
print(stage_d_matches["reservation_dose"].value_counts())

# ── Combine into final GP reservation history ─────────────────────────────────
gp_res_history_final = pd.concat([
    gp_res_history,
    stage_d_matches
], ignore_index=True)

# Sanity check: no duplicates
assert gp_res_history_final["gp_lgd_code"].duplicated().sum() == 0, \
    "Duplicate GP LGD codes in final history!"

print(f"\nFinal GP reservation history: {len(gp_res_history_final)} GPs")
print(f"  Stages A-C: {len(gp_res_history)} GPs")
print(f"  Stage D:    {len(stage_d_matches)} GPs")
print(f"\nFinal dose distribution:")
print(gp_res_history_final["reservation_dose"].value_counts())
print(f"\nFinal match stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

# ── Save ──────────────────────────────────────────────────────────────────────
gp_res_history_final.to_csv(
    OUTPUT_DIR / "gp_reservation_history.csv", index=False
)
print(f"\nSaved: gp_reservation_history.csv")

Matched reservation rows: 5595
Unique GP LGD codes:      4346

GP LGD codes with conflicting dose: 370
Sample conflicts:
gp_lgd_code gp_name_lgd     district_lgd  dose_values  n_rows
     294376        Kuda         Sanchore never | once       4
     294618       Sawar            Churu never | once       3
      33594      Borada            Kekri never | once       3
      33713      Sadara            Kekri once | twice       4
      33714      Salari            Kekri once | twice       3
      33772      Moyana           Beawar once | twice       2
      33813        Nand            Ajmer once | twice       2
      33814      Nandla            Ajmer once | twice       2
      33915      Karoda  Kotputli-Behror once | twice       2
      34066       Basni Khairthal-Tijara once | twice       2

Saved 1160 conflict rows for review.

GP reservation history (non-conflicted): 3976 GPs

Dose distribution:
reservation_dose
once     1932
never    1418
twice     626
Name: count, dtype: int64

Ma

In [9]:
# =============================================================================
# CELL 7 — Load constrained valid MC file and explode to long format
# =============================================================================

mc = pd.read_csv(MC_PATH, dtype={"DHSCLUST": int})
print(f"MC (constrained valid): {len(mc)} clusters")
print(f"Columns: {mc.columns.tolist()}")

# ── Verify this is the constrained valid file ─────────────────────────────────
print(f"\nSample primary_gp_prob values (should be high, ~0.4-1.0):")
print(mc["primary_gp_prob"].describe())

# ── Explode gp_distribution to long format ────────────────────────────────────
rows = []
for _, row in mc.iterrows():
    gp_dist = parse_gp_dist(row["gp_distribution"])
    total_sims = sum(gp_dist.values())
    if total_sims == 0:
        # Cluster has no GP in radius — preserve with NA GP
        rows.append({
            "DHSCLUST":    int(row["DHSCLUST"]),
            "gp_lgd_code": pd.NA,
            "gp_prob":     np.nan
        })
        continue
    for gp_code, count in gp_dist.items():
        rows.append({
            "DHSCLUST":    int(row["DHSCLUST"]),
            "gp_lgd_code": normalize_lgd_code(gp_code),
            "gp_prob":     float(count) / total_sims
        })

mc_long = pd.DataFrame(rows)
mc_long["DHSCLUST"] = mc_long["DHSCLUST"].astype(int)

print(f"\nLong-format cluster-GP pairs: {len(mc_long)}")
print(f"Unique clusters:              {mc_long['DHSCLUST'].nunique()}")
print(f"Unique GPs:                   {mc_long['gp_lgd_code'].nunique()}")
print(f"Rows with no GP (NA):         {mc_long['gp_lgd_code'].isna().sum()}")

# ── Verify probabilities sum to 1.0 per cluster ───────────────────────────────
prob_sums = (
    mc_long[mc_long["gp_lgd_code"].notna()]
    .groupby("DHSCLUST")["gp_prob"].sum()
)
print(f"\nPer-cluster gp_prob sum (should all be 1.0):")
print(prob_sums.describe())
print(f"Clusters not summing to 1.0 (tol 1e-6): "
      f"{(~np.isclose(prob_sums, 1.0, atol=1e-6)).sum()}")

# ── Spot check: cluster 290573 ────────────────────────────────────────────────
print(f"\nSpot check — cluster 290573:")
print(mc_long[mc_long["DHSCLUST"]==290573][
    ["gp_lgd_code","gp_prob"]].to_string(index=False))
print(f"Sum: {mc_long[mc_long['DHSCLUST']==290573]['gp_prob'].sum():.6f}")

MC (constrained valid): 1189 clusters
Columns: ['DHSCLUST', 'URBAN_RURA', 'DHSREGNA', 'round', 'displacement_R_m', 'primary_gp', 'primary_gp_prob', 'n_gps_hit', 'n_in_district', 'n_outside_district', 'n_outside_any_gp', 'gp_distribution', 'gp_coverage_rate']

Sample primary_gp_prob values (should be high, ~0.4-1.0):
count    1189.000000
mean        0.435863
std         0.172301
min         0.034800
25%         0.317000
50%         0.408000
75%         0.536000
max         1.000000
Name: primary_gp_prob, dtype: float64

Long-format cluster-GP pairs: 9733
Unique clusters:              1189
Unique GPs:                   6064
Rows with no GP (NA):         0

Per-cluster gp_prob sum (should all be 1.0):
count    1.189000e+03
mean     1.000000e+00
std      1.288433e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: gp_prob, dtype: float64
Clusters not summing to 1.0 (tol 1e-6): 0

Spot check — cluster 290573:
gp_lgd_code  g

In [10]:
# =============================================================================
# CELL 8 — Merge GP reservation history onto long-format cluster-GP file
# =============================================================================

# ── Merge ─────────────────────────────────────────────────────────────────────
cluster_gp_res = mc_long.merge(
    gp_res_history_final[[
        "gp_lgd_code", "reservation_dose", "reservation_dose_n",
        "reserved_women_2005", "reserved_women_2010",
        "gp_name_lgd", "district_lgd", "subdistrict_lgd", "match_stage"
    ]],
    on="gp_lgd_code",
    how="left"
)

n_matched   = cluster_gp_res["reservation_dose"].notna().sum()
n_unmatched = cluster_gp_res["reservation_dose"].isna().sum()
n_total     = len(cluster_gp_res)

print(f"After reservation merge:")
print(f"  Total cluster-GP rows:    {n_total}")
print(f"  Matched rows:             {n_matched} ({n_matched/n_total:.1%})")
print(f"  Unmatched rows:           {n_unmatched} ({n_unmatched/n_total:.1%})")
print(f"  Clusters with any match:  "
      f"{cluster_gp_res[cluster_gp_res['reservation_dose'].notna()]['DHSCLUST'].nunique()}")

# ── Probability-weighted coverage ─────────────────────────────────────────────
known_prob_mass = cluster_gp_res.loc[
    cluster_gp_res["reservation_dose"].notna(), "gp_prob"
].sum()
total_prob_mass = cluster_gp_res["gp_prob"].sum()
print(f"\nProbability-weighted coverage:")
print(f"  Known reservation mass:   {known_prob_mass:.1f}")
print(f"  Total GP mass:            {total_prob_mass:.1f}")
print(f"  Share known:              {known_prob_mass/total_prob_mass:.3f}")

# ── Dose distribution among matched rows ──────────────────────────────────────
print(f"\nDose distribution among matched rows:")
print(cluster_gp_res["reservation_dose"].value_counts(dropna=False))

# ── Match stage breakdown among matched rows ──────────────────────────────────
print(f"\nMatch stage breakdown:")
print(cluster_gp_res["match_stage"].value_counts(dropna=False))

# ── Spot check cluster 290573 ─────────────────────────────────────────────────
print(f"\nSpot check — cluster 290573:")
print(cluster_gp_res[cluster_gp_res["DHSCLUST"]==290573][[
    "gp_lgd_code","gp_prob","reservation_dose","match_stage"
]].to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
cluster_gp_res.to_csv(
    OUTPUT_DIR / "cluster_gp_res_long.csv", index=False
)
print(f"\nSaved: cluster_gp_res_long.csv")

After reservation merge:
  Total cluster-GP rows:    9733
  Matched rows:             3640 (37.4%)
  Unmatched rows:           6093 (62.6%)
  Clusters with any match:  1084

Probability-weighted coverage:
  Known reservation mass:   463.8
  Total GP mass:            1189.0
  Share known:              0.390

Dose distribution among matched rows:
reservation_dose
NaN      6093
once     1752
never    1300
twice     588
Name: count, dtype: int64

Match stage breakdown:
match_stage
NaN                            6093
A_exact_gp_district_samiti     2059
B_exact_gp_district            1240
C_unique_gp_name_statewide      189
D_district_only_unique_dose     152
Name: count, dtype: int64

Spot check — cluster 290573:
gp_lgd_code  gp_prob reservation_dose                match_stage
      39494 0.958217            never C_unique_gp_name_statewide
      39491 0.041783              NaN                        NaN

Saved: cluster_gp_res_long.csv


In [11]:
# =============================================================================
# CELL 9 — Aggregate to cluster-level treatment probabilities
# =============================================================================

# ── Per-cluster known treatment mass ─────────────────────────────────────────
cluster_known = (
    cluster_gp_res[cluster_gp_res["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc  = ("gp_prob", "sum"),
        p_never_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res.loc[x.index, "reservation_dose"] == "never"].sum()),
        p_once_mc             = ("gp_prob", lambda x:
            x[cluster_gp_res.loc[x.index, "reservation_dose"] == "once"].sum()),
        p_twice_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res.loc[x.index, "reservation_dose"] == "twice"].sum()),
    )
)

# ── All 1,189 clusters — fill zeros for unlinked ──────────────────────────────
all_clusters = mc[["DHSCLUST","DHSREGNA","primary_gp",
                   "primary_gp_prob","n_gps_hit"]].copy()
all_clusters["DHSCLUST"] = all_clusters["DHSCLUST"].astype(int)

treat_probs_new = all_clusters.merge(
    cluster_known, on="DHSCLUST", how="left"
)

for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_new[col] = treat_probs_new[col].fillna(0.0)

# ── Derived variables ─────────────────────────────────────────────────────────
treat_probs_new["p_unknown_treatment_mc"] = (
    1 - treat_probs_new["p_known_treatment_mc"]
).clip(lower=0)

treat_probs_new["p_any_reserved_mc"] = (
    treat_probs_new["p_once_mc"] + treat_probs_new["p_twice_mc"]
)

treat_probs_new["expected_dose_mc"] = (
    treat_probs_new["p_once_mc"] + 2 * treat_probs_new["p_twice_mc"]
)

treat_probs_new["treatment_certainty_mc"] = treat_probs_new[[
    "p_never_mc", "p_once_mc", "p_twice_mc"
]].max(axis=1)

# ── Primary GP dose ───────────────────────────────────────────────────────────
primary_dose = (
    cluster_gp_res[cluster_gp_res["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False)
    .first()[["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":     "primary_gp_matched",
        "reservation_dose":"primary_gp_dose"
    })
)
treat_probs_new = treat_probs_new.merge(
    primary_dose, on="DHSCLUST", how="left"
)

# ── Define working clusters: any matched GP in radius ────────────────────────
treat_probs_new["is_working_cluster"] = (
    treat_probs_new["p_known_treatment_mc"] > 0
)

working_clusters_new = treat_probs_new[
    treat_probs_new["is_working_cluster"]
].copy()

print(f"Total clusters:              {len(treat_probs_new)}")
print(f"Working clusters (>0 known): {len(working_clusters_new)}")
print(f"\np_known_treatment_mc distribution (working clusters):")
print(working_clusters_new["p_known_treatment_mc"].describe())

bins   = [0, 0.25, 0.50, 0.75, 0.90, 0.999, 1.01]
labels = ["0-25%", "25-50%", "50-75%", "75-90%", "90-99%", "100%"]
working_clusters_new["known_mass_bin"] = pd.cut(
    working_clusters_new["p_known_treatment_mc"],
    bins=bins, labels=labels, right=False
)
print(f"\nKnown treatment mass bins:")
print(working_clusters_new["known_mass_bin"].value_counts().sort_index())

print(f"\nFully linked (>=99.9%): "
      f"{(working_clusters_new['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=75% linked:           "
      f"{(working_clusters_new['p_known_treatment_mc'] >= 0.75).sum()}")
print(f">=90% linked:           "
      f"{(working_clusters_new['p_known_treatment_mc'] >= 0.90).sum()}")

print(f"\nDose distribution (working clusters):")
print(f"  Mean p_never_mc:  {working_clusters_new['p_never_mc'].mean():.3f}")
print(f"  Mean p_once_mc:   {working_clusters_new['p_once_mc'].mean():.3f}")
print(f"  Mean p_twice_mc:  {working_clusters_new['p_twice_mc'].mean():.3f}")
print(f"  Mean treatment certainty: "
      f"{working_clusters_new['treatment_certainty_mc'].mean():.3f}")

print(f"\nSpot check — cluster 290573:")
print(treat_probs_new[treat_probs_new["DHSCLUST"]==290573][[
    "DHSCLUST","p_known_treatment_mc","p_never_mc",
    "p_once_mc","p_twice_mc","treatment_certainty_mc"
]].to_string(index=False))

Total clusters:              1189
Working clusters (>0 known): 1084

p_known_treatment_mc distribution (working clusters):
count    1084.000000
mean        0.427849
std         0.291518
min         0.001000
25%         0.171693
50%         0.387529
75%         0.665978
max         1.000000
Name: p_known_treatment_mc, dtype: float64

Known treatment mass bins:
known_mass_bin
0-25%     384
25-50%    272
50-75%    238
75-90%    104
90-99%     68
100%       18
Name: count, dtype: int64

Fully linked (>=99.9%): 18
>=75% linked:           190
>=90% linked:           86

Dose distribution (working clusters):
  Mean p_never_mc:  0.153
  Mean p_once_mc:   0.208
  Mean p_twice_mc:  0.067
  Mean treatment certainty: 0.342

Spot check — cluster 290573:
 DHSCLUST  p_known_treatment_mc  p_never_mc  p_once_mc  p_twice_mc  treatment_certainty_mc
   290573              0.958217    0.958217        0.0         0.0                0.958217


In [12]:
# =============================================================================
# CELL 9B — Diagnose working cluster definition
# =============================================================================

# Distribution of p_known_treatment_mc
print("Full distribution including very low known mass:")
low_known = working_clusters_new[
    working_clusters_new["p_known_treatment_mc"] < 0.10
]
print(f"  Clusters with <10% known mass: {len(low_known)}")
print(f"  Clusters with <25% known mass: "
      f"{(working_clusters_new['p_known_treatment_mc'] < 0.25).sum()}")

# What does the original 552 correspond to?
# In the original pipeline, working clusters were those where
# the PRIMARY GP had a matched reservation history
# i.e. primary_gp_prob > 0 AND primary_gp_dose is not NaN

primary_gp_matched = cluster_gp_res[
    cluster_gp_res["reservation_dose"].notna()
].sort_values(["DHSCLUST","gp_prob"], ascending=[True,False]) \
 .groupby("DHSCLUST")["gp_lgd_code"].first()

# Check: for how many clusters is the highest-probability GP matched?
mc_with_primary = mc[["DHSCLUST","primary_gp"]].copy()
mc_with_primary["DHSCLUST"] = mc_with_primary["DHSCLUST"].astype(int)
mc_with_primary["primary_gp"] = mc_with_primary["primary_gp"].apply(
    normalize_lgd_code
)
mc_with_primary["primary_gp_has_history"] = mc_with_primary["primary_gp"].isin(
    set(gp_res_history_final["gp_lgd_code"])
)

print(f"\nClusters where primary GP has reservation history: "
      f"{mc_with_primary['primary_gp_has_history'].sum()}")
print(f"Clusters where primary GP does NOT have history:   "
      f"{(~mc_with_primary['primary_gp_has_history']).sum()}")

# Of the 1,084 working clusters, how many have primary GP matched?
working_with_primary = mc_with_primary[
    mc_with_primary["DHSCLUST"].isin(
        working_clusters_new["DHSCLUST"]
    )
]["primary_gp_has_history"].sum()
print(f"\nOf 1,084 working clusters, primary GP matched: {working_with_primary}")
print(f"Of 1,084 working clusters, primary GP NOT matched: "
      f"{1084 - working_with_primary}")

# p_known distribution for clusters where primary GP IS matched vs not
treat_probs_new["primary_gp_norm"] = treat_probs_new["primary_gp"].apply(
    normalize_lgd_code
)
treat_probs_new["primary_gp_has_history"] = treat_probs_new["primary_gp_norm"].isin(
    set(gp_res_history_final["gp_lgd_code"])
)

primary_matched = treat_probs_new[treat_probs_new["primary_gp_has_history"]]
primary_unmatched = treat_probs_new[~treat_probs_new["primary_gp_has_history"]]

print(f"\nClusters where primary GP matched: {len(primary_matched)}")
print(f"  >=75% known mass: "
      f"{(primary_matched['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known:     "
      f"{primary_matched['p_known_treatment_mc'].mean():.3f}")

print(f"\nClusters where primary GP NOT matched: {len(primary_unmatched)}")
print(f"  >=75% known mass: "
      f"{(primary_unmatched['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known:     "
      f"{primary_unmatched['p_known_treatment_mc'].mean():.3f}")

Full distribution including very low known mass:
  Clusters with <10% known mass: 161
  Clusters with <25% known mass: 384

Clusters where primary GP has reservation history: 465
Clusters where primary GP does NOT have history:   724

Of 1,084 working clusters, primary GP matched: 465
Of 1,084 working clusters, primary GP NOT matched: 619

Clusters where primary GP matched: 465
  >=75% known mass: 190
  Mean p_known:     0.709

Clusters where primary GP NOT matched: 724
  >=75% known mass: 0
  Mean p_known:     0.186


In [13]:
# =============================================================================
# CELL 9C — Why 465 vs original 552? Diagnose gap
# =============================================================================

# Original 552 working clusters from the comprehensive lookup file
original_treat = pd.read_csv(
    Path("../outputs/merged_rajasthan_gp") /
    "nfhs4_rj_valid_mc_treatment_status_probabilities_COMPREHENSIVE_LOOKUP.csv",
    dtype={"DHSCLUST": int}
)
original_working = set(original_treat["DHSCLUST"].unique())
new_primary_matched = set(
    treat_probs_new[treat_probs_new["primary_gp_has_history"]]["DHSCLUST"]
)

in_original_not_new = original_working - new_primary_matched
in_new_not_original = new_primary_matched - original_working
in_both             = original_working & new_primary_matched

print(f"Original working clusters:        {len(original_working)}")
print(f"New primary-matched clusters:     {len(new_primary_matched)}")
print(f"In both:                          {len(in_both)}")
print(f"In original but NOT new:          {len(in_original_not_new)}")
print(f"In new but NOT original:          {len(in_new_not_original)}")

# For clusters in original but not new: what is their primary GP?
# and is it in gp_res_history_final?
missing_clusters = treat_probs_new[
    treat_probs_new["DHSCLUST"].isin(in_original_not_new)
][["DHSCLUST","primary_gp","primary_gp_norm","primary_gp_prob",
   "primary_gp_has_history","p_known_treatment_mc"]].copy()

print(f"\nSample clusters in original but missing from new pipeline:")
print(missing_clusters.head(20).to_string(index=False))

# Check: are their primary GPs in gp_res_history_final at all?
missing_primary_gps = set(missing_clusters["primary_gp_norm"].dropna())
in_history = missing_primary_gps & set(gp_res_history_final["gp_lgd_code"])
print(f"\nMissing clusters' primary GPs in gp_res_history_final: "
      f"{len(in_history)} of {len(missing_primary_gps)}")
print(f"Missing clusters' primary GPs NOT in history: "
      f"{len(missing_primary_gps - in_history)}")

# What p_known does original treat_probs show for these missing clusters?
original_missing = original_treat[
    original_treat["DHSCLUST"].isin(in_original_not_new)
][["DHSCLUST","p_known_treatment_mc","primary_gp","primary_gp_prob",
   "primary_gp_dose"]].copy()
print(f"\nOriginal treat_probs for missing clusters:")
print(original_missing.describe())
print(f"\nPrimary GP dose distribution among missing clusters:")
print(original_missing["primary_gp_dose"].value_counts(dropna=False))

Original working clusters:        552
New primary-matched clusters:     465
In both:                          420
In original but NOT new:          132
In new but NOT original:          45

Sample clusters in original but missing from new pipeline:
 DHSCLUST  primary_gp primary_gp_norm  primary_gp_prob  primary_gp_has_history  p_known_treatment_mc
   290011     39408.0           39408           0.4200                   False              0.431349
   290024     34728.0           34728           0.3210                   False              0.387195
   290028     41134.0           41134           0.6192                   False              0.102384
   290030     38064.0           38064           0.3260                   False              0.293000
   290052     36612.0           36612           0.5170                   False              0.303313
   290068     38899.0           38899           0.4960                   False              0.040377
   290069     36205.0           36205       

In [14]:
# Are the 39 "unknown" primary GP dose clusters worth recovering?
original_unknown_primary = original_treat[
    original_treat["DHSCLUST"].isin(in_original_not_new) &
    original_treat["primary_gp_dose"].eq("unknown")
][["DHSCLUST","p_known_treatment_mc","primary_gp_prob",
   "primary_gp_dose"]].copy()

print(f"Missing clusters with unknown primary GP dose: {len(original_unknown_primary)}")
print(f"Their p_known_treatment_mc distribution:")
print(original_unknown_primary["p_known_treatment_mc"].describe())
print(f">=75% known mass: "
      f"{(original_unknown_primary['p_known_treatment_mc'] >= 0.75).sum()}")

# And the 93 with known primary GP dose
original_known_primary = original_treat[
    original_treat["DHSCLUST"].isin(in_original_not_new) &
    original_treat["primary_gp_dose"].ne("unknown") &
    original_treat["primary_gp_dose"].notna()
][["DHSCLUST","p_known_treatment_mc","primary_gp_prob",
   "primary_gp_dose"]].copy()

print(f"\nMissing clusters with known primary GP dose: {len(original_known_primary)}")
print(f"Their p_known_treatment_mc distribution:")
print(original_known_primary["p_known_treatment_mc"].describe())
print(f">=75% known mass: "
      f"{(original_known_primary['p_known_treatment_mc'] >= 0.75).sum()}")
print(f">=90% known mass: "
      f"{(original_known_primary['p_known_treatment_mc'] >= 0.90).sum()}")

Missing clusters with unknown primary GP dose: 39
Their p_known_treatment_mc distribution:
count    39.000000
mean      0.387135
std       0.153946
min       0.000000
25%       0.314991
50%       0.402000
75%       0.473920
max       0.696482
Name: p_known_treatment_mc, dtype: float64
>=75% known mass: 0

Missing clusters with known primary GP dose: 93
Their p_known_treatment_mc distribution:
count    93.000000
mean      0.672279
std       0.177613
min       0.291000
25%       0.539000
50%       0.668810
75%       0.811340
max       1.000000
Name: p_known_treatment_mc, dtype: float64
>=75% known mass: 32
>=90% known mass: 12


In [15]:
# =============================================================================
# CELL 9D — Stage E: high-confidence fuzzy GP name matching
# Recover clusters whose primary GP name is a close but not exact match
# to a reservation record within the same historical district
# =============================================================================

from difflib import SequenceMatcher

def similarity(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

FUZZY_THRESHOLD = 0.90  # GP name similarity >= 90%

# GPs in MC that are still unmatched (not in gp_res_history_final)
already_matched = set(gp_res_history_final["gp_lgd_code"])

gp_lookup_unmatched = gp_lookup[
    ~gp_lookup["gp_lgd_code"].isin(already_matched)
].copy()

print(f"LGD GPs still unmatched after Stages A-D: {len(gp_lookup_unmatched)}")

# Focus on GPs that are primary GPs for clusters in the gap
# (in original working sample but not in our new primary-matched set)
gap_primary_gps = set(
    treat_probs_new[
        treat_probs_new["DHSCLUST"].isin(in_original_not_new)
    ]["primary_gp_norm"].dropna()
)
print(f"Primary GPs of gap clusters: {len(gap_primary_gps)}")

# Narrow to unmatched LGD GPs that are primary GPs for gap clusters
gp_lookup_gap = gp_lookup_unmatched[
    gp_lookup_unmatched["gp_lgd_code"].isin(gap_primary_gps)
].copy()
print(f"Unmatched LGD GPs that are primary GPs of gap clusters: {len(gp_lookup_gap)}")

# Build reservation candidates by district
res_by_district = (
    res_clean
    .groupby(["district_norm_hist", "gp_norm"])
    .agg(
        n_dose_values       = ("reservation_dose",    "nunique"),
        reservation_dose    = ("reservation_dose",    "first"),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        reservation_dose_n  = ("reservation_dose_n",  "first")
    )
    .reset_index()
)
# Only unique dose matches
res_by_district = res_by_district[
    res_by_district["n_dose_values"] == 1
].drop(columns="n_dose_values")

# Fuzzy match each gap GP against reservation candidates in same district
fuzzy_matches = []

for _, gp_row in gp_lookup_gap.iterrows():
    dist   = gp_row["district_norm_hist"]
    gp_n   = gp_row["gp_norm"]
    lgd_code = gp_row["gp_lgd_code"]

    candidates = res_by_district[
        res_by_district["district_norm_hist"] == dist
    ].copy()

    if candidates.empty:
        continue

    candidates["sim"] = candidates["gp_norm"].apply(
        lambda x: similarity(gp_n, x)
    )
    best = candidates.nlargest(1, "sim").iloc[0]

    if best["sim"] >= FUZZY_THRESHOLD:
        fuzzy_matches.append({
            "gp_lgd_code":        lgd_code,
            "gp_name_lgd":        gp_row["gp_name_lgd"],
            "district_lgd":       gp_row["district_lgd"],
            "subdistrict_lgd":    gp_row["subdistrict_lgd"],
            "reservation_dose":   best["reservation_dose"],
            "reservation_dose_n": best["reservation_dose_n"],
            "reserved_women_2005":best["reserved_women_2005"],
            "reserved_women_2010":best["reserved_women_2010"],
            "match_stage":        "E_fuzzy_gp_name_district",
            "fuzzy_similarity":   best["sim"],
            "res_gp_matched":     best["gp_norm"]
        })

stage_e_df = pd.DataFrame(fuzzy_matches)
print(f"\nStage E fuzzy matches (similarity >= {FUZZY_THRESHOLD}): "
      f"{len(stage_e_df)}")

if len(stage_e_df) > 0:
    print(f"\nDose distribution:")
    print(stage_e_df["reservation_dose"].value_counts())
    print(f"\nSimilarity score distribution:")
    print(stage_e_df["fuzzy_similarity"].describe())
    print(f"\nSample matches (review for correctness):")
    print(stage_e_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd",
        "res_gp_matched","fuzzy_similarity","reservation_dose"
    ]].head(20).to_string(index=False))

LGD GPs still unmatched after Stages A-D: 7046
Primary GPs of gap clusters: 126
Unmatched LGD GPs that are primary GPs of gap clusters: 126

Stage E fuzzy matches (similarity >= 0.9): 70

Dose distribution:
reservation_dose
once     35
never    28
twice     7
Name: count, dtype: int64

Similarity score distribution:
count    70.000000
mean      0.929762
std       0.019377
min       0.900000
25%       0.909091
50%       0.933333
75%       0.941176
max       0.972973
Name: fuzzy_similarity, dtype: float64

Sample matches (review for correctness):
gp_lgd_code          gp_name_lgd     district_lgd      res_gp_matched  fuzzy_similarity reservation_dose
      33590          Bhagwanpura            Kekri        bhagwantpura          0.956522             once
      33631          Gurha Khurd            Kekri          gudhakhurd          0.900000            never
      33691           Bheemrawas            Kekri          bheemdawas          0.900000            never
      33738            Ralawa

In [16]:
# =============================================================================
# CELL 9E — Add Stage E matches and rebuild final GP reservation history
# =============================================================================

# Append Stage E to gp_res_history_final
gp_res_history_final_v2 = pd.concat([
    gp_res_history_final,
    stage_e_df[[
        "gp_lgd_code", "gp_name_lgd", "district_lgd", "subdistrict_lgd",
        "reservation_dose", "reservation_dose_n",
        "reserved_women_2005", "reserved_women_2010", "match_stage"
    ]]
], ignore_index=True)

# Verify no duplicates
assert gp_res_history_final_v2["gp_lgd_code"].duplicated().sum() == 0, \
    "Duplicate GP LGD codes after Stage E!"

print(f"GP reservation history after Stage E:")
print(f"  Stages A-D: {len(gp_res_history_final)} GPs")
print(f"  Stage E:    {len(stage_e_df)} GPs")
print(f"  Total:      {len(gp_res_history_final_v2)} GPs")
print(f"\nFull match stage breakdown:")
print(gp_res_history_final_v2["match_stage"].value_counts())

# ── Rebuild cluster_gp_res with Stage E ──────────────────────────────────────
cluster_gp_res_v2 = mc_long.merge(
    gp_res_history_final_v2[[
        "gp_lgd_code", "reservation_dose", "reservation_dose_n",
        "reserved_women_2005", "reserved_women_2010",
        "gp_name_lgd", "district_lgd", "subdistrict_lgd", "match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

print(f"\nRebuilt cluster_gp_res:")
print(f"  Matched rows:            {cluster_gp_res_v2['reservation_dose'].notna().sum()}")
print(f"  Unmatched rows:          {cluster_gp_res_v2['reservation_dose'].isna().sum()}")
print(f"  Clusters with any match: "
      f"{cluster_gp_res_v2[cluster_gp_res_v2['reservation_dose'].notna()]['DHSCLUST'].nunique()}")

# ── Recompute cluster-level treatment probabilities ───────────────────────────
cluster_known_v2 = (
    cluster_gp_res_v2[cluster_gp_res_v2["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v2.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v2.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v2.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v2 = all_clusters.merge(
    cluster_known_v2, on="DHSCLUST", how="left"
)
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v2[col] = treat_probs_v2[col].fillna(0.0)

treat_probs_v2["p_unknown_treatment_mc"] = (
    1 - treat_probs_v2["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v2["p_any_reserved_mc"]    = (
    treat_probs_v2["p_once_mc"] + treat_probs_v2["p_twice_mc"]
)
treat_probs_v2["expected_dose_mc"]     = (
    treat_probs_v2["p_once_mc"] + 2 * treat_probs_v2["p_twice_mc"]
)
treat_probs_v2["treatment_certainty_mc"] = treat_probs_v2[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

# Primary GP dose
primary_dose_v2 = (
    cluster_gp_res_v2[cluster_gp_res_v2["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v2 = treat_probs_v2.merge(
    primary_dose_v2, on="DHSCLUST", how="left"
)

# Primary GP matched flag
treat_probs_v2["primary_gp_norm"] = treat_probs_v2["primary_gp"].apply(
    normalize_lgd_code
)
treat_probs_v2["primary_gp_has_history"] = treat_probs_v2["primary_gp_norm"].isin(
    set(gp_res_history_final_v2["gp_lgd_code"])
)

# ── Working cluster summary ───────────────────────────────────────────────────
working_v2 = treat_probs_v2[treat_probs_v2["primary_gp_has_history"]].copy()

print(f"\nWorking clusters (primary GP matched): {len(working_v2)}")
print(f"  Fully linked (>=99.9%): "
      f"{(working_v2['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  >=90% linked:           "
      f"{(working_v2['p_known_treatment_mc'] >= 0.90).sum()}")
print(f"  >=75% linked:           "
      f"{(working_v2['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known:           "
      f"{working_v2['p_known_treatment_mc'].mean():.3f}")

# Overlap with original 552
new_working_set = set(working_v2["DHSCLUST"])
print(f"\nOverlap with original 552:")
print(f"  In both:                {len(new_working_set & original_working)}")
print(f"  In original not new:    {len(original_working - new_working_set)}")
print(f"  In new not original:    {len(new_working_set - original_working)}")

# Spot check
print(f"\nSpot check — cluster 290573:")
print(treat_probs_v2[treat_probs_v2["DHSCLUST"]==290573][[
    "DHSCLUST","p_known_treatment_mc","p_never_mc",
    "p_once_mc","p_twice_mc","primary_gp_dose"
]].to_string(index=False))

# ── Update working variables ──────────────────────────────────────────────────
gp_res_history_final = gp_res_history_final_v2
cluster_gp_res       = cluster_gp_res_v2
treat_probs_new      = treat_probs_v2
working_clusters_new = working_v2

# Save updated files
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
print(f"\nSaved updated gp_reservation_history.csv and cluster_gp_res_long.csv")

GP reservation history after Stage E:
  Stages A-D: 4153 GPs
  Stage E:    70 GPs
  Total:      4223 GPs

Full match stage breakdown:
match_stage
A_exact_gp_district_samiti     2277
B_exact_gp_district            1514
C_unique_gp_name_statewide      185
D_district_only_unique_dose     177
E_fuzzy_gp_name_district         70
Name: count, dtype: int64

Rebuilt cluster_gp_res:
  Matched rows:            3783
  Unmatched rows:          5950
  Clusters with any match: 1091

Working clusters (primary GP matched): 541
  Fully linked (>=99.9%): 24
  >=90% linked:           103
  >=75% linked:           235
  Mean p_known:           0.716

Overlap with original 552:
  In both:                495
  In original not new:    57
  In new not original:    46

Spot check — cluster 290573:
 DHSCLUST  p_known_treatment_mc  p_never_mc  p_once_mc  p_twice_mc primary_gp_dose
   290573              0.958217    0.958217        0.0         0.0           never

Saved updated gp_reservation_history.csv and clus

In [17]:
# =============================================================================
# CELL 10 — Save cluster-level treatment probabilities
# =============================================================================

# Final working sample: primary GP matched clusters only
final_treat_probs = working_clusters_new[[
    "DHSCLUST", "DHSREGNA",
    "p_never_mc", "p_once_mc", "p_twice_mc",
    "p_any_reserved_mc", "expected_dose_mc",
    "p_known_treatment_mc", "p_unknown_treatment_mc",
    "treatment_certainty_mc",
    "primary_gp", "primary_gp_prob", "primary_gp_dose",
    "n_gps_hit"
]].copy()

print(f"Final treatment probability file:")
print(f"  Working clusters:       {len(final_treat_probs)}")
print(f"  Fully linked (>=99.9%): "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  >=75% linked:           "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known:           "
      f"{final_treat_probs['p_known_treatment_mc'].mean():.3f}")
print(f"  Mean treatment certainty: "
      f"{final_treat_probs['treatment_certainty_mc'].mean():.3f}")

bins   = [0, 0.25, 0.50, 0.75, 0.90, 0.999, 1.01]
labels = ["0-25%","25-50%","50-75%","75-90%","90-99%","100%"]
final_treat_probs["known_mass_bin"] = pd.cut(
    final_treat_probs["p_known_treatment_mc"],
    bins=bins, labels=labels, right=False
)
print(f"\nKnown treatment mass distribution:")
print(final_treat_probs["known_mass_bin"].value_counts().sort_index())

print(f"\nDose distribution:")
print(f"  Mean p_never_mc:          {final_treat_probs['p_never_mc'].mean():.3f}")
print(f"  Mean p_once_mc:           {final_treat_probs['p_once_mc'].mean():.3f}")
print(f"  Mean p_twice_mc:          {final_treat_probs['p_twice_mc'].mean():.3f}")

print(f"\nPrimary GP dose distribution:")
print(final_treat_probs["primary_gp_dose"].value_counts(dropna=False))

# District breakdown
district_summary = (
    final_treat_probs.merge(
        mc[["DHSCLUST","DHSREGNA"]].drop_duplicates(),
        on="DHSCLUST", how="left", suffixes=("","_mc")
    )
    .groupby("DHSREGNA")
    .agg(
        n_clusters      = ("DHSCLUST",              "nunique"),
        mean_p_known    = ("p_known_treatment_mc",  "mean"),
        n_fully_linked  = ("p_known_treatment_mc",  lambda x: (x>=0.999).sum()),
        n_75pct_linked  = ("p_known_treatment_mc",  lambda x: (x>=0.75).sum())
    )
    .sort_values("n_clusters", ascending=False)
    .reset_index()
)
print(f"\nDistrict breakdown (top 10 by cluster count):")
print(district_summary.head(10).to_string(index=False))

# Save
final_treat_probs.to_csv(
    OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False
)
print(f"\nSaved: cluster_treatment_probs_rj.csv")

Final treatment probability file:
  Working clusters:       541
  Fully linked (>=99.9%): 24
  >=75% linked:           235
  Mean p_known:           0.716
  Mean treatment certainty: 0.577

Known treatment mass distribution:
known_mass_bin
0-25%       0
25-50%     70
50-75%    236
75-90%    132
90-99%     79
100%       24
Name: count, dtype: int64

Dose distribution:
  Mean p_never_mc:          0.258
  Mean p_once_mc:           0.348
  Mean p_twice_mc:          0.110

Primary GP dose distribution:
primary_gp_dose
once     263
never    196
twice     82
Name: count, dtype: int64

District breakdown (top 10 by cluster count):
    DHSREGNA  n_clusters  mean_p_known  n_fully_linked  n_75pct_linked
       Ajmer          30      0.822359               3              22
     Bikaner          23      0.741796               2              11
    Bhilwara          23      0.780790               1              14
        Kota          22      0.751106               1              11
  Jhunjhunun  

In [18]:
# =============================================================================
# CELL 11 — Identify unmatched GPs and build priority list for manual lookup
# =============================================================================

# ── Isolated clusters: single GP in radius, that GP is unmatched ──────────────
# These are geometrically isolated but their GP has no reservation history
# They should NOT be counted as "fully linked" since the GP is unknown
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)
print(f"Geometrically isolated clusters with unmatched sole GP: {len(isolated_clusters)}")

analysis_clusters = set(final_treat_probs["DHSCLUST"]) - isolated_clusters
print(f"Analysis clusters (working, excl. isolated): {len(analysis_clusters)}")

# ── Unmatched GP rows within analysis clusters ────────────────────────────────
unmatched = cluster_gp_res[
    cluster_gp_res["DHSCLUST"].isin(analysis_clusters) &
    cluster_gp_res["gp_lgd_code"].notna() &
    cluster_gp_res["reservation_dose"].isna()
].copy()

# Merge p_known_treatment_mc from final_treat_probs for binning
unmatched = unmatched.merge(
    final_treat_probs[["DHSCLUST","p_known_treatment_mc"]],
    on="DHSCLUST", how="left"
)

print(f"\nUnmatched GP rows in analysis clusters: {len(unmatched)}")
print(f"Unique unmatched GPs:                   {unmatched['gp_lgd_code'].nunique()}")

# ── Aggregate per GP ──────────────────────────────────────────────────────────
gp_summary = (
    unmatched
    .groupby("gp_lgd_code", as_index=False)
    .agg(
        total_prob_mass             = ("gp_prob",               "sum"),
        n_clusters                  = ("DHSCLUST",              "nunique"),
        max_known_mass_in_clusters  = ("p_known_treatment_mc",  "max"),
        mean_known_mass_in_clusters = ("p_known_treatment_mc",  "mean"),
        min_known_mass_in_clusters  = ("p_known_treatment_mc",  "min"),
    )
    .sort_values("max_known_mass_in_clusters", ascending=False)
    .reset_index(drop=True)
)

def assign_bin(val):
    if val >= 0.999:  return "90-99%+"
    elif val >= 0.90: return "90-99%"
    elif val >= 0.75: return "75-90%"
    elif val >= 0.50: return "50-75%"
    elif val >= 0.25: return "25-50%"
    else:             return "0-25%"

gp_summary["known_mass_bin"] = (
    gp_summary["max_known_mass_in_clusters"].apply(assign_bin)
)

print(f"\nKnown mass bin distribution:")
print(gp_summary["known_mass_bin"].value_counts().sort_index())

# ── Add GP names from LGD crosswalk ──────────────────────────────────────────
gp_names = (
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]]
    .rename(columns={
        "gp_name_lgd":   "gp_name",
        "district_lgd":  "district",
        "subdistrict_lgd":"subdistrict_samiti"
    })
)

export_df = (
    gp_summary
    .merge(gp_names, on="gp_lgd_code", how="left")
    [[
        "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
        "total_prob_mass", "n_clusters", "known_mass_bin",
        "max_known_mass_in_clusters", "mean_known_mass_in_clusters",
        "min_known_mass_in_clusters",
    ]]
    .reset_index(drop=True)
)

export_df["reserved_women_2005"] = ""
export_df["reserved_women_2010"] = ""
export_df["notes"] = ""

print(f"\nTotal unmatched GPs for export: {len(export_df)}")
print(f"\nTop 15 priority GPs:")
print(export_df.head(15)[[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "known_mass_bin","max_known_mass_in_clusters","n_clusters"
]].to_string(index=False))

# ── Which priority GPs are sole unmatched GP in a cluster? ───────────────────
priority_gps = set(
    export_df[export_df["known_mass_bin"].isin(["90-99%+","90-99%"])]
    ["gp_lgd_code"].astype(str)
)
print(f"\nPriority GPs (90-99%+ and 90-99% bins): {len(priority_gps)}")

cluster_unmatched_summary = (
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(analysis_clusters) &
        cluster_gp_res["reservation_dose"].isna() &
        cluster_gp_res["gp_lgd_code"].notna()
    ]
    .groupby("DHSCLUST")["gp_lgd_code"]
    .apply(set)
    .reset_index()
    .rename(columns={"gp_lgd_code":"unmatched_gps"})
)
cluster_unmatched_summary["n_unmatched"] = (
    cluster_unmatched_summary["unmatched_gps"].apply(len)
)
cluster_unmatched_summary["all_priority"] = (
    cluster_unmatched_summary["unmatched_gps"].apply(
        lambda s: s.issubset(priority_gps)
    )
)

would_fully_link = cluster_unmatched_summary[
    cluster_unmatched_summary["all_priority"]
]["DHSCLUST"].tolist()

print(f"Clusters that become fully linked if all priority GPs matched: "
      f"{len(would_fully_link)}")

# Solo pushers
gp_impact = []
for gp in priority_gps:
    solo = cluster_unmatched_summary[
        cluster_unmatched_summary["unmatched_gps"].apply(lambda s: s == {gp})
    ]["DHSCLUST"].tolist()
    contrib = cluster_unmatched_summary[
        cluster_unmatched_summary["unmatched_gps"].apply(lambda s: gp in s) &
        cluster_unmatched_summary["all_priority"]
    ]["DHSCLUST"].tolist()
    gp_impact.append({
        "gp_lgd_code":           gp,
        "n_solo_fully_linked":   len(solo),
        "n_contrib_fully_linked":len(contrib),
    })

gp_impact_df = (
    pd.DataFrame(gp_impact)
    .merge(export_df[["gp_lgd_code","gp_name","district",
                       "subdistrict_samiti","known_mass_bin",
                       "max_known_mass_in_clusters"]],
           on="gp_lgd_code", how="left")
    .sort_values("n_solo_fully_linked", ascending=False)
)

solo_pushers = gp_impact_df[gp_impact_df["n_solo_fully_linked"] > 0]
print(f"\nGPs that are sole unmatched GP in at least one cluster:")
print(f"  {len(solo_pushers)} GPs → unlock "
      f"{solo_pushers['n_solo_fully_linked'].sum()} clusters")
print(solo_pushers[[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "n_solo_fully_linked","max_known_mass_in_clusters"
]].to_string(index=False))

Geometrically isolated clusters with unmatched sole GP: 51
Analysis clusters (working, excl. isolated): 490

Unmatched GP rows in analysis clusters: 2128
Unique unmatched GPs:                   1719

Known mass bin distribution:
known_mass_bin
25-50%    342
50-75%    873
75-90%    383
90-99%    121
Name: count, dtype: int64

Total unmatched GPs for export: 1719

Top 15 priority GPs:
gp_lgd_code          gp_name district subdistrict_samiti known_mass_bin  max_known_mass_in_clusters  n_clusters
     236071           Genoli Bhilwara         Mandalgarh         90-99%                    0.992151           2
      35853 Motoron Ka Khera Bhilwara         Mandalgarh         90-99%                    0.992151           2
      40277             Doti     Kota             Kanwas         90-99%                    0.985507           2
      40314          Gadepan     Kota              Digod         90-99%                    0.985507           1
      42037  Toda Ka Gothara     Tonk              Deo

In [19]:
# Diagnose why no solo pushers
# Show clusters closest to fully linked and what's blocking them
near_complete = cluster_unmatched_summary[
    cluster_unmatched_summary["n_unmatched"] == 1
].copy()

near_complete = near_complete.merge(
    final_treat_probs[["DHSCLUST","p_known_treatment_mc"]],
    on="DHSCLUST", how="left"
).sort_values("p_known_treatment_mc", ascending=False)

print(f"Clusters with exactly 1 unmatched GP: {len(near_complete)}")
print(f"\nTop 20 by p_known_treatment_mc:")
print(near_complete.head(20)[
    ["DHSCLUST","p_known_treatment_mc","unmatched_gps"]
].to_string(index=False))

# Are any of those single unmatched GPs in priority set?
near_complete["sole_gp"] = near_complete["unmatched_gps"].apply(
    lambda s: list(s)[0]
)
near_complete["sole_gp_is_priority"] = near_complete["sole_gp"].isin(priority_gps)
print(f"\nClusters with 1 unmatched GP where that GP is in priority set: "
      f"{near_complete['sole_gp_is_priority'].sum()}")
print(f"Clusters with 1 unmatched GP where that GP is NOT in priority set: "
      f"{(~near_complete['sole_gp_is_priority']).sum()}")

# What bins are the sole unmatched GPs in?
near_complete["sole_gp_bin"] = near_complete["sole_gp"].map(
    gp_summary.set_index("gp_lgd_code")["known_mass_bin"]
)
print(f"\nBin distribution of sole unmatched GPs:")
print(near_complete["sole_gp_bin"].value_counts(dropna=False))

Clusters with exactly 1 unmatched GP: 0

Top 20 by p_known_treatment_mc:
Empty DataFrame
Columns: [DHSCLUST, p_known_treatment_mc, unmatched_gps]
Index: []

Clusters with 1 unmatched GP where that GP is in priority set: 0
Clusters with 1 unmatched GP where that GP is NOT in priority set: 0

Bin distribution of sole unmatched GPs:
Series([], Name: count, dtype: int64)


In [20]:
# Quick projection: cluster gains from matching all 121 priority GPs
priority_mass_resolved = (
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(analysis_clusters) &
        cluster_gp_res["gp_lgd_code"].isin(priority_gps) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST", as_index=False)["gp_prob"]
    .sum()
    .rename(columns={"gp_prob": "mass_resolved"})
)

projected = final_treat_probs[["DHSCLUST","p_known_treatment_mc"]].merge(
    priority_mass_resolved, on="DHSCLUST", how="left"
)
projected["mass_resolved"]  = projected["mass_resolved"].fillna(0)
projected["p_known_after"]  = (
    projected["p_known_treatment_mc"] + projected["mass_resolved"]
).clip(upper=1.0)

print("Projected state after matching all 121 priority GPs:")
print(f"  Fully linked (>=99.9%): {(projected['p_known_after'] >= 0.999).sum()} "
      f"(currently {(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()})")
print(f"  >=90% linked:           {(projected['p_known_after'] >= 0.90).sum()} "
      f"(currently {(final_treat_probs['p_known_treatment_mc'] >= 0.90).sum()})")
print(f"  >=75% linked:           {(projected['p_known_after'] >= 0.75).sum()} "
      f"(currently {(final_treat_probs['p_known_treatment_mc'] >= 0.75).sum()})")

bins   = [0, 0.25, 0.50, 0.75, 0.90, 0.999, 1.01]
labels = ["0-25%","25-50%","50-75%","75-90%","90-99%","100%"]
projected["bin_after"] = pd.cut(
    projected["p_known_after"], bins=bins, labels=labels, right=False
)
print(f"\nFull distribution after matching 121 priority GPs:")
print(projected["bin_after"].value_counts().sort_index())

print(f"\nEstimated women (mean 25.6 per cluster):")
for label, thresh in [("Fully linked", 0.999), (">=75%", 0.75), (">=50%", 0.50)]:
    n = (projected["p_known_after"] >= thresh).sum()
    print(f"  {label}: {n} clusters ≈ {n*25.6:,.0f} women")

Projected state after matching all 121 priority GPs:
  Fully linked (>=99.9%): 78 (currently 24)
  >=90% linked:           111 (currently 103)
  >=75% linked:           241 (currently 235)

Full distribution after matching 121 priority GPs:
bin_after
0-25%       0
25-50%     70
50-75%    230
75-90%    130
90-99%     33
100%       78
Name: count, dtype: int64

Estimated women (mean 25.6 per cluster):
  Fully linked: 78 clusters ≈ 1,997 women
  >=75%: 241 clusters ≈ 6,170 women
  >=50%: 471 clusters ≈ 12,058 women


In [21]:
# =============================================================================
# CELL 12 — Export priority GP list for manual lookup
# =============================================================================

from openpyxl.styles import PatternFill

# ── Tier assignment ───────────────────────────────────────────────────────────
# Tier 1: in 90-99% or 90-99%+ bin (121 GPs — directly push clusters to fully linked)
# Tier 2: in 75-90% bin (383 GPs — improve known mass substantially)
# Tier 3: in 50-75% bin (873 GPs — lower priority)

def assign_tier(bin_label):
    if bin_label in ["90-99%+", "90-99%"]: return "Tier 1 — pushes cluster to fully linked"
    elif bin_label == "75-90%":             return "Tier 2 — high priority"
    elif bin_label == "50-75%":             return "Tier 3 — medium priority"
    else:                                   return "Tier 4 — lower priority"

export_df["lookup_tier"] = export_df["known_mass_bin"].apply(assign_tier)

# Sort by tier then by max known mass descending
tier_order = {
    "Tier 1 — pushes cluster to fully linked": 0,
    "Tier 2 — high priority":                  1,
    "Tier 3 — medium priority":                2,
    "Tier 4 — lower priority":                 3,
}
export_df["tier_sort"] = export_df["lookup_tier"].map(tier_order)
export_df = export_df.sort_values(
    ["tier_sort", "max_known_mass_in_clusters"],
    ascending=[True, False]
).drop(columns="tier_sort").reset_index(drop=True)

# ── Column order ──────────────────────────────────────────────────────────────
export_cols = [
    "lookup_tier", "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "known_mass_bin", "max_known_mass_in_clusters", "n_clusters",
    "total_prob_mass", "mean_known_mass_in_clusters",
    "reserved_women_2005", "reserved_women_2010", "notes"
]
export_df = export_df[export_cols]

print(f"Export summary:")
print(f"  Tier 1 (90-99%+/90-99%): {(export_df['lookup_tier'].str.contains('Tier 1')).sum()} GPs")
print(f"  Tier 2 (75-90%):          {(export_df['lookup_tier'].str.contains('Tier 2')).sum()} GPs")
print(f"  Tier 3 (50-75%):          {(export_df['lookup_tier'].str.contains('Tier 3')).sum()} GPs")
print(f"  Total:                    {len(export_df)} GPs")

print(f"\nTier 1 GPs (first 20):")
print(export_df[export_df["lookup_tier"].str.contains("Tier 1")][[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "max_known_mass_in_clusters","n_clusters"
]].head(20).to_string(index=False))

# ── Export to Excel ───────────────────────────────────────────────────────────
OUTPUT_PATH = OUTPUT_DIR / "unmatched_gps_for_manual_review.xlsx"

tier_fills = {
    "Tier 1": PatternFill("solid", fgColor="C6EFCE"),  # green
    "Tier 2": PatternFill("solid", fgColor="FFEB9C"),  # yellow
    "Tier 3": PatternFill("solid", fgColor="DAEEF3"),  # light blue
    "Tier 4": PatternFill("solid", fgColor="F2F2F2"),  # grey
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    export_df.to_excel(writer, index=False, sheet_name="Unmatched GPs")
    ws = writer.sheets["Unmatched GPs"]
    ws.freeze_panes = "A2"

    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        tier_val = str(row[0].value)
        fill = next(
            (f for k, f in tier_fills.items() if k in tier_val),
            tier_fills["Tier 4"]
        )
        for cell in row:
            cell.fill = fill

print(f"\nSaved: {OUTPUT_PATH}")
print(f"Total GPs exported: {len(export_df)}")

Export summary:
  Tier 1 (90-99%+/90-99%): 121 GPs
  Tier 2 (75-90%):          383 GPs
  Tier 3 (50-75%):          873 GPs
  Total:                    1719 GPs

Tier 1 GPs (first 20):
gp_lgd_code          gp_name         district subdistrict_samiti  max_known_mass_in_clusters  n_clusters
     236071           Genoli         Bhilwara         Mandalgarh                    0.992151           2
      35853 Motoron Ka Khera         Bhilwara         Mandalgarh                    0.992151           2
      40277             Doti             Kota             Kanwas                    0.985507           2
      40314          Gadepan             Kota              Digod                    0.985507           1
      42037  Toda Ka Gothara             Tonk              Deoli                    0.970170           1
      42027           Panwar             Tonk              Deoli                    0.970170           1
      40957         Roopawas             Pali               Pali                 

In [22]:
# =============================================================================
# CELL 13 — Final summary
# =============================================================================

print("=" * 65)
print("BUILD_RJ_SAMPLE — FINAL SUMMARY")
print("=" * 65)

print(f"\n── Matching pipeline ────────────────────────────────────────")
print(f"  Stage A (GP + district + samiti, exact):  "
      f"{(gp_res_history_final['match_stage']=='A_exact_gp_district_samiti').sum()} GPs")
print(f"  Stage B (GP + district, exact):           "
      f"{(gp_res_history_final['match_stage']=='B_exact_gp_district').sum()} GPs")
print(f"  Stage C (GP name unique statewide):       "
      f"{(gp_res_history_final['match_stage']=='C_unique_gp_name_statewide').sum()} GPs")
print(f"  Stage D (GP + district, unique dose):     "
      f"{(gp_res_history_final['match_stage']=='D_district_only_unique_dose').sum()} GPs")
print(f"  Stage E (fuzzy GP name + district ≥0.90): "
      f"{(gp_res_history_final['match_stage']=='E_fuzzy_gp_name_district').sum()} GPs")
print(f"  Total GP reservation histories:           "
      f"{len(gp_res_history_final)} GPs")
print(f"  Conflict-excluded GPs:                    "
      f"{len(conflict_gp_codes)}")

print(f"\n── Working sample ───────────────────────────────────────────")
print(f"  Total rural RJ NFHS-4 clusters (MC):      "
      f"{len(mc)}")
print(f"  Working clusters (primary GP matched):     "
      f"{len(final_treat_probs)}")
print(f"  Geometrically isolated (excl.):            "
      f"{len(isolated_clusters)}")
print(f"  Analysis clusters:                         "
      f"{len(analysis_clusters)}")

print(f"\n── Linkage quality ──────────────────────────────────────────")
print(f"  Fully linked (p_known ≥ 99.9%):           "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  ≥90% linked:                              "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.90).sum()}")
print(f"  ≥75% linked:                              "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known_treatment_mc:                "
      f"{final_treat_probs['p_known_treatment_mc'].mean():.3f}")
print(f"  Mean treatment_certainty_mc:              "
      f"{final_treat_probs['treatment_certainty_mc'].mean():.3f}")

print(f"\n── Projected gains from manual lookup ───────────────────────")
print(f"  Priority GPs to look up (Tier 1):         121")
print(f"  Clusters pushed to fully linked:           +54 → 78 total")
print(f"  Clusters pushed to ≥75%:                  +6  → 241 total")

print(f"\n── Output files (outputs/final_rj_sample/) ──────────────────")
for fname in sorted(OUTPUT_DIR.iterdir()):
    size_kb = fname.stat().st_size / 1024
    print(f"  {fname.name:<55} {size_kb:>8.1f} KB")

print(f"\n── Comparison with original pipeline ────────────────────────")
print(f"  {'Metric':<40} {'Original':>10} {'New':>10}")
print(f"  {'-'*60}")
print(f"  {'Working clusters':<40} {'552':>10} {'541':>10}")
print(f"  {'Fully linked':<40} {'11':>10} {'24':>10}")
print(f"  {'>=75% linked':<40} {'177':>10} {'235':>10}")
print(f"  {'Mean p_known':<40} {'0.653':>10} {'0.716':>10}")
print(f"  {'Mean treatment certainty':<40} {'0.528':>10} {'0.577':>10}")
print(f"  {'GP reservation histories':<40} {'3752':>10} {'4223':>10}")

BUILD_RJ_SAMPLE — FINAL SUMMARY

── Matching pipeline ────────────────────────────────────────
  Stage A (GP + district + samiti, exact):  2277 GPs
  Stage B (GP + district, exact):           1514 GPs
  Stage C (GP name unique statewide):       185 GPs
  Stage D (GP + district, unique dose):     177 GPs
  Stage E (fuzzy GP name + district ≥0.90): 70 GPs
  Total GP reservation histories:           4223 GPs
  Conflict-excluded GPs:                    370

── Working sample ───────────────────────────────────────────
  Total rural RJ NFHS-4 clusters (MC):      1189
  Working clusters (primary GP matched):     541
  Geometrically isolated (excl.):            51
  Analysis clusters:                         490

── Linkage quality ──────────────────────────────────────────
  Fully linked (p_known ≥ 99.9%):           24
  ≥90% linked:                              103
  ≥75% linked:                              235
  Mean p_known_treatment_mc:                0.716
  Mean treatment_certainty_mc

In [23]:
# =============================================================================
# CELL 14 — Load 2005 sarpanch publication and inspect structure
# =============================================================================

sarpanch_2005 = pd.read_excel(
    "/Users/sonalideliwala/Downloads/2005 SARPANCH publication-FINAL.xlsx"
)

print(f"Shape: {sarpanch_2005.shape}")
print(f"Columns: {sarpanch_2005.columns.tolist()}")
print(f"\nFirst 10 rows:")
print(sarpanch_2005.head(10).to_string(index=False))
print(f"\nColumn value samples:")
for col in sarpanch_2005.columns:
    vals = sarpanch_2005[col].dropna().unique()[:5]
    print(f"  {col}: {vals}")

/Users/sonalideliwala/Library/Python/3.12/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Shape: (9383, 8)
Columns: ['S.No.', 'DISTRICT', 'PANCHAYAT SAMITI', 'GRAM PANCHAYAT', 'CATEG. OF POST OF SARPANCH', 'NAME OF ELECTED SARPANCH', 'ELECTED CANDIDATE (MALE/ FEMALE)', 'CATEGORY -GEN, SC, ST, OBC']

First 10 rows:
S.No. DISTRICT PANCHAYAT SAMITI GRAM PANCHAYAT CATEG. OF POST OF SARPANCH NAME OF ELECTED SARPANCH ELECTED CANDIDATE (MALE/ FEMALE) CATEGORY -GEN, SC, ST, OBC
  NaN      NaN              NaN            NaN                        NaN                      NaN                              NaN                        NaN
    1    AJMER            ARAIN       AAKODIYA                        OBC                   GAJMAL                             MALE                        OBC
    2    AJMER            ARAIN         AJGARA                         SC                     LADU                             MALE                         SC
    3    AJMER            ARAIN          ARAIN                         ST              KANHIYA LAL                             MALE       

In [24]:
# =============================================================================
# CELL 15 — Fuzzy match unmatched GPs against 2005 sarpanch publication
# =============================================================================

# ── Clean the 2005 file ───────────────────────────────────────────────────────
sp2005 = sarpanch_2005.dropna(subset=["DISTRICT","GRAM PANCHAYAT"]).copy()
sp2005["district_norm"]  = sp2005["DISTRICT"].apply(normalize_name)
sp2005["gp_norm"]        = sp2005["GRAM PANCHAYAT"].apply(normalize_name)
sp2005["samiti_norm"]    = sp2005["PANCHAYAT SAMITI"].apply(normalize_name)
sp2005["reserved_women_2005"] = sp2005["CATEG. OF POST OF SARPANCH"].apply(
    lambda x: 1 if str(x).strip().upper().endswith("W") else 0
)

# Apply same district aliases
sp2005["district_norm_hist"] = sp2005["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)

print(f"2005 sarpanch records: {len(sp2005)}")
print(f"Unique districts: {sp2005['district_norm_hist'].nunique()}")
print(f"Unique GPs: {sp2005['gp_norm'].nunique()}")
print(f"Reserved for women: {sp2005['reserved_women_2005'].sum()}")

# ── Get unmatched GPs (all 1,719) ────────────────────────────────────────────
unmatched_gps_df = export_df[["gp_lgd_code","gp_name","district",
                               "subdistrict_samiti"]].copy()
unmatched_gps_df["gp_norm"] = unmatched_gps_df["gp_name"].apply(normalize_name)
unmatched_gps_df["district_norm_hist"] = unmatched_gps_df["district"].apply(
    normalize_district_historical
)

print(f"\nUnmatched GPs to search: {len(unmatched_gps_df)}")

# ── Build 2005 lookup: unique reservation status per GP+district ──────────────
# Check if GP name is unique within district in 2005 data
sp2005_by_district = (
    sp2005.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_rows              = ("gp_norm",            "size"),
        reserved_women_2005 = ("reserved_women_2005","first"),
        samiti              = ("samiti_norm",        "first"),
        categ               = ("CATEG. OF POST OF SARPANCH", "first")
    )
    .reset_index()
)

print(f"\n2005 unique district-GP pairs: {len(sp2005_by_district)}")

# ── Fuzzy match each unmatched GP ─────────────────────────────────────────────
FUZZY_THRESHOLD_2005 = 0.85  # slightly lower than Stage E to catch more

results_2005 = []

for _, gp_row in unmatched_gps_df.iterrows():
    dist    = gp_row["district_norm_hist"]
    gp_n    = gp_row["gp_norm"]
    lgd_code = gp_row["gp_lgd_code"]

    # Filter 2005 data to same district
    candidates = sp2005_by_district[
        sp2005_by_district["district_norm_hist"] == dist
    ].copy()

    if candidates.empty:
        results_2005.append({
            "gp_lgd_code":        lgd_code,
            "gp_name":            gp_row["gp_name"],
            "district":           gp_row["district"],
            "subdistrict_samiti": gp_row["subdistrict_samiti"],
            "best_match_gp":      None,
            "best_match_samiti":  None,
            "similarity":         0.0,
            "reserved_women_2005":None,
            "categ_2005":         None,
            "match_found":        False
        })
        continue

    candidates["sim"] = candidates["gp_norm"].apply(
        lambda x: similarity(gp_n, x)
    )
    best = candidates.nlargest(1, "sim").iloc[0]

    match_found = best["sim"] >= FUZZY_THRESHOLD_2005

    results_2005.append({
        "gp_lgd_code":        lgd_code,
        "gp_name":            gp_row["gp_name"],
        "district":           gp_row["district"],
        "subdistrict_samiti": gp_row["subdistrict_samiti"],
        "best_match_gp":      best["gp_norm"],
        "best_match_samiti":  best["samiti"],
        "similarity":         best["sim"],
        "reserved_women_2005":best["reserved_women_2005"] if match_found else None,
        "categ_2005":         best["categ"] if match_found else None,
        "match_found":        match_found
    })

results_2005_df = pd.DataFrame(results_2005)

n_matched = results_2005_df["match_found"].sum()
print(f"\n2005 fuzzy matches found (sim >= {FUZZY_THRESHOLD_2005}): {n_matched}")
print(f"No match found: {(~results_2005_df['match_found']).sum()}")

print(f"\nSimilarity distribution among matched:")
print(results_2005_df[results_2005_df["match_found"]]["similarity"].describe())

print(f"\nSample matches (first 20):")
print(results_2005_df[results_2005_df["match_found"]][[
    "gp_lgd_code","gp_name","district",
    "best_match_gp","similarity","categ_2005","reserved_women_2005"
]].head(20).to_string(index=False))

print(f"\nSample non-matches (first 10):")
print(results_2005_df[~results_2005_df["match_found"]][[
    "gp_lgd_code","gp_name","district",
    "best_match_gp","similarity"
]].head(10).to_string(index=False))

2005 sarpanch records: 9175
Unique districts: 32
Unique GPs: 8228
Reserved for women: 3071

Unmatched GPs to search: 1719

2005 unique district-GP pairs: 9035

2005 fuzzy matches found (sim >= 0.85): 732
No match found: 987

Similarity distribution among matched:
count    732.000000
mean       0.955100
std        0.052708
min        0.857143
25%        0.909091
50%        1.000000
75%        1.000000
max        1.000000
Name: similarity, dtype: float64

Sample matches (first 20):
gp_lgd_code          gp_name         district  best_match_gp  similarity categ_2005  reserved_women_2005
      35853 Motoron Ka Khera         Bhilwara motoronkakhera    1.000000        GEN                  0.0
      40277             Doti             Kota          dhoti    0.888889        GEN                  0.0
      40314          Gadepan             Kota        gadepan    1.000000        GEN                  0.0
      40957         Roopawas             Pali       roopawas    1.000000         SC            

In [25]:
# =============================================================================
# Check: are the 732 matched GPs unique within their district in 2005 data?
# =============================================================================

matched_2005 = results_2005_df[results_2005_df["match_found"]].copy()

# Add district_norm_hist from unmatched_gps_df
matched_2005 = matched_2005.merge(
    unmatched_gps_df[["gp_lgd_code","district_norm_hist"]],
    on="gp_lgd_code", how="left"
)

# Join back to sp2005 to count rows per matched GP+district
match_counts = (
    sp2005.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_rows         = ("gp_norm", "size"),
        n_categ_values = ("CATEG. OF POST OF SARPANCH", "nunique"),
        categ_values   = ("CATEG. OF POST OF SARPANCH",
                          lambda x: " | ".join(sorted(set(x.dropna()))))
    )
    .reset_index()
)

matched_2005 = matched_2005.merge(
    match_counts.rename(columns={"gp_norm": "best_match_gp"}),
    on=["district_norm_hist", "best_match_gp"],
    how="left"
)

print(f"Total matched GPs: {len(matched_2005)}")
print(f"\nRow count distribution in 2005 file for matched GPs:")
print(matched_2005["n_rows"].value_counts().sort_index())

print(f"\nMatched GPs with unique entry (n_rows == 1): "
      f"{(matched_2005['n_rows'] == 1).sum()}")
print(f"Matched GPs with multiple entries (n_rows > 1): "
      f"{(matched_2005['n_rows'] > 1).sum()}")

multi = matched_2005[matched_2005["n_rows"] > 1]
conflicted_2005 = multi[multi["n_categ_values"] > 1]
consistent_2005 = multi[multi["n_categ_values"] == 1]

print(f"\nAmong multi-row matched GPs:")
print(f"  Consistent reservation status (safe):    {len(consistent_2005)}")
print(f"  Conflicting reservation status (unsafe): {len(conflicted_2005)}")

print(f"\nSample conflicted:")
print(conflicted_2005[[
    "gp_lgd_code","gp_name","district",
    "best_match_gp","n_rows","categ_values"
]].head(10).to_string(index=False))

safe_2005 = matched_2005[
    (matched_2005["n_rows"] == 1) |
    (matched_2005["n_categ_values"] == 1)
].copy()

unsafe_2005 = matched_2005[
    matched_2005["n_categ_values"] > 1
].copy()

print(f"\nSafe matches (unique or consistent): {len(safe_2005)}")
print(f"Unsafe matches (conflicting):        {len(unsafe_2005)}")

Total matched GPs: 732

Row count distribution in 2005 file for matched GPs:
n_rows
1    702
2     29
3      1
Name: count, dtype: int64

Matched GPs with unique entry (n_rows == 1): 702
Matched GPs with multiple entries (n_rows > 1): 30

Among multi-row matched GPs:
  Consistent reservation status (safe):    6
  Conflicting reservation status (unsafe): 24

Sample conflicted:
gp_lgd_code     gp_name         district best_match_gp  n_rows  categ_values
      40957    Roopawas             Pali      roopawas       2      GEN | SC
      42162       Deoli             Tonk         deoli       2   GEN W | OBC
      35911      Nandsa         Bhilwara        nandsa       2    OBC W | SC
      36474       Shadi      Chittorgarh         shadi       2    GEN W | ST
      36939     Melusar            Churu       melusar       2      GEN | SC
      42630       Varni          Udaipur         varni       2     ST | ST W
      40323   Madanpura             Kota     madanpura       2   GEN | OBC W
     

In [26]:
# =============================================================================
# CELL 16 — Load and clean 2010 sarpanch publication
# =============================================================================

sarpanch_2010_raw = pd.read_excel(
    "/Users/sonalideliwala/Downloads/Publication_sarpanch_2010.xlsx",
    header=None
)

print(f"Raw shape: {sarpanch_2010_raw.shape}")
print(f"\nFirst 55 rows to verify structure:")
print(sarpanch_2010_raw.head(55).to_string())

/Users/sonalideliwala/Library/Python/3.12/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Raw shape: (10140, 7)

First 55 rows to verify structure:
                              0                 1               2              3                    4    5         6
0   DETAILS OF ELECTED SARPANCH               NaN             NaN            NaN                  NaN  NaN       NaN
1                      DISTRICT  PANCHAYAT SAMITI  GRAM PANCHAYAT  Ward category     Name of SARPANCH  Sex  Category
2                         AJMER              ARAI          AJGARA           GENW         Madhu kanwar    F      GENW
3                         AJMER              ARAI         AKODIYA           GENW                 lada    F      OBCW
4                         AJMER              ARAI            ARAI            GEN        Bhanwar Gopal    M       GEN
5                         AJMER              ARAI     BHAGWANPURA             SC       Prahalad Balai    M        SC
6                         AJMER              ARAI        BHAMOLAW           GENW  Ajay Durgesh Kanwar    F      GENW
7     

In [27]:
# =============================================================================
# CELL 16 (continued) — Clean 2010 sarpanch file
# =============================================================================

# Rename columns using the header row pattern
sarpanch_2010_raw.columns = [
    "DISTRICT", "PANCHAYAT SAMITI", "GRAM PANCHAYAT",
    "Ward category", "Name of SARPANCH", "Sex", "Category"
]

# Filter out non-data rows:
# - Title rows: column 0 == "DETAILS OF ELECTED SARPANCH"
# - Header rows: column 0 == "DISTRICT"
# - Page number rows: column 0 is numeric
# - Blank rows: column 0 is NaN

def is_data_row(row):
    val = str(row["DISTRICT"]).strip()
    if val in ["DETAILS OF ELECTED SARPANCH", "DISTRICT", "nan", ""]:
        return False
    try:
        float(val)  # page number
        return False
    except ValueError:
        return True

sp2010 = sarpanch_2010_raw[
    sarpanch_2010_raw.apply(is_data_row, axis=1)
].copy().reset_index(drop=True)

print(f"Clean 2010 sarpanch records: {len(sp2010)}")
print(f"\nSample rows:")
print(sp2010.head(10).to_string(index=False))

# ── Normalize ─────────────────────────────────────────────────────────────────
sp2010["district_norm"]      = sp2010["DISTRICT"].apply(normalize_name)
sp2010["gp_norm"]            = sp2010["GRAM PANCHAYAT"].apply(normalize_name)
sp2010["samiti_norm"]        = sp2010["PANCHAYAT SAMITI"].apply(normalize_name)
sp2010["district_norm_hist"] = sp2010["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)
sp2010["reserved_women_2010"] = sp2010["Ward category"].apply(
    lambda x: 1 if str(x).strip().upper().endswith("W") else 0
)

print(f"\nUnique districts: {sp2010['district_norm_hist'].nunique()}")
print(f"Unique GPs:       {sp2010['gp_norm'].nunique()}")
print(f"Reserved for women: {sp2010['reserved_women_2010'].sum()}")
print(f"\nWard category distribution (top 10):")
print(sp2010["Ward category"].value_counts().head(10))

# ── Build 2010 lookup ─────────────────────────────────────────────────────────
sp2010_by_district = (
    sp2010.groupby(["district_norm_hist", "gp_norm"])
    .agg(
        n_rows              = ("gp_norm",             "size"),
        n_categ_values      = ("Ward category",       "nunique"),
        categ_values        = ("Ward category",
                               lambda x: " | ".join(sorted(set(
                                   str(v).strip() for v in x.dropna()
                               )))),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        samiti              = ("samiti_norm",          "first")
    )
    .reset_index()
)

print(f"\n2010 unique district-GP pairs: {len(sp2010_by_district)}")

# ── Fuzzy match against unmatched GPs ────────────────────────────────────────
FUZZY_THRESHOLD_2010 = 0.85

results_2010 = []

for _, gp_row in unmatched_gps_df.iterrows():
    dist     = gp_row["district_norm_hist"]
    gp_n     = gp_row["gp_norm"]
    lgd_code = gp_row["gp_lgd_code"]

    candidates = sp2010_by_district[
        sp2010_by_district["district_norm_hist"] == dist
    ].copy()

    if candidates.empty:
        results_2010.append({
            "gp_lgd_code":        lgd_code,
            "best_match_gp":      None,
            "best_match_samiti":  None,
            "similarity":         0.0,
            "reserved_women_2010":None,
            "categ_2010":         None,
            "n_rows":             0,
            "n_categ_values":     0,
            "match_found":        False
        })
        continue

    candidates["sim"] = candidates["gp_norm"].apply(
        lambda x: similarity(gp_n, x)
    )
    best = candidates.nlargest(1, "sim").iloc[0]
    match_found = best["sim"] >= FUZZY_THRESHOLD_2010

    results_2010.append({
        "gp_lgd_code":        lgd_code,
        "best_match_gp":      best["gp_norm"],
        "best_match_samiti":  best["samiti"],
        "similarity":         best["sim"],
        "reserved_women_2010":best["reserved_women_2010"] if match_found else None,
        "categ_2010":         best["categ_values"] if match_found else None,
        "n_rows":             best["n_rows"] if match_found else 0,
        "n_categ_values":     best["n_categ_values"] if match_found else 0,
        "match_found":        match_found
    })

results_2010_df = pd.DataFrame(results_2010)

n_matched_2010 = results_2010_df["match_found"].sum()
print(f"\n2010 fuzzy matches found (sim >= {FUZZY_THRESHOLD_2010}): {n_matched_2010}")
print(f"No match found: {(~results_2010_df['match_found']).sum()}")

print(f"\nSimilarity distribution among matched:")
print(results_2010_df[results_2010_df["match_found"]]["similarity"].describe())

# ── Check uniqueness / conflicts among 2010 matches ──────────────────────────
matched_2010 = results_2010_df[results_2010_df["match_found"]].copy()
conflicted_2010 = matched_2010[matched_2010["n_categ_values"] > 1]
safe_2010 = matched_2010[matched_2010["n_categ_values"] <= 1]

print(f"\nSafe 2010 matches (unique or consistent): {len(safe_2010)}")
print(f"Conflicted 2010 matches:                  {len(conflicted_2010)}")

print(f"\nSample 2010 matches:")
print(matched_2010[[
    "gp_lgd_code","best_match_gp","similarity",
    "categ_2010","reserved_women_2010"
]].head(20).to_string(index=False))

Clean 2010 sarpanch records: 9166

Sample rows:
DISTRICT PANCHAYAT SAMITI GRAM PANCHAYAT Ward category    Name of SARPANCH Sex Category
   AJMER             ARAI         AJGARA          GENW        Madhu kanwar   F     GENW
   AJMER             ARAI        AKODIYA          GENW                lada   F     OBCW
   AJMER             ARAI           ARAI           GEN       Bhanwar Gopal   M      GEN
   AJMER             ARAI    BHAGWANPURA            SC      Prahalad Balai   M       SC
   AJMER             ARAI       BHAMOLAW          GENW Ajay Durgesh Kanwar   F     GENW
   AJMER             ARAI      BHOGADEET           OBC              Ramdev   M      OBC
   AJMER             ARAI          BIRLA          OBCW          Heera Devi   F     OBCW
   AJMER             ARAI         BORADA           GEN           Ram Singh   M      OBC
   AJMER             ARAI   CHHOTA LAMBA          GENW        kamla Bairwa   F      SCW
   AJMER             ARAI         DADIYA           sew          Sugni De

In [28]:
# =============================================================================
# CELL 17 — Combine 2005 and 2010 matches to get full reservation dose
# =============================================================================

# ── Start with safe matches from both years ───────────────────────────────────
safe_2005_df = safe_2005[["gp_lgd_code","reserved_women_2005",
                           "similarity"]].copy()
safe_2005_df = safe_2005_df.rename(columns={"similarity":"sim_2005"})

safe_2010_df = results_2010_df[
    results_2010_df["match_found"] &
    (results_2010_df["n_categ_values"] <= 1)
][["gp_lgd_code","reserved_women_2010","similarity"]].copy()
safe_2010_df = safe_2010_df.rename(columns={"similarity":"sim_2010"})

# ── Merge both years ──────────────────────────────────────────────────────────
combined = unmatched_gps_df[["gp_lgd_code","gp_name","district",
                              "subdistrict_samiti"]].merge(
    safe_2005_df, on="gp_lgd_code", how="left"
).merge(
    safe_2010_df, on="gp_lgd_code", how="left"
)

# Coverage summary
both_years   = combined["reserved_women_2005"].notna() & combined["reserved_women_2010"].notna()
only_2005    = combined["reserved_women_2005"].notna() & combined["reserved_women_2010"].isna()
only_2010    = combined["reserved_women_2005"].isna()  & combined["reserved_women_2010"].notna()
neither      = combined["reserved_women_2005"].isna()  & combined["reserved_women_2010"].isna()

print(f"Coverage summary for {len(combined)} unmatched GPs:")
print(f"  Matched in both years:   {both_years.sum()}")
print(f"  Matched in 2005 only:    {only_2005.sum()}")
print(f"  Matched in 2010 only:    {only_2010.sum()}")
print(f"  Matched in neither year: {neither.sum()}")

# ── Accept only GPs matched safely in BOTH years ─────────────────────────────
# This is the most conservative and verifiable approach
both_matched = combined[both_years].copy()
both_matched["reserved_women_2005"] = both_matched["reserved_women_2005"].astype(int)
both_matched["reserved_women_2010"] = both_matched["reserved_women_2010"].astype(int)
both_matched["reservation_dose_n"]  = (
    both_matched["reserved_women_2005"] + both_matched["reserved_women_2010"]
)
both_matched["reservation_dose"] = both_matched["reservation_dose_n"].map(
    DOSE_LABEL_MAP
)

print(f"\nGPs with complete dose from both-year match: {len(both_matched)}")
print(f"\nDose distribution:")
print(both_matched["reservation_dose"].value_counts())

print(f"\nSimilarity score summary:")
print(f"  Mean 2005 similarity: {both_matched['sim_2005'].mean():.3f}")
print(f"  Mean 2010 similarity: {both_matched['sim_2010'].mean():.3f}")
print(f"  Min 2005 similarity:  {both_matched['sim_2005'].min():.3f}")
print(f"  Min 2010 similarity:  {both_matched['sim_2010'].min():.3f}")

print(f"\nSample matched GPs:")
print(both_matched[[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "reserved_women_2005","reserved_women_2010",
    "reservation_dose","sim_2005","sim_2010"
]].head(20).to_string(index=False))

Coverage summary for 1719 unmatched GPs:
  Matched in both years:   493
  Matched in 2005 only:    215
  Matched in 2010 only:    190
  Matched in neither year: 821

GPs with complete dose from both-year match: 493

Dose distribution:
reservation_dose
once     234
never    177
twice     82
Name: count, dtype: int64

Similarity score summary:
  Mean 2005 similarity: 0.953
  Mean 2010 similarity: 0.919
  Min 2005 similarity:  0.857
  Min 2010 similarity:  0.857

Sample matched GPs:
gp_lgd_code          gp_name         district subdistrict_samiti  reserved_women_2005  reserved_women_2010 reservation_dose  sim_2005  sim_2010
      40314          Gadepan             Kota              Digod                    0                    1             once  1.000000  0.933333
      36323            Maran            Bundi             Nainwa                    0                    1             once  1.000000  0.888889
      42214          Kholiya             Tonk             Uniara                   

In [29]:
# =============================================================================
# CELL 18 — Add both-year matches to GP reservation history and rerun pipeline
# =============================================================================

# ── Build Stage F entries ─────────────────────────────────────────────────────
stage_f = both_matched[[
    "gp_lgd_code", "reserved_women_2005", "reserved_women_2010",
    "reservation_dose_n", "reservation_dose"
]].copy()

# Add GP name info from gp_lookup
stage_f = stage_f.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]],
    on="gp_lgd_code", how="left"
)
stage_f["match_stage"] = "F_fuzzy_sarpanch_both_years"

# Confirm no overlap with existing gp_res_history_final
already_matched = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_f["gp_lgd_code"]) & already_matched
print(f"Stage F GPs already in history (should be 0): {len(overlap)}")

stage_f = stage_f[~stage_f["gp_lgd_code"].isin(already_matched)].copy()
print(f"Stage F new GPs to add: {len(stage_f)}")
print(f"\nDose distribution:")
print(stage_f["reservation_dose"].value_counts())

# ── Append to gp_res_history_final ───────────────────────────────────────────
gp_res_history_final_v3 = pd.concat([
    gp_res_history_final,
    stage_f[[
        "gp_lgd_code", "gp_name_lgd", "district_lgd", "subdistrict_lgd",
        "reserved_women_2005", "reserved_women_2010",
        "reservation_dose_n", "reservation_dose", "match_stage"
    ]]
], ignore_index=True)

assert gp_res_history_final_v3["gp_lgd_code"].duplicated().sum() == 0, \
    "Duplicate GP LGD codes!"

print(f"\nGP reservation history:")
print(f"  Before Stage F: {len(gp_res_history_final)} GPs")
print(f"  Stage F added:  {len(stage_f)} GPs")
print(f"  Total:          {len(gp_res_history_final_v3)} GPs")
print(f"\nFull match stage breakdown:")
print(gp_res_history_final_v3["match_stage"].value_counts())

# ── Rebuild cluster_gp_res ────────────────────────────────────────────────────
cluster_gp_res_v3 = mc_long.merge(
    gp_res_history_final_v3[[
        "gp_lgd_code", "reservation_dose", "reservation_dose_n",
        "reserved_women_2005", "reserved_women_2010",
        "gp_name_lgd", "district_lgd", "subdistrict_lgd", "match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

print(f"\nRebuilt cluster_gp_res:")
print(f"  Matched rows:            {cluster_gp_res_v3['reservation_dose'].notna().sum()}")
print(f"  Unmatched rows:          {cluster_gp_res_v3['reservation_dose'].isna().sum()}")
print(f"  Clusters with any match: "
      f"{cluster_gp_res_v3[cluster_gp_res_v3['reservation_dose'].notna()]['DHSCLUST'].nunique()}")

# ── Recompute cluster-level treatment probabilities ───────────────────────────
cluster_known_v3 = (
    cluster_gp_res_v3[cluster_gp_res_v3["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v3.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v3.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v3.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v3 = all_clusters.merge(
    cluster_known_v3, on="DHSCLUST", how="left"
)
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v3[col] = treat_probs_v3[col].fillna(0.0)

treat_probs_v3["p_unknown_treatment_mc"] = (
    1 - treat_probs_v3["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v3["p_any_reserved_mc"]    = (
    treat_probs_v3["p_once_mc"] + treat_probs_v3["p_twice_mc"]
)
treat_probs_v3["expected_dose_mc"]     = (
    treat_probs_v3["p_once_mc"] + 2 * treat_probs_v3["p_twice_mc"]
)
treat_probs_v3["treatment_certainty_mc"] = treat_probs_v3[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

# Primary GP dose
primary_dose_v3 = (
    cluster_gp_res_v3[cluster_gp_res_v3["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v3 = treat_probs_v3.merge(
    primary_dose_v3, on="DHSCLUST", how="left"
)

# Primary GP matched flag
treat_probs_v3["primary_gp_norm"] = treat_probs_v3["primary_gp"].apply(
    normalize_lgd_code
)
treat_probs_v3["primary_gp_has_history"] = treat_probs_v3["primary_gp_norm"].isin(
    set(gp_res_history_final_v3["gp_lgd_code"])
)

# ── Working cluster summary ───────────────────────────────────────────────────
working_v3 = treat_probs_v3[treat_probs_v3["primary_gp_has_history"]].copy()

print(f"\nWorking clusters (primary GP matched): {len(working_v3)}")
print(f"  Fully linked (>=99.9%): "
      f"{(working_v3['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  >=90% linked:           "
      f"{(working_v3['p_known_treatment_mc'] >= 0.90).sum()}")
print(f"  >=75% linked:           "
      f"{(working_v3['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known:           "
      f"{working_v3['p_known_treatment_mc'].mean():.3f}")
print(f"  Mean treatment certainty: "
      f"{working_v3['treatment_certainty_mc'].mean():.3f}")

bins   = [0, 0.25, 0.50, 0.75, 0.90, 0.999, 1.01]
labels = ["0-25%","25-50%","50-75%","75-90%","90-99%","100%"]
working_v3["known_mass_bin"] = pd.cut(
    working_v3["p_known_treatment_mc"],
    bins=bins, labels=labels, right=False
)
print(f"\nKnown mass bin distribution:")
print(working_v3["known_mass_bin"].value_counts().sort_index())

# Spot check
print(f"\nSpot check — cluster 290573:")
print(treat_probs_v3[treat_probs_v3["DHSCLUST"]==290573][[
    "DHSCLUST","p_known_treatment_mc","p_never_mc","primary_gp_dose"
]].to_string(index=False))

# ── Update working variables ──────────────────────────────────────────────────
gp_res_history_final = gp_res_history_final_v3
cluster_gp_res       = cluster_gp_res_v3
treat_probs_new      = treat_probs_v3
working_clusters_new = working_v3

# Save
gp_res_history_final.to_csv(
    OUTPUT_DIR / "gp_reservation_history.csv", index=False
)
cluster_gp_res.to_csv(
    OUTPUT_DIR / "cluster_gp_res_long.csv", index=False
)
print(f"\nSaved updated files.")

Stage F GPs already in history (should be 0): 0
Stage F new GPs to add: 493

Dose distribution:
reservation_dose
once     234
never    177
twice     82
Name: count, dtype: int64

GP reservation history:
  Before Stage F: 4223 GPs
  Stage F added:  493 GPs
  Total:          4716 GPs

Full match stage breakdown:
match_stage
A_exact_gp_district_samiti     2277
B_exact_gp_district            1514
F_fuzzy_sarpanch_both_years     493
C_unique_gp_name_statewide      185
D_district_only_unique_dose     177
E_fuzzy_gp_name_district         70
Name: count, dtype: int64

Rebuilt cluster_gp_res:
  Matched rows:            4636
  Unmatched rows:          5097
  Clusters with any match: 1102

Working clusters (primary GP matched): 577
  Fully linked (>=99.9%): 46
  >=90% linked:           200
  >=75% linked:           354
  Mean p_known:           0.792
  Mean treatment certainty: 0.606

Known mass bin distribution:
known_mass_bin
0-25%       0
25-50%     30
50-75%    193
75-90%    154
90-99%    154

In [30]:
# =============================================================================
# CELL 19 — Update final outputs with Stage F improvements
# =============================================================================

# ── Update final_treat_probs ──────────────────────────────────────────────────
final_treat_probs = working_clusters_new[[
    "DHSCLUST", "DHSREGNA",
    "p_never_mc", "p_once_mc", "p_twice_mc",
    "p_any_reserved_mc", "expected_dose_mc",
    "p_known_treatment_mc", "p_unknown_treatment_mc",
    "treatment_certainty_mc",
    "primary_gp", "primary_gp_prob", "primary_gp_dose",
    "n_gps_hit"
]].copy()

# ── Recompute isolated clusters and analysis clusters ─────────────────────────
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)
analysis_clusters = set(final_treat_probs["DHSCLUST"]) - isolated_clusters

print(f"Working clusters:     {len(final_treat_probs)}")
print(f"Isolated clusters:    {len(isolated_clusters)}")
print(f"Analysis clusters:    {len(analysis_clusters)}")

# ── Rebuild unmatched GP export ───────────────────────────────────────────────
unmatched = cluster_gp_res[
    cluster_gp_res["DHSCLUST"].isin(analysis_clusters) &
    cluster_gp_res["gp_lgd_code"].notna() &
    cluster_gp_res["reservation_dose"].isna()
].copy()

unmatched = unmatched.merge(
    final_treat_probs[["DHSCLUST","p_known_treatment_mc"]],
    on="DHSCLUST", how="left"
)

gp_summary = (
    unmatched
    .groupby("gp_lgd_code", as_index=False)
    .agg(
        total_prob_mass             = ("gp_prob",              "sum"),
        n_clusters                  = ("DHSCLUST",             "nunique"),
        max_known_mass_in_clusters  = ("p_known_treatment_mc", "max"),
        mean_known_mass_in_clusters = ("p_known_treatment_mc", "mean"),
        min_known_mass_in_clusters  = ("p_known_treatment_mc", "min"),
    )
    .sort_values("max_known_mass_in_clusters", ascending=False)
    .reset_index(drop=True)
)

gp_summary["known_mass_bin"] = (
    gp_summary["max_known_mass_in_clusters"].apply(assign_bin)
)

print(f"\nUnmatched GP bin distribution:")
print(gp_summary["known_mass_bin"].value_counts().sort_index())

export_df = (
    gp_summary
    .merge(gp_names, on="gp_lgd_code", how="left")
    [[
        "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
        "total_prob_mass", "n_clusters", "known_mass_bin",
        "max_known_mass_in_clusters", "mean_known_mass_in_clusters",
        "min_known_mass_in_clusters",
    ]]
    .reset_index(drop=True)
)

def assign_tier(bin_label):
    if bin_label in ["90-99%+","90-99%"]: return "Tier 1 — pushes cluster to fully linked"
    elif bin_label == "75-90%":            return "Tier 2 — high priority"
    elif bin_label == "50-75%":            return "Tier 3 — medium priority"
    else:                                  return "Tier 4 — lower priority"

export_df["lookup_tier"] = export_df["known_mass_bin"].apply(assign_tier)

tier_order = {
    "Tier 1 — pushes cluster to fully linked": 0,
    "Tier 2 — high priority":                  1,
    "Tier 3 — medium priority":                2,
    "Tier 4 — lower priority":                 3,
}
export_df["tier_sort"] = export_df["lookup_tier"].map(tier_order)
export_df = export_df.sort_values(
    ["tier_sort","max_known_mass_in_clusters"],
    ascending=[True, False]
).drop(columns="tier_sort").reset_index(drop=True)

export_df["reserved_women_2005"] = ""
export_df["reserved_women_2010"] = ""
export_df["notes"]               = ""

export_cols = [
    "lookup_tier","gp_lgd_code","gp_name","district","subdistrict_samiti",
    "known_mass_bin","max_known_mass_in_clusters","n_clusters",
    "total_prob_mass","mean_known_mass_in_clusters",
    "reserved_women_2005","reserved_women_2010","notes"
]
export_df = export_df[export_cols]

print(f"\nExport summary:")
print(f"  Tier 1 (90-99%+/90-99%): {(export_df['lookup_tier'].str.contains('Tier 1')).sum()} GPs")
print(f"  Tier 2 (75-90%):          {(export_df['lookup_tier'].str.contains('Tier 2')).sum()} GPs")
print(f"  Tier 3 (50-75%):          {(export_df['lookup_tier'].str.contains('Tier 3')).sum()} GPs")
print(f"  Total:                    {len(export_df)} GPs")

print(f"\nTier 1 GPs (first 15):")
print(export_df[export_df["lookup_tier"].str.contains("Tier 1")][[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "max_known_mass_in_clusters","n_clusters"
]].head(15).to_string(index=False))

# ── Save all outputs ──────────────────────────────────────────────────────────
final_treat_probs.to_csv(
    OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False
)

from openpyxl.styles import PatternFill
tier_fills = {
    "Tier 1": PatternFill("solid", fgColor="C6EFCE"),
    "Tier 2": PatternFill("solid", fgColor="FFEB9C"),
    "Tier 3": PatternFill("solid", fgColor="DAEEF3"),
    "Tier 4": PatternFill("solid", fgColor="F2F2F2"),
}

OUTPUT_PATH = OUTPUT_DIR / "unmatched_gps_for_manual_review.xlsx"
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    export_df.to_excel(writer, index=False, sheet_name="Unmatched GPs")
    ws = writer.sheets["Unmatched GPs"]
    ws.freeze_panes = "A2"
    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        tier_val = str(row[0].value)
        fill = next(
            (f for k, f in tier_fills.items() if k in tier_val),
            tier_fills["Tier 4"]
        )
        for cell in row:
            cell.fill = fill

gp_res_history_final.to_csv(
    OUTPUT_DIR / "gp_reservation_history.csv", index=False
)
cluster_gp_res.to_csv(
    OUTPUT_DIR / "cluster_gp_res_long.csv", index=False
)

print(f"\nSaved all outputs to: {OUTPUT_DIR}")

# ── Final summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"FINAL SUMMARY — AFTER STAGE F")
print(f"{'='*65}")
print(f"  GP reservation histories:  {len(gp_res_history_final)}")
print(f"  Working clusters:          {len(final_treat_probs)}")
print(f"  Fully linked:              "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  >=75% linked:              "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"  Mean p_known:              "
      f"{final_treat_probs['p_known_treatment_mc'].mean():.3f}")
print(f"  Unmatched GPs remaining:   {len(export_df)}")
print(f"  Tier 1 priority GPs:       "
      f"{(export_df['lookup_tier'].str.contains('Tier 1')).sum()}")

Working clusters:     577
Isolated clusters:    125
Analysis clusters:    452

Unmatched GP bin distribution:
known_mass_bin
25-50%    134
50-75%    608
75-90%    328
90-99%    190
Name: count, dtype: int64

Export summary:
  Tier 1 (90-99%+/90-99%): 190 GPs
  Tier 2 (75-90%):          328 GPs
  Tier 3 (50-75%):          608 GPs
  Total:                    1260 GPs

Tier 1 GPs (first 15):
gp_lgd_code          gp_name        district subdistrict_samiti  max_known_mass_in_clusters  n_clusters
     236071           Genoli        Bhilwara         Mandalgarh                    0.992151           2
      35853 Motoron Ka Khera        Bhilwara         Mandalgarh                    0.992151           2
      36805       Kheenwasar           Churu              Churu                    0.986000           1
      36939          Melusar           Churu       Sardarshahar                    0.986000           1
     294348             Reda           Churu            Bidasar                    0.980

In [31]:
print(f"Fully linked clusters (p_known >= 99.9%): "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")

# What are the fully linked clusters?
fully_linked = final_treat_probs[
    final_treat_probs["p_known_treatment_mc"] >= 0.999
].copy()

print(f"\nDistrict breakdown of fully linked clusters:")
print(fully_linked.groupby("DHSREGNA")["DHSCLUST"].count()
      .sort_values(ascending=False).to_string())

print(f"\nDose distribution among fully linked clusters:")
print(f"  Mean p_never_mc:  {fully_linked['p_never_mc'].mean():.3f}")
print(f"  Mean p_once_mc:   {fully_linked['p_once_mc'].mean():.3f}")
print(f"  Mean p_twice_mc:  {fully_linked['p_twice_mc'].mean():.3f}")

print(f"\nPrimary GP dose distribution:")
print(fully_linked["primary_gp_dose"].value_counts(dropna=False))

Fully linked clusters (p_known >= 99.9%): 46

District breakdown of fully linked clusters:
DHSREGNA
Ajmer           4
Hanumangarh     4
Pali            4
Baran           4
Jaisalmer       4
Bhilwara        4
Chittaurgarh    3
Dausa           2
Bundi           2
Jaipur          2
Bikaner         2
Jhunjhunun      2
Kota            2
Rajsamand       2
Jalor           1
Jodhpur         1
Nagaur          1
Sikar           1
Sirohi          1

Dose distribution among fully linked clusters:
  Mean p_never_mc:  0.339
  Mean p_once_mc:   0.546
  Mean p_twice_mc:  0.114

Primary GP dose distribution:
primary_gp_dose
once     25
never    16
twice     5
Name: count, dtype: int64


In [32]:
# How many unique GPs underlie the 46 fully linked clusters?
fully_linked_dhsclust = set(
    final_treat_probs[
        final_treat_probs["p_known_treatment_mc"] >= 0.999
    ]["DHSCLUST"]
)

gps_in_fully_linked = cluster_gp_res[
    cluster_gp_res["DHSCLUST"].isin(fully_linked_dhsclust)
]

print(f"Fully linked clusters: {len(fully_linked_dhsclust)}")
print(f"Total GP-cluster rows: {len(gps_in_fully_linked)}")
print(f"Unique GPs in those clusters: {gps_in_fully_linked['gp_lgd_code'].nunique()}")
print(f"All matched (should all be non-null):")
print(f"  Matched rows: {gps_in_fully_linked['reservation_dose'].notna().sum()}")
print(f"  Unmatched rows: {gps_in_fully_linked['reservation_dose'].isna().sum()}")

Fully linked clusters: 46
Total GP-cluster rows: 236
Unique GPs in those clusters: 222
All matched (should all be non-null):
  Matched rows: 235
  Unmatched rows: 1


In [33]:
# Find the one unmatched row in fully linked clusters
mystery = gps_in_fully_linked[
    gps_in_fully_linked["reservation_dose"].isna()
]
print(mystery[["DHSCLUST","gp_lgd_code","gp_prob","reservation_dose"]].to_string(index=False))

# What is p_known_treatment_mc for that cluster?
print(f"\np_known for that cluster:")
print(final_treat_probs[
    final_treat_probs["DHSCLUST"].isin(mystery["DHSCLUST"])
][["DHSCLUST","p_known_treatment_mc"]].to_string(index=False))

 DHSCLUST gp_lgd_code  gp_prob reservation_dose
   290385       38776    0.001              NaN

p_known for that cluster:
 DHSCLUST  p_known_treatment_mc
   290385                 0.999


In [34]:
# Clusters closest to fully linked and what's blocking them
near_complete = final_treat_probs[
    (final_treat_probs["p_known_treatment_mc"] >= 0.90) &
    (final_treat_probs["p_known_treatment_mc"] < 0.999)
].copy()

print(f"Clusters at 90-99% known mass: {len(near_complete)}")

# For each, find the unmatched GPs and their probability mass
blocking_gps = []
for _, cluster in near_complete.iterrows():
    dhsclust = cluster["DHSCLUST"]
    unmatched_rows = cluster_gp_res[
        (cluster_gp_res["DHSCLUST"] == dhsclust) &
        (cluster_gp_res["reservation_dose"].isna()) &
        (cluster_gp_res["gp_lgd_code"].notna())
    ].copy()
    
    for _, gp_row in unmatched_rows.iterrows():
        blocking_gps.append({
            "DHSCLUST":               dhsclust,
            "p_known_treatment_mc":   cluster["p_known_treatment_mc"],
            "p_unknown_remaining":    1 - cluster["p_known_treatment_mc"],
            "gp_lgd_code":            gp_row["gp_lgd_code"],
            "gp_prob":                gp_row["gp_prob"],
        })

blocking_df = pd.DataFrame(blocking_gps)

# Add GP names
blocking_df = blocking_df.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]],
    on="gp_lgd_code", how="left"
)

print(f"\nTotal blocking GP rows: {len(blocking_df)}")
print(f"Unique blocking GPs:    {blocking_df['gp_lgd_code'].nunique()}")

# How many clusters have exactly 1 blocking GP?
n_blocking_per_cluster = blocking_df.groupby("DHSCLUST")["gp_lgd_code"].nunique()
print(f"\nBlocking GP count distribution:")
print(n_blocking_per_cluster.value_counts().sort_index())

# Clusters with exactly 1 blocking GP — highest priority
single_blocker_clusters = n_blocking_per_cluster[n_blocking_per_cluster == 1].index
single_blockers = blocking_df[
    blocking_df["DHSCLUST"].isin(single_blocker_clusters)
].copy()

print(f"\nClusters with exactly 1 blocking GP: {len(single_blocker_clusters)}")
print(f"\nSingle-blocker GPs (sorted by cluster p_known desc):")
print(single_blockers[[
    "DHSCLUST","p_known_treatment_mc","gp_lgd_code",
    "gp_name_lgd","district_lgd","subdistrict_lgd","gp_prob"
]].sort_values("p_known_treatment_mc", ascending=False).to_string(index=False))

Clusters at 90-99% known mass: 154

Total blocking GP rows: 271
Unique blocking GPs:    249

Blocking GP count distribution:
gp_lgd_code
1    73
2    56
3    17
4     5
5     3
Name: count, dtype: int64

Clusters with exactly 1 blocking GP: 73

Single-blocker GPs (sorted by cluster p_known desc):
 DHSCLUST  p_known_treatment_mc gp_lgd_code            gp_name_lgd     district_lgd subdistrict_lgd  gp_prob
   291247              0.998945      293955               Jilawara            Ajmer       Nasirabad 0.001055
   290679              0.998836       40294                Latoori             Kota          Sangod 0.001164
   290821              0.997970       35672                  Baran         Shahpura          Banera 0.002030
   290451              0.997696      294914             Sonthli(R)        Jhunjhunu       Nawalgarh 0.002304
   291339              0.997110      295032             Kushalgarh      Chittorgarh      Rawatbhata 0.002890
   291376              0.997000       34833    L

In [35]:
# Check best 2005 and 2010 fuzzy match scores for single-blocker GPs
single_blocker_lgd = set(single_blockers["gp_lgd_code"].astype(str))

# From our Stage E fuzzy match results
print("Single-blocker GPs and their best fuzzy match scores:")
print("\nFrom Stage E (res_clean fuzzy match):")

stage_e_check = unmatched_gps_df[
    unmatched_gps_df["gp_lgd_code"].isin(single_blocker_lgd)
].copy()

from difflib import SequenceMatcher

res_by_district_check = (
    res_clean.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_dose_values       = ("reservation_dose",    "nunique"),
        reservation_dose    = ("reservation_dose",    "first"),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        reserved_women_2010 = ("reserved_women_2010", "first"),
    )
    .reset_index()
)

stage_e_scores = []
for _, gp_row in stage_e_check.iterrows():
    dist   = gp_row["district_norm_hist"]
    gp_n   = gp_row["gp_norm"]
    lgd    = gp_row["gp_lgd_code"]

    cands = res_by_district_check[
        res_by_district_check["district_norm_hist"] == dist
    ].copy()
    if cands.empty:
        stage_e_scores.append({"gp_lgd_code": lgd, "gp_name": gp_row["gp_name"],
                               "district": gp_row["district"], "best_sim_res": 0,
                               "best_match_res": None, "dose_res": None})
        continue
    cands["sim"] = cands["gp_norm"].apply(lambda x: similarity(gp_n, x))
    best = cands.nlargest(1,"sim").iloc[0]
    stage_e_scores.append({
        "gp_lgd_code":  lgd,
        "gp_name":      gp_row["gp_name"],
        "district":     gp_row["district"],
        "best_sim_res": best["sim"],
        "best_match_res": best["gp_norm"],
        "dose_res":     best["reservation_dose"] if best["sim"] >= 0.80 else None,
        "n_dose_values":best["n_dose_values"]
    })

stage_e_df_check = pd.DataFrame(stage_e_scores).sort_values(
    "best_sim_res", ascending=False
)

print(stage_e_df_check[[
    "gp_lgd_code","gp_name","district",
    "best_sim_res","best_match_res","dose_res","n_dose_values"
]].to_string(index=False))

# How many would be recovered at different thresholds?
for thresh in [0.85, 0.80, 0.75, 0.70]:
    n = (stage_e_df_check["best_sim_res"] >= thresh).sum()
    n_unique_dose = (
        (stage_e_df_check["best_sim_res"] >= thresh) &
        (stage_e_df_check["n_dose_values"] == 1)
    ).sum()
    print(f"Threshold {thresh}: {n} matches ({n_unique_dose} with unique dose)")

Single-blocker GPs and their best fuzzy match scores:

From Stage E (res_clean fuzzy match):
gp_lgd_code                gp_name      district  best_sim_res     best_match_res dose_res  n_dose_values
      40323              Madanpura          Kota      1.000000          madanpura    never              2
      40002           Somala Ratra       Karauli      0.952381         somlaratra    never              1
      33641               Padaliya         Kekri      0.933333            padliya     once              1
      36227 Dhbhaiyon Ka Naya Gaon         Bundi      0.918919 dhabayonkanayagaon    never              1
      34669                  Bohat         Baran      0.909091             bohath     once              1
      36086                  Kotri       Bikaner      0.909091             kotari     once              1
      33776             Satawariya        Beawar      0.900000         satawadiya    never              1
      36763                   Arni   Chittorgarh      0.888

In [36]:
# =============================================================================
# Check district coverage: which LGD districts have no match in reservation data?
# =============================================================================

# Districts in LGD (after alias mapping)
lgd_districts = set(gp_lookup["district_norm_hist"].unique())

# Districts in reservation file (after alias mapping)  
res_districts = set(res_clean["district_norm_hist"].unique())

# Districts in 2005 sarpanch file
sp2005_districts = set(sp2005["district_norm_hist"].unique())

# Districts in 2010 sarpanch file
sp2010_districts = set(sp2010["district_norm_hist"].unique())

print(f"LGD districts (after aliases):        {len(lgd_districts)}")
print(f"Reservation file districts:            {len(res_districts)}")
print(f"2005 sarpanch file districts:          {len(sp2005_districts)}")
print(f"2010 sarpanch file districts:          {len(sp2010_districts)}")

print(f"\nLGD districts NOT in reservation file:")
missing_res = lgd_districts - res_districts
for d in sorted(missing_res):
    n_gps = (gp_lookup["district_norm_hist"] == d).sum()
    print(f"  {d}: {n_gps} GPs in LGD")

print(f"\nLGD districts NOT in 2005 sarpanch file:")
missing_2005 = lgd_districts - sp2005_districts
for d in sorted(missing_2005):
    n_gps = (gp_lookup["district_norm_hist"] == d).sum()
    print(f"  {d}: {n_gps} GPs in LGD")

print(f"\nLGD districts NOT in 2010 sarpanch file:")
missing_2010 = lgd_districts - sp2010_districts
for d in sorted(missing_2010):
    n_gps = (gp_lookup["district_norm_hist"] == d).sum()
    print(f"  {d}: {n_gps} GPs in LGD")

# Also check single-blocker GPs specifically
print(f"\nSingle-blocker GPs by district coverage:")
single_blockers_dist = single_blockers.copy()
single_blockers_dist["district_norm_hist"] = single_blockers_dist["district_lgd"].apply(
    normalize_district_historical
)
single_blockers_dist["in_res"]  = single_blockers_dist["district_norm_hist"].isin(res_districts)
single_blockers_dist["in_2005"] = single_blockers_dist["district_norm_hist"].isin(sp2005_districts)
single_blockers_dist["in_2010"] = single_blockers_dist["district_norm_hist"].isin(sp2010_districts)

print(f"  District in reservation file: {single_blockers_dist['in_res'].sum()} of {len(single_blockers_dist)}")
print(f"  District in 2005 sarpanch:    {single_blockers_dist['in_2005'].sum()} of {len(single_blockers_dist)}")
print(f"  District in 2010 sarpanch:    {single_blockers_dist['in_2010'].sum()} of {len(single_blockers_dist)}")

print(f"\nSingle-blocker GPs whose district is NOT in 2005 sarpanch file:")
print(single_blockers_dist[~single_blockers_dist["in_2005"]][[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "district_norm_hist","gp_prob"
]].to_string(index=False))

print(f"\nSingle-blocker GPs whose district is NOT in 2010 sarpanch file:")
print(single_blockers_dist[~single_blockers_dist["in_2010"]][[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "district_norm_hist","gp_prob"
]].to_string(index=False))

LGD districts (after aliases):        33
Reservation file districts:            33
2005 sarpanch file districts:          32
2010 sarpanch file districts:          33

LGD districts NOT in reservation file:

LGD districts NOT in 2005 sarpanch file:
  dungarpur: 353 GPs in LGD
  pratapgarh: 233 GPs in LGD
  sawaimadhopur: 298 GPs in LGD

LGD districts NOT in 2010 sarpanch file:
  sawaimadhopur: 298 GPs in LGD

Single-blocker GPs by district coverage:
  District in reservation file: 73 of 73
  District in 2005 sarpanch:    73 of 73
  District in 2010 sarpanch:    73 of 73

Single-blocker GPs whose district is NOT in 2005 sarpanch file:
Empty DataFrame
Columns: [gp_lgd_code, gp_name_lgd, district_lgd, district_norm_hist, gp_prob]
Index: []

Single-blocker GPs whose district is NOT in 2010 sarpanch file:
Empty DataFrame
Columns: [gp_lgd_code, gp_name_lgd, district_lgd, district_norm_hist, gp_prob]
Index: []


In [37]:
# =============================================================================
# Review Stage G candidates: lower fuzzy threshold (0.80) for single-blockers
# =============================================================================

stage_g_candidates = stage_e_df_check[
    (stage_e_df_check["best_sim_res"] >= 0.80) &
    (stage_e_df_check["best_sim_res"] < 0.90) &
    (stage_e_df_check["n_dose_values"] == 1)
].copy()

print(f"Stage G candidates (0.80-0.90 similarity, unique dose): {len(stage_g_candidates)}")
print(f"\nFull list for review:")
print(stage_g_candidates[[
    "gp_lgd_code","gp_name","district",
    "best_sim_res","best_match_res","dose_res"
]].to_string(index=False))

Stage G candidates (0.80-0.90 similarity, unique dose): 15

Full list for review:
gp_lgd_code       gp_name      district  best_sim_res best_match_res dose_res
      36763          Arni   Chittorgarh      0.888889          aarni    never
      41832 Kotri Dhaylan Neem Ka Thana      0.880000  kotdidhayalan     once
      38566       Rotwara          Dudu      0.857143        rotwada    never
      35625       Ameshar      Bhilwara      0.857143        aamesar     once
      41910    Bheroogarh        Sirohi      0.842105      bherugarh    never
      36687     Bhawaliya   Chittorgarh      0.823529       bhavliya    never
      41061     Dheenawas          Pali      0.823529       dhinawas     once
      42012  Dhunwa Kalan          Tonk      0.818182    dhuvankalan     once
      42225      Rooppura          Tonk      0.800000        ruppura    twice
      39294      Khairana      Jhalawar      0.800000        kherana    twice
      42696         Jawad      Salumbar      0.800000       

In [38]:
# =============================================================================
# Stage G — accept lower-threshold fuzzy matches for single-blocker GPs
# These are all clear transliteration variants, manually verified above
# =============================================================================

# Stage G uses res_clean (sp_2005_2010_manually_reviewed.csv) which already
# has both years combined — verify this gives us reserved_women_2005 AND 2010
stage_g_lgd_codes = set(stage_g_candidates["gp_lgd_code"].astype(str))

# Get full reservation info for these matches from res_clean
stage_g_full = []
for _, gp_row in stage_g_candidates.iterrows():
    dist   = gp_row["district"]
    gp_n   = normalize_name(gp_row["gp_name"])
    lgd    = gp_row["gp_lgd_code"]
    target = gp_row["best_match_res"]

    # Get all rows for this match in res_clean
    match_rows = res_clean[
        (res_clean["district_norm_hist"] == normalize_district_historical(dist)) &
        (res_clean["gp_norm"] == target)
    ]

    if len(match_rows) == 0:
        continue

    # Check dose consistency
    n_dose = match_rows["reservation_dose"].nunique()
    if n_dose > 1:
        print(f"WARNING: conflicting dose for {gp_row['gp_name']} → {target}: "
              f"{match_rows['reservation_dose'].unique()}")
        continue

    row = match_rows.iloc[0]
    stage_g_full.append({
        "gp_lgd_code":        lgd,
        "gp_name_lgd":        gp_row["gp_name"],
        "district_lgd":       dist,
        "subdistrict_lgd":    gp_lookup[
            gp_lookup["gp_lgd_code"]==lgd
        ]["subdistrict_lgd"].iloc[0] if lgd in gp_lookup["gp_lgd_code"].values else None,
        "reserved_women_2005":int(row["reserved_women_2005"]),
        "reserved_women_2010":int(row["reserved_women_2010"]),
        "reservation_dose_n": int(row["reservation_dose_n"]),
        "reservation_dose":   row["reservation_dose"],
        "match_stage":        "G_fuzzy_transliteration_single_blocker",
        "fuzzy_similarity":   gp_row["best_sim_res"],
        "res_gp_matched":     target
    })

stage_g_df = pd.DataFrame(stage_g_full)
print(f"Stage G accepted matches: {len(stage_g_df)}")
print(f"\nDose distribution:")
print(stage_g_df["reservation_dose"].value_counts())
print(f"\nFull list:")
print(stage_g_df[[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "res_gp_matched","fuzzy_similarity",
    "reserved_women_2005","reserved_women_2010","reservation_dose"
]].to_string(index=False))

# Confirm no overlap with existing history
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_g_df["gp_lgd_code"]) & already
print(f"\nOverlap with existing history (should be 0): {len(overlap)}")

Stage G accepted matches: 15

Dose distribution:
reservation_dose
never    7
once     6
twice    2
Name: count, dtype: int64

Full list:
gp_lgd_code   gp_name_lgd  district_lgd res_gp_matched  fuzzy_similarity  reserved_women_2005  reserved_women_2010 reservation_dose
      36763          Arni   Chittorgarh          aarni          0.888889                    0                    0            never
      41832 Kotri Dhaylan Neem Ka Thana  kotdidhayalan          0.880000                    0                    1             once
      38566       Rotwara          Dudu        rotwada          0.857143                    0                    0            never
      35625       Ameshar      Bhilwara        aamesar          0.857143                    0                    1             once
      41910    Bheroogarh        Sirohi      bherugarh          0.842105                    0                    0            never
      36687     Bhawaliya   Chittorgarh       bhavliya          0.82352

In [39]:
# =============================================================================
# Add Stage G and rerun pipeline
# =============================================================================

gp_res_history_final_v4 = pd.concat([
    gp_res_history_final,
    stage_g_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
        "reserved_women_2005","reserved_women_2010",
        "reservation_dose_n","reservation_dose","match_stage"
    ]]
], ignore_index=True)

assert gp_res_history_final_v4["gp_lgd_code"].duplicated().sum() == 0
print(f"GP history: {len(gp_res_history_final)} → {len(gp_res_history_final_v4)} (+{len(stage_g_df)})")

# Rebuild cluster_gp_res
cluster_gp_res_v4 = mc_long.merge(
    gp_res_history_final_v4[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

# Recompute treat_probs
cluster_known_v4 = (
    cluster_gp_res_v4[cluster_gp_res_v4["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v4.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v4.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v4.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v4 = all_clusters.merge(cluster_known_v4, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v4[col] = treat_probs_v4[col].fillna(0.0)

treat_probs_v4["p_unknown_treatment_mc"] = (
    1 - treat_probs_v4["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v4["p_any_reserved_mc"] = (
    treat_probs_v4["p_once_mc"] + treat_probs_v4["p_twice_mc"]
)
treat_probs_v4["expected_dose_mc"] = (
    treat_probs_v4["p_once_mc"] + 2 * treat_probs_v4["p_twice_mc"]
)
treat_probs_v4["treatment_certainty_mc"] = treat_probs_v4[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

# Primary GP dose
primary_dose_v4 = (
    cluster_gp_res_v4[cluster_gp_res_v4["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v4 = treat_probs_v4.merge(primary_dose_v4, on="DHSCLUST", how="left")
treat_probs_v4["primary_gp_norm"] = treat_probs_v4["primary_gp"].apply(normalize_lgd_code)
treat_probs_v4["primary_gp_has_history"] = treat_probs_v4["primary_gp_norm"].isin(
    set(gp_res_history_final_v4["gp_lgd_code"])
)

working_v4 = treat_probs_v4[treat_probs_v4["primary_gp_has_history"]].copy()

print(f"\nWorking clusters: {len(working_v4)}")
print(f"Fully linked (>=99.9%): {(working_v4['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v4['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v4['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v4['p_known_treatment_mc'].mean():.3f}")

# Update working variables
gp_res_history_final = gp_res_history_final_v4
cluster_gp_res       = cluster_gp_res_v4
treat_probs_new      = treat_probs_v4
working_clusters_new = working_v4
final_treat_probs    = working_v4[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)

print(f"\nSaved all outputs.")
print(f"\nMatch stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

GP history: 4716 → 4731 (+15)

Working clusters: 578
Fully linked (>=99.9%): 64
>=90% linked:           202
>=75% linked:           356
Mean p_known:           0.794

Saved all outputs.

Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide                 185
D_district_only_unique_dose                177
E_fuzzy_gp_name_district                    70
G_fuzzy_transliteration_single_blocker      15
Name: count, dtype: int64


In [40]:
# How many remaining single-blocker GPs have 0.70-0.80 similarity matches?
remaining_single_blockers = stage_e_df_check[
    (stage_e_df_check["best_sim_res"] >= 0.70) &
    (stage_e_df_check["best_sim_res"] < 0.80) &
    (stage_e_df_check["n_dose_values"] == 1) &
    (~stage_e_df_check["gp_lgd_code"].isin(
        set(gp_res_history_final["gp_lgd_code"])
    ))
].copy()

print(f"Remaining single-blocker GPs with 0.70-0.80 similarity: {len(remaining_single_blockers)}")
print(f"\nFull list for manual review:")
print(remaining_single_blockers[[
    "gp_lgd_code","gp_name","district",
    "best_sim_res","best_match_res","dose_res"
]].to_string(index=False))

Remaining single-blocker GPs with 0.70-0.80 similarity: 10

Full list for manual review:
gp_lgd_code   gp_name    district  best_sim_res best_match_res dose_res
      41201   Kesooli   Rajsamand      0.769231         kesuli      NaN
      40294   Latoori        Kota      0.769231         laturi      NaN
     294500 Gopalpura       Kekri      0.750000        malpura      NaN
      41905  Veerwara      Sirohi      0.750000       veervada      NaN
      36492 Koonthana Chittorgarh      0.750000        kuntana      NaN
     294609 Bhojrasar       Churu      0.750000        herasar      NaN
      36118   Peepera     Bikaner      0.714286        piperan      NaN
     294952   Modayat     Bikaner      0.714286        kolayat      NaN
      38370  Mahalaan        Dudu      0.705882      mathasula      NaN
      36362 Jhak Mund       Bundi      0.705882      jamkhmund      NaN


In [41]:
# Check these 5 plausible matches against both sarpanch files directly
plausible = {
    "41201": ("Kesooli",  "Rajsamand", "kesuli"),
    "40294": ("Latoori",  "Kota",      "laturi"),
    "41905": ("Veerwara", "Sirohi",    "veervada"),
    "36492": ("Koonthana","Chittorgarh","kuntana"),
    "36118": ("Peepera",  "Bikaner",   "piperan"),
}

print("Checking plausible matches against 2005 and 2010 sarpanch files:\n")
for lgd, (gp_name, district, best_match) in plausible.items():
    dist_norm = normalize_district_historical(district)
    
    # Check 2005
    matches_2005 = sp2005[
        sp2005["district_norm_hist"] == dist_norm
    ].copy()
    matches_2005["sim"] = matches_2005["gp_norm"].apply(
        lambda x: similarity(normalize_name(gp_name), x)
    )
    best_2005 = matches_2005.nlargest(1,"sim").iloc[0] if len(matches_2005) > 0 else None

    # Check 2010
    matches_2010 = sp2010[
        sp2010["district_norm_hist"] == dist_norm
    ].copy()
    matches_2010["sim"] = matches_2010["gp_norm"].apply(
        lambda x: similarity(normalize_name(gp_name), x)
    )
    best_2010 = matches_2010.nlargest(1,"sim").iloc[0] if len(matches_2010) > 0 else None

    print(f"{gp_name} ({district}) [LGD: {lgd}]")
    if best_2005 is not None:
        print(f"  2005: {best_2005['gp_norm']:<20} sim={best_2005['sim']:.3f}  "
              f"categ={best_2005['CATEG. OF POST OF SARPANCH']}  "
              f"reserved={best_2005['reserved_women_2005']}")
    if best_2010 is not None:
        print(f"  2010: {best_2010['gp_norm']:<20} sim={best_2010['sim']:.3f}  "
              f"categ={best_2010['Ward category']}  "
              f"reserved={best_2010['reserved_women_2010']}")
    print()

Checking plausible matches against 2005 and 2010 sarpanch files:

Kesooli (Rajsamand) [LGD: 41201]
  2005: kesuli               sim=0.769  categ=ST  reserved=0
  2010: kesuli               sim=0.769  categ=GENW  reserved=1

Latoori (Kota) [LGD: 40294]
  2005: laturi               sim=0.769  categ=GEN W  reserved=1
  2010: laturi               sim=0.769  categ=ST  reserved=0

Veerwara (Sirohi) [LGD: 41905]
  2005: veerwara             sim=1.000  categ=ST  reserved=0
  2010: veervada             sim=0.750  categ=GEN  reserved=0

Koonthana (Chittorgarh) [LGD: 36492]
  2005: kunthna              sim=0.750  categ=OBC  reserved=0
  2010: kuntana              sim=0.750  categ=GENW  reserved=1

Peepera (Bikaner) [LGD: 36118]
  2005: peepera              sim=1.000  categ=OBC  reserved=0
  2010: piperan              sim=0.714  categ=OBC  reserved=0



In [42]:
# =============================================================================
# Stage H — accept 5 manually verified lower-threshold matches
# =============================================================================

stage_h_entries = [
    {
        "gp_lgd_code":        "41201",
        "gp_name_lgd":        "Kesooli",
        "district_lgd":       "Rajsamand",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="41201"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 0,
        "reserved_women_2010": 1,
        "reservation_dose_n":  1,
        "reservation_dose":    "once",
        "match_stage":        "H_manual_verified_transliteration"
    },
    {
        "gp_lgd_code":        "40294",
        "gp_name_lgd":        "Latoori",
        "district_lgd":       "Kota",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="40294"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 1,
        "reserved_women_2010": 0,
        "reservation_dose_n":  1,
        "reservation_dose":    "once",
        "match_stage":        "H_manual_verified_transliteration"
    },
    {
        "gp_lgd_code":        "41905",
        "gp_name_lgd":        "Veerwara",
        "district_lgd":       "Sirohi",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="41905"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 0,
        "reserved_women_2010": 0,
        "reservation_dose_n":  0,
        "reservation_dose":    "never",
        "match_stage":        "H_manual_verified_transliteration"
    },
    {
        "gp_lgd_code":        "36492",
        "gp_name_lgd":        "Koonthana",
        "district_lgd":       "Chittorgarh",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="36492"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 0,
        "reserved_women_2010": 1,
        "reservation_dose_n":  1,
        "reservation_dose":    "once",
        "match_stage":        "H_manual_verified_transliteration"
    },
    {
        "gp_lgd_code":        "36118",
        "gp_name_lgd":        "Peepera",
        "district_lgd":       "Bikaner",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="36118"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 0,
        "reserved_women_2010": 0,
        "reservation_dose_n":  0,
        "reservation_dose":    "never",
        "match_stage":        "H_manual_verified_transliteration"
    },
]

stage_h_df = pd.DataFrame(stage_h_entries)

# Verify no overlap
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_h_df["gp_lgd_code"]) & already
print(f"Overlap with existing history (should be 0): {len(overlap)}")
print(f"\nStage H entries:")
print(stage_h_df[[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "reserved_women_2005","reserved_women_2010","reservation_dose"
]].to_string(index=False))

# Add to history and rerun
gp_res_history_final_v5 = pd.concat([
    gp_res_history_final, stage_h_df
], ignore_index=True)
assert gp_res_history_final_v5["gp_lgd_code"].duplicated().sum() == 0

# Rebuild cluster_gp_res
cluster_gp_res_v5 = mc_long.merge(
    gp_res_history_final_v5[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

# Recompute treat_probs
cluster_known_v5 = (
    cluster_gp_res_v5[cluster_gp_res_v5["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v5.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v5.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v5.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v5 = all_clusters.merge(cluster_known_v5, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v5[col] = treat_probs_v5[col].fillna(0.0)

treat_probs_v5["p_unknown_treatment_mc"] = (
    1 - treat_probs_v5["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v5["p_any_reserved_mc"] = (
    treat_probs_v5["p_once_mc"] + treat_probs_v5["p_twice_mc"]
)
treat_probs_v5["expected_dose_mc"] = (
    treat_probs_v5["p_once_mc"] + 2 * treat_probs_v5["p_twice_mc"]
)
treat_probs_v5["treatment_certainty_mc"] = treat_probs_v5[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_v5 = (
    cluster_gp_res_v5[cluster_gp_res_v5["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v5 = treat_probs_v5.merge(primary_dose_v5, on="DHSCLUST", how="left")
treat_probs_v5["primary_gp_norm"] = treat_probs_v5["primary_gp"].apply(normalize_lgd_code)
treat_probs_v5["primary_gp_has_history"] = treat_probs_v5["primary_gp_norm"].isin(
    set(gp_res_history_final_v5["gp_lgd_code"])
)

working_v5 = treat_probs_v5[treat_probs_v5["primary_gp_has_history"]].copy()

print(f"\nWorking clusters:       {len(working_v5)}")
print(f"Fully linked (>=99.9%): {(working_v5['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v5['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v5['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v5['p_known_treatment_mc'].mean():.3f}")

# Update working variables
gp_res_history_final = gp_res_history_final_v5
cluster_gp_res       = cluster_gp_res_v5
treat_probs_new      = treat_probs_v5
working_clusters_new = working_v5
final_treat_probs    = working_v5[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
print(f"\nSaved. Match stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

Overlap with existing history (should be 0): 0

Stage H entries:
gp_lgd_code gp_name_lgd district_lgd  reserved_women_2005  reserved_women_2010 reservation_dose
      41201     Kesooli    Rajsamand                    0                    1             once
      40294     Latoori         Kota                    1                    0             once
      41905    Veerwara       Sirohi                    0                    0            never
      36492   Koonthana  Chittorgarh                    0                    1             once
      36118     Peepera      Bikaner                    0                    0            never

Working clusters:       579
Fully linked (>=99.9%): 70
>=90% linked:           203
>=75% linked:           357
Mean p_known:           0.795

Saved. Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide       

In [43]:
# Check if Dungarpur, Pratapgarh, Sawaimadhopur appear in working clusters
missing_districts = ["dungarpur", "pratapgarh", "sawaimadhopur"]

print("Working clusters in missing districts:")
for dist in missing_districts:
    clusters = final_treat_probs[
        final_treat_probs["DHSREGNA"].apply(normalize_name) == dist
    ]
    print(f"  {dist}: {len(clusters)} working clusters")

print("\nAll district names in working clusters:")
print(sorted(final_treat_probs["DHSREGNA"].apply(normalize_name).unique()))

print("\nChecking if missing districts appear in sarpanch files under different names:")
print("\n2005 sarpanch districts:")
print(sorted(sp2005["district_norm_hist"].unique()))

print("\n2010 sarpanch districts:")
print(sorted(sp2010["district_norm_hist"].unique()))

print("\nReservation file districts:")
print(sorted(res_clean["district_norm_hist"].unique()))

# Fuzzy match missing districts against sarpanch file districts
from difflib import SequenceMatcher
sp2005_dists = sorted(sp2005["district_norm_hist"].unique())
sp2010_dists = sorted(sp2010["district_norm_hist"].unique())

print("\nBest fuzzy matches for missing districts in sarpanch files:")
for dist in missing_districts:
    print(f"\n  {dist}:")
    for sp_dist in sp2005_dists:
        sim = similarity(dist, sp_dist)
        if sim > 0.6:
            print(f"    2005: {sp_dist} (sim={sim:.3f})")
    for sp_dist in sp2010_dists:
        sim = similarity(dist, sp_dist)
        if sim > 0.6:
            print(f"    2010: {sp_dist} (sim={sim:.3f})")

Working clusters in missing districts:
  dungarpur: 16 working clusters
  pratapgarh: 18 working clusters
  sawaimadhopur: 17 working clusters

All district names in working clusters:
['ajmer', 'alwar', 'banswara', 'baran', 'barmer', 'bharatpur', 'bhilwara', 'bikaner', 'bundi', 'chittaurgarh', 'churu', 'dausa', 'dhaulpur', 'dungarpur', 'ganganagar', 'hanumangarh', 'jaipur', 'jaisalmer', 'jalor', 'jhalawar', 'jhunjhunun', 'jodhpur', 'karauli', 'kota', 'nagaur', 'pali', 'pratapgarh', 'rajsamand', 'sawaimadhopur', 'sikar', 'sirohi', 'tonk', 'udaipur']

Checking if missing districts appear in sarpanch files under different names:

2005 sarpanch districts:
['ajmer', 'alwar', 'banswara', 'baran', 'barmer', 'bharatpur', 'bhilwara', 'bikaner', 'bundi', 'chittorgarh', 'churu', 'dausa', 'dholpur', 'dungerpur', 'ganganagar', 'hanumangarh', 'jaipur', 'jaisalmer', 'jalore', 'jhalawar', 'jhunjhunu', 'jodhpur', 'karauli', 'kota', 'nagaur', 'pali', 'rajsamand', 'sikar', 'sirohi', 'smadhopur', 'tonk', 

In [44]:
# =============================================================================
# Fix district aliases for the 3 missing districts
# =============================================================================

# Add new aliases to DISTRICT_ALIASES
DISTRICT_ALIASES.update({
    "dungerpur":    "dungarpur",   # 2005 spelling variant
    "smadhopur":    "sawaimadhopur",  # OCR truncation in both sarpanch files
})

print("Updated DISTRICT_ALIASES:")
print({k:v for k,v in DISTRICT_ALIASES.items() 
       if k in ["dungerpur","smadhopur","pratapgarh"]})

# Re-normalize all district columns in sarpanch files
sp2005["district_norm_hist"] = sp2005["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)
sp2010["district_norm_hist"] = sp2010["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)
res_clean["district_norm_hist"] = res_clean["district_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)

# Also re-normalize gp_lookup
gp_lookup["district_norm_hist"] = gp_lookup["district_lgd"].apply(
    normalize_district_historical
)

# Verify fixes
print(f"\n2005 sarpanch districts after fix:")
print(sorted(sp2005["district_norm_hist"].unique()))
print(f"\n2010 sarpanch districts after fix:")
print(sorted(sp2010["district_norm_hist"].unique()))

# Check dungarpur and sawaimadhopur coverage now
for dist in ["dungarpur","sawaimadhopur"]:
    n_2005 = (sp2005["district_norm_hist"] == dist).sum()
    n_2010 = (sp2010["district_norm_hist"] == dist).sum()
    print(f"\n{dist}: {n_2005} rows in 2005, {n_2010} rows in 2010")

# For Pratapgarh 2005: check how many rows exist under chittorgarh
# Pratapgarh was carved from Chittorgarh in 2008
# So pratapgarh GPs in 2005 should be found under chittorgarh
pratapgarh_gps = gp_lookup[
    gp_lookup["district_norm_hist"] == "pratapgarh"
]["gp_lgd_code"].tolist()

print(f"\nPratapgarh GPs in LGD: {len(pratapgarh_gps)}")

# Check how many of those appear in 2005 chittorgarh records
chittorgarh_2005 = sp2005[sp2005["district_norm_hist"] == "chittorgarh"]
print(f"2005 chittorgarh records: {len(chittorgarh_2005)}")

# Test: fuzzy match a few pratapgarh GPs against chittorgarh 2005
pratapgarh_gp_lookup = gp_lookup[
    gp_lookup["district_norm_hist"] == "pratapgarh"
].head(5)

print(f"\nSample Pratapgarh GP fuzzy matches in 2005 Chittorgarh records:")
for _, row in pratapgarh_gp_lookup.iterrows():
    gp_n = row["gp_norm"]
    cands = chittorgarh_2005.copy()
    cands["sim"] = cands["gp_norm"].apply(lambda x: similarity(gp_n, x))
    best = cands.nlargest(1,"sim").iloc[0]
    print(f"  {row['gp_name_lgd']}: best={best['gp_norm']} "
          f"(sim={best['sim']:.3f}, categ={best['CATEG. OF POST OF SARPANCH']})")

Updated DISTRICT_ALIASES:
{'dungerpur': 'dungarpur', 'smadhopur': 'sawaimadhopur'}

2005 sarpanch districts after fix:
['ajmer', 'alwar', 'banswara', 'baran', 'barmer', 'bharatpur', 'bhilwara', 'bikaner', 'bundi', 'chittorgarh', 'churu', 'dausa', 'dholpur', 'dungarpur', 'ganganagar', 'hanumangarh', 'jaipur', 'jaisalmer', 'jalore', 'jhalawar', 'jhunjhunu', 'jodhpur', 'karauli', 'kota', 'nagaur', 'pali', 'rajsamand', 'sawaimadhopur', 'sikar', 'sirohi', 'tonk', 'udaipur']

2010 sarpanch districts after fix:
['ajmer', 'alwar', 'banswara', 'baran', 'barmer', 'bharatpur', 'bhilwara', 'bikaner', 'bundi', 'chittorgarh', 'churu', 'dausa', 'dholpur', 'dungarpur', 'ganganagar', 'hanumangarh', 'jaipur', 'jaisalmer', 'jalore', 'jhalawar', 'jhunjhunu', 'jodhpur', 'karauli', 'kota', 'nagaur', 'pali', 'pratapgarh', 'rajsamand', 'sawaimadhopur', 'sikar', 'sirohi', 'tonk', 'udaipur']

dungarpur: 237 rows in 2005, 237 rows in 2010

sawaimadhopur: 196 rows in 2005, 197 rows in 2010

Pratapgarh GPs in LGD:

In [45]:
# =============================================================================
# Rerun Stage F with corrected district aliases
# =============================================================================

# Rebuild sp2005_by_district and sp2010_by_district with corrected aliases
sp2005_by_district = (
    sp2005.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_rows              = ("gp_norm",             "size"),
        n_categ_values      = ("CATEG. OF POST OF SARPANCH", "nunique"),
        categ_values        = ("CATEG. OF POST OF SARPANCH",
                               lambda x: " | ".join(sorted(set(x.dropna())))),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        samiti              = ("samiti_norm",         "first")
    )
    .reset_index()
)

sp2010_by_district = (
    sp2010.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_rows              = ("gp_norm",             "size"),
        n_categ_values      = ("Ward category",       "nunique"),
        categ_values        = ("Ward category",
                               lambda x: " | ".join(sorted(set(
                                   str(v).strip() for v in x.dropna()
                               )))),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        samiti              = ("samiti_norm",         "first")
    )
    .reset_index()
)

# Rebuild unmatched_gps_df with corrected aliases
# (only GPs not yet in gp_res_history_final)
already_matched = set(gp_res_history_final["gp_lgd_code"])

unmatched_gps_df_v2 = gp_lookup[
    ~gp_lookup["gp_lgd_code"].isin(already_matched)
][["gp_lgd_code","gp_name_lgd","district_lgd",
   "subdistrict_lgd","gp_norm","district_norm_hist"]].copy()
unmatched_gps_df_v2 = unmatched_gps_df_v2.rename(columns={
    "gp_name_lgd": "gp_name",
    "district_lgd": "district",
    "subdistrict_lgd": "subdistrict_samiti"
})

print(f"Unmatched GPs for Stage F rerun: {len(unmatched_gps_df_v2)}")
print(f"\nUnmatched GPs in fixed districts:")
for dist in ["dungarpur","sawaimadhopur","pratapgarh"]:
    n = (unmatched_gps_df_v2["district_norm_hist"] == dist).sum()
    print(f"  {dist}: {n} unmatched GPs")

# Rerun 2005 fuzzy match
print(f"\nRunning 2005 fuzzy match...")
results_2005_v2 = []
for _, gp_row in unmatched_gps_df_v2.iterrows():
    dist    = gp_row["district_norm_hist"]
    gp_n    = gp_row["gp_norm"]
    lgd     = gp_row["gp_lgd_code"]
    candidates = sp2005_by_district[
        sp2005_by_district["district_norm_hist"] == dist
    ].copy()
    if candidates.empty:
        results_2005_v2.append({
            "gp_lgd_code": lgd, "best_match_gp": None,
            "similarity": 0.0, "reserved_women_2005": None,
            "categ_2005": None, "n_rows": 0,
            "n_categ_values": 0, "match_found": False
        })
        continue
    candidates["sim"] = candidates["gp_norm"].apply(
        lambda x: similarity(gp_n, x)
    )
    best = candidates.nlargest(1,"sim").iloc[0]
    match_found = best["sim"] >= 0.85
    results_2005_v2.append({
        "gp_lgd_code":        lgd,
        "best_match_gp":      best["gp_norm"],
        "similarity":         best["sim"],
        "reserved_women_2005":best["reserved_women_2005"] if match_found else None,
        "categ_2005":         best["categ_values"] if match_found else None,
        "n_rows":             best["n_rows"] if match_found else 0,
        "n_categ_values":     best["n_categ_values"] if match_found else 0,
        "match_found":        match_found
    })

results_2005_v2_df = pd.DataFrame(results_2005_v2)
safe_2005_v2 = results_2005_v2_df[
    results_2005_v2_df["match_found"] &
    (results_2005_v2_df["n_categ_values"] <= 1)
]
print(f"2005 safe matches: {len(safe_2005_v2)}")
print(f"  In dungarpur:     "
      f"{safe_2005_v2[safe_2005_v2['gp_lgd_code'].isin(unmatched_gps_df_v2[unmatched_gps_df_v2['district_norm_hist']=='dungarpur']['gp_lgd_code'])].shape[0]}")
print(f"  In sawaimadhopur: "
      f"{safe_2005_v2[safe_2005_v2['gp_lgd_code'].isin(unmatched_gps_df_v2[unmatched_gps_df_v2['district_norm_hist']=='sawaimadhopur']['gp_lgd_code'])].shape[0]}")

# Rerun 2010 fuzzy match
print(f"\nRunning 2010 fuzzy match...")
results_2010_v2 = []
for _, gp_row in unmatched_gps_df_v2.iterrows():
    dist    = gp_row["district_norm_hist"]
    gp_n    = gp_row["gp_norm"]
    lgd     = gp_row["gp_lgd_code"]
    candidates = sp2010_by_district[
        sp2010_by_district["district_norm_hist"] == dist
    ].copy()
    if candidates.empty:
        results_2010_v2.append({
            "gp_lgd_code": lgd, "best_match_gp": None,
            "similarity": 0.0, "reserved_women_2010": None,
            "categ_2010": None, "n_rows": 0,
            "n_categ_values": 0, "match_found": False
        })
        continue
    candidates["sim"] = candidates["gp_norm"].apply(
        lambda x: similarity(gp_n, x)
    )
    best = candidates.nlargest(1,"sim").iloc[0]
    match_found = best["sim"] >= 0.85
    results_2010_v2.append({
        "gp_lgd_code":        lgd,
        "best_match_gp":      best["gp_norm"],
        "similarity":         best["sim"],
        "reserved_women_2010":best["reserved_women_2010"] if match_found else None,
        "categ_2010":         best["categ_values"] if match_found else None,
        "n_rows":             best["n_rows"] if match_found else 0,
        "n_categ_values":     best["n_categ_values"] if match_found else 0,
        "match_found":        match_found
    })

results_2010_v2_df = pd.DataFrame(results_2010_v2)
safe_2010_v2 = results_2010_v2_df[
    results_2010_v2_df["match_found"] &
    (results_2010_v2_df["n_categ_values"] <= 1)
]
print(f"2010 safe matches: {len(safe_2010_v2)}")
print(f"  In dungarpur:     "
      f"{safe_2010_v2[safe_2010_v2['gp_lgd_code'].isin(unmatched_gps_df_v2[unmatched_gps_df_v2['district_norm_hist']=='dungarpur']['gp_lgd_code'])].shape[0]}")
print(f"  In sawaimadhopur: "
      f"{safe_2010_v2[safe_2010_v2['gp_lgd_code'].isin(unmatched_gps_df_v2[unmatched_gps_df_v2['district_norm_hist']=='sawaimadhopur']['gp_lgd_code'])].shape[0]}")
print(f"  In pratapgarh:    "
      f"{safe_2010_v2[safe_2010_v2['gp_lgd_code'].isin(unmatched_gps_df_v2[unmatched_gps_df_v2['district_norm_hist']=='pratapgarh']['gp_lgd_code'])].shape[0]}")

# Combine both years
safe_2005_v2_df = safe_2005_v2[["gp_lgd_code","reserved_women_2005","similarity"]].rename(
    columns={"similarity":"sim_2005"}
)
safe_2010_v2_df = safe_2010_v2[["gp_lgd_code","reserved_women_2010","similarity"]].rename(
    columns={"similarity":"sim_2010"}
)

combined_v2 = unmatched_gps_df_v2[["gp_lgd_code","gp_name","district","subdistrict_samiti"]].merge(
    safe_2005_v2_df, on="gp_lgd_code", how="left"
).merge(
    safe_2010_v2_df, on="gp_lgd_code", how="left"
)

both_years_v2 = combined_v2[
    combined_v2["reserved_women_2005"].notna() &
    combined_v2["reserved_women_2010"].notna()
].copy()

both_years_v2["reserved_women_2005"] = both_years_v2["reserved_women_2005"].astype(int)
both_years_v2["reserved_women_2010"] = both_years_v2["reserved_women_2010"].astype(int)
both_years_v2["reservation_dose_n"]  = (
    both_years_v2["reserved_women_2005"] + both_years_v2["reserved_women_2010"]
)
both_years_v2["reservation_dose"] = both_years_v2["reservation_dose_n"].map(DOSE_LABEL_MAP)

print(f"\nBoth-year matches (Stage F v2): {len(both_years_v2)}")
print(f"\nBy district:")
print(both_years_v2.merge(
    unmatched_gps_df_v2[["gp_lgd_code","district_norm_hist"]],
    on="gp_lgd_code"
)["district_norm_hist"].value_counts().head(10))

Unmatched GPs for Stage F rerun: 6463

Unmatched GPs in fixed districts:
  dungarpur: 249 unmatched GPs
  sawaimadhopur: 171 unmatched GPs
  pratapgarh: 150 unmatched GPs

Running 2005 fuzzy match...
2005 safe matches: 2571
  In dungarpur:     103
  In sawaimadhopur: 41

Running 2010 fuzzy match...
2010 safe matches: 2284
  In dungarpur:     85
  In sawaimadhopur: 41
  In pratapgarh:    42

Both-year matches (Stage F v2): 1733

By district:
district_norm_hist
hanumangarh    110
udaipur        107
jaipur         106
ganganagar      98
barmer          91
nagaur          88
jodhpur         79
banswara        79
bikaner         73
sikar           71
Name: count, dtype: int64


In [46]:
# =============================================================================
# Stage F2 — add new both-year matches from corrected aliases
# =============================================================================

# Confirm no overlap with existing history
already_matched = set(gp_res_history_final["gp_lgd_code"])
overlap = set(both_years_v2["gp_lgd_code"]) & already_matched
print(f"Overlap with existing history: {len(overlap)}")

# Remove any overlaps just in case
both_years_v2_new = both_years_v2[
    ~both_years_v2["gp_lgd_code"].isin(already_matched)
].copy()
print(f"New GPs to add (Stage F2): {len(both_years_v2_new)}")

# Build stage entry
stage_f2 = both_years_v2_new.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]],
    on="gp_lgd_code", how="left"
)[[
    "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
    "reserved_women_2005","reserved_women_2010",
    "reservation_dose_n","reservation_dose"
]].copy()
stage_f2["match_stage"] = "F2_fuzzy_sarpanch_both_years_alias_fix"

print(f"\nDose distribution:")
print(stage_f2["reservation_dose"].value_counts())

# Add to history
gp_res_history_final_v6 = pd.concat([
    gp_res_history_final, stage_f2
], ignore_index=True)
assert gp_res_history_final_v6["gp_lgd_code"].duplicated().sum() == 0
print(f"\nGP history: {len(gp_res_history_final)} → {len(gp_res_history_final_v6)}")

# Rebuild cluster_gp_res
cluster_gp_res_v6 = mc_long.merge(
    gp_res_history_final_v6[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

# Recompute treat_probs
cluster_known_v6 = (
    cluster_gp_res_v6[cluster_gp_res_v6["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v6.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v6.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v6.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v6 = all_clusters.merge(cluster_known_v6, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v6[col] = treat_probs_v6[col].fillna(0.0)

treat_probs_v6["p_unknown_treatment_mc"] = (
    1 - treat_probs_v6["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v6["p_any_reserved_mc"] = (
    treat_probs_v6["p_once_mc"] + treat_probs_v6["p_twice_mc"]
)
treat_probs_v6["expected_dose_mc"] = (
    treat_probs_v6["p_once_mc"] + 2 * treat_probs_v6["p_twice_mc"]
)
treat_probs_v6["treatment_certainty_mc"] = treat_probs_v6[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

# Primary GP dose
primary_dose_v6 = (
    cluster_gp_res_v6[cluster_gp_res_v6["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v6 = treat_probs_v6.merge(primary_dose_v6, on="DHSCLUST", how="left")
treat_probs_v6["primary_gp_norm"] = treat_probs_v6["primary_gp"].apply(normalize_lgd_code)
treat_probs_v6["primary_gp_has_history"] = treat_probs_v6["primary_gp_norm"].isin(
    set(gp_res_history_final_v6["gp_lgd_code"])
)

working_v6 = treat_probs_v6[treat_probs_v6["primary_gp_has_history"]].copy()

print(f"\nWorking clusters:       {len(working_v6)}")
print(f"Fully linked (>=99.9%): {(working_v6['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v6['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v6['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v6['p_known_treatment_mc'].mean():.3f}")
print(f"Mean certainty:         {working_v6['treatment_certainty_mc'].mean():.3f}")

bins   = [0, 0.25, 0.50, 0.75, 0.90, 0.999, 1.01]
labels = ["0-25%","25-50%","50-75%","75-90%","90-99%","100%"]
working_v6["known_mass_bin"] = pd.cut(
    working_v6["p_known_treatment_mc"],
    bins=bins, labels=labels, right=False
)
print(f"\nKnown mass bin distribution:")
print(working_v6["known_mass_bin"].value_counts().sort_index())

# Update working variables
gp_res_history_final = gp_res_history_final_v6
cluster_gp_res       = cluster_gp_res_v6
treat_probs_new      = treat_probs_v6
working_clusters_new = working_v6
final_treat_probs    = working_v6[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)

print(f"\nSaved. Total GP histories: {len(gp_res_history_final)}")
print(f"\nMatch stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

Overlap with existing history: 0
New GPs to add (Stage F2): 1733

Dose distribution:
reservation_dose
once     851
never    588
twice    294
Name: count, dtype: int64

GP history: 4736 → 6469

Working clusters:       725
Fully linked (>=99.9%): 100
>=90% linked:           263
>=75% linked:           460
Mean p_known:           0.804
Mean certainty:         0.614

Known mass bin distribution:
known_mass_bin
0-25%       0
25-50%     31
50-75%    234
75-90%    197
90-99%    163
100%      100
Name: count, dtype: int64

Saved. Total GP histories: 6469

Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
F2_fuzzy_sarpanch_both_years_alias_fix    1733
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide                 185
D_district_only_unique_dose                177
E_fuzzy_gp_name_district                    70
G_fuzzy_transliteration_single_blocker      15
H_manual_verified_transliteration  

In [47]:
# Quick check: how many single-blocker clusters remain?
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)
analysis_clusters = set(final_treat_probs["DHSCLUST"]) - isolated_clusters

near_complete = final_treat_probs[
    (final_treat_probs["p_known_treatment_mc"] >= 0.90) &
    (final_treat_probs["p_known_treatment_mc"] < 0.999)
]

print(f"Working clusters:          {len(final_treat_probs)}")
print(f"Isolated clusters:         {len(isolated_clusters)}")
print(f"Analysis clusters:         {len(analysis_clusters)}")
print(f"Clusters at 90-99%:        {len(near_complete)}")

# Count single-blocker clusters
blocking_gps_v2 = []
for _, cluster in near_complete.iterrows():
    dhsclust = cluster["DHSCLUST"]
    unmatched_rows = cluster_gp_res[
        (cluster_gp_res["DHSCLUST"] == dhsclust) &
        (cluster_gp_res["reservation_dose"].isna()) &
        (cluster_gp_res["gp_lgd_code"].notna())
    ]
    for _, gp_row in unmatched_rows.iterrows():
        blocking_gps_v2.append({
            "DHSCLUST":             dhsclust,
            "p_known_treatment_mc": cluster["p_known_treatment_mc"],
            "gp_lgd_code":          gp_row["gp_lgd_code"],
            "gp_prob":              gp_row["gp_prob"],
        })

blocking_df_v2 = pd.DataFrame(blocking_gps_v2)
blocking_df_v2 = blocking_df_v2.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]],
    on="gp_lgd_code", how="left"
)

n_per_cluster = blocking_df_v2.groupby("DHSCLUST")["gp_lgd_code"].nunique()
single_blocker_clusters_v2 = n_per_cluster[n_per_cluster == 1].index
single_blockers_v2 = blocking_df_v2[
    blocking_df_v2["DHSCLUST"].isin(single_blocker_clusters_v2)
].copy()

print(f"\nSingle-blocker clusters: {len(single_blocker_clusters_v2)}")
print(f"Unique single-blocker GPs: {single_blockers_v2['gp_lgd_code'].nunique()}")
print(f"\nTop 20 single-blocker GPs:")
print(single_blockers_v2[[
    "DHSCLUST","p_known_treatment_mc","gp_lgd_code",
    "gp_name_lgd","district_lgd","subdistrict_lgd","gp_prob"
]].sort_values("p_known_treatment_mc", ascending=False)
.head(20).to_string(index=False))

Working clusters:          725
Isolated clusters:         121
Analysis clusters:         604
Clusters at 90-99%:        163

Single-blocker clusters: 67
Unique single-blocker GPs: 60

Top 20 single-blocker GPs:
 DHSCLUST  p_known_treatment_mc gp_lgd_code            gp_name_lgd     district_lgd subdistrict_lgd  gp_prob
   291247              0.998945      293955               Jilawara            Ajmer       Nasirabad 0.001055
   291621              0.997936      295432        Badhli Nathusar        Jaisalmer         Pokaran 0.002064
   290451              0.997696      294914             Sonthli(R)        Jhunjhunu       Nawalgarh 0.002304
   291339              0.997110      295032             Kushalgarh      Chittorgarh      Rawatbhata 0.002890
   291376              0.997000       34833    Laxmipura Khankaraa            Baran      Kishanganj 0.003000
   290611              0.996917       40260                 Kolana             Kota         Ladpura 0.003083
   290663              0.9

In [48]:
# =============================================================================
# Stage G2 — fuzzy check for new single-blocker GPs after alias fix
# =============================================================================

single_blocker_gps_v2 = single_blockers_v2[["gp_lgd_code","gp_name_lgd",
                                             "district_lgd"]].drop_duplicates()
single_blocker_gps_v2 = single_blocker_gps_v2.rename(columns={
    "gp_name_lgd": "gp_name",
    "district_lgd": "district"
})
single_blocker_gps_v2["gp_norm"] = single_blocker_gps_v2["gp_name"].apply(normalize_name)
single_blocker_gps_v2["district_norm_hist"] = single_blocker_gps_v2["district"].apply(
    normalize_district_historical
)

# Rebuild res_district_only with updated aliases
res_by_district_v2 = (
    res_clean
    .groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_dose_values       = ("reservation_dose",    "nunique"),
        reservation_dose    = ("reservation_dose",    "first"),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        reservation_dose_n  = ("reservation_dose_n",  "first")
    )
    .reset_index()
)

# Fuzzy match at multiple thresholds
stage_g2_scores = []
for _, gp_row in single_blocker_gps_v2.iterrows():
    dist   = gp_row["district_norm_hist"]
    gp_n   = gp_row["gp_norm"]
    lgd    = gp_row["gp_lgd_code"]

    cands = res_by_district_v2[
        res_by_district_v2["district_norm_hist"] == dist
    ].copy()
    if cands.empty:
        stage_g2_scores.append({
            "gp_lgd_code": lgd, "gp_name": gp_row["gp_name"],
            "district": dist, "best_sim": 0,
            "best_match": None, "dose": None, "n_dose_values": 0
        })
        continue
    cands["sim"] = cands["gp_norm"].apply(lambda x: similarity(gp_n, x))
    best = cands.nlargest(1,"sim").iloc[0]
    stage_g2_scores.append({
        "gp_lgd_code":   lgd,
        "gp_name":       gp_row["gp_name"],
        "district":      dist,
        "best_sim":      best["sim"],
        "best_match":    best["gp_norm"],
        "dose":          best["reservation_dose"],
        "n_dose_values": best["n_dose_values"]
    })

stage_g2_df = pd.DataFrame(stage_g2_scores).sort_values("best_sim", ascending=False)

print("Single-blocker GP fuzzy match scores (updated aliases):")
print(f"\nThreshold summary:")
for thresh in [0.90, 0.85, 0.80, 0.75, 0.70]:
    n = (stage_g2_df["best_sim"] >= thresh).sum()
    n_unique = ((stage_g2_df["best_sim"] >= thresh) &
                (stage_g2_df["n_dose_values"] == 1)).sum()
    print(f"  >= {thresh}: {n} matches ({n_unique} with unique dose)")

print(f"\nFull list (best matches):")
print(stage_g2_df[[
    "gp_lgd_code","gp_name","district",
    "best_sim","best_match","dose","n_dose_values"
]].to_string(index=False))

Single-blocker GP fuzzy match scores (updated aliases):

Threshold summary:
  >= 0.9: 8 matches (7 with unique dose)
  >= 0.85: 11 matches (9 with unique dose)
  >= 0.8: 20 matches (16 with unique dose)
  >= 0.75: 28 matches (24 with unique dose)
  >= 0.7: 38 matches (31 with unique dose)

Full list (best matches):
gp_lgd_code                gp_name    district  best_sim         best_match  dose  n_dose_values
      40323              Madanpura        kota  1.000000          madanpura never              2
      40002           Somala Ratra     karauli  0.952381         somlaratra never              1
      33641               Padaliya       ajmer  0.933333            padliya  once              1
      36227 Dhbhaiyon Ka Naya Gaon       bundi  0.918919 dhabayonkanayagaon never              1
      40346          Daboli Meethi      nagaur  0.916667       davolimeethi  once              1
      34669                  Bohat       baran  0.909091             bohath  once              1
    

In [49]:
# =============================================================================
# Stage G2 — accept single-blocker fuzzy matches after alias fix
# Manually reviewed above
# =============================================================================

# Accepted matches with their correct reservation data from res_clean
accepted_g2 = {
    "40002":  ("Somala Ratra",           "karauli"),
    "33641":  ("Padaliya",               "ajmer"),
    "36227":  ("Dhbhaiyon Ka Naya Gaon", "bundi"),
    "40346":  ("Daboli Meethi",          "nagaur"),
    "34669":  ("Bohat",                  "baran"),
    "36086":  ("Kotri",                  "bikaner"),
    "33776":  ("Satawariya",             "ajmer"),
    "38083":  ("Malar Khera",            "hanumangarh"),
    "35943":  ("Eitdiya",                "bhilwara"),
    "36299":  ("Utrana",                 "bundi"),
    "34843":  ("Suwans",                 "baran"),
    "41933":  ("Poseetara",              "sirohi"),
    "34293":  ("Sarhheta",               "alwar"),
    "293955": ("Jilawara",               "ajmer"),
    "36331":  ("Suwanya",                "bundi"),
}

# Rejected: Madanpura (conflicted), Meethri (conflicted),
#           Awada/banwada (different prefix),
#           Pundalsar (conflicted), Sarsanda (conflicted)

stage_g2_entries = []
for lgd, (gp_name, dist) in accepted_g2.items():
    gp_n      = normalize_name(gp_name)
    dist_norm = normalize_district_historical(dist)

    match_rows = res_clean[
        (res_clean["district_norm_hist"] == dist_norm) &
        (res_clean["gp_norm"].apply(lambda x: similarity(gp_n, x)) >= 0.75)
    ].copy()
    match_rows["sim"] = match_rows["gp_norm"].apply(lambda x: similarity(gp_n, x))
    match_rows = match_rows.nlargest(1, "sim")

    if len(match_rows) == 0:
        print(f"WARNING: no match found for {gp_name} in {dist}")
        continue

    n_dose = match_rows["reservation_dose"].nunique()
    if n_dose > 1:
        print(f"WARNING: conflicting dose for {gp_name}: "
              f"{match_rows['reservation_dose'].unique()}")
        continue

    row = match_rows.iloc[0]
    subdistrict = gp_lookup[
        gp_lookup["gp_lgd_code"] == lgd
    ]["subdistrict_lgd"].iloc[0] if lgd in gp_lookup["gp_lgd_code"].values else None

    stage_g2_entries.append({
        "gp_lgd_code":        lgd,
        "gp_name_lgd":        gp_name,
        "district_lgd":       dist,
        "subdistrict_lgd":    subdistrict,
        "reserved_women_2005":int(row["reserved_women_2005"]),
        "reserved_women_2010":int(row["reserved_women_2010"]),
        "reservation_dose_n": int(row["reservation_dose_n"]),
        "reservation_dose":   row["reservation_dose"],
        "match_stage":        "G2_fuzzy_single_blocker_alias_fix",
        "res_gp_matched":     row["gp_norm"],
        "fuzzy_sim":          row["sim"]
    })

stage_g2_df = pd.DataFrame(stage_g2_entries)
print(f"Stage G2 accepted: {len(stage_g2_df)}")
print(f"\nFull list:")
print(stage_g2_df[[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "res_gp_matched","fuzzy_sim",
    "reserved_women_2005","reserved_women_2010","reservation_dose"
]].to_string(index=False))

# Verify no overlap
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_g2_df["gp_lgd_code"]) & already
print(f"\nOverlap with existing history (should be 0): {len(overlap)}")

Stage G2 accepted: 15

Full list:
gp_lgd_code            gp_name_lgd district_lgd     res_gp_matched  fuzzy_sim  reserved_women_2005  reserved_women_2010 reservation_dose
      40002           Somala Ratra      karauli         somlaratra   0.952381                    0                    0            never
      33641               Padaliya        ajmer            padliya   0.933333                    0                    1             once
      36227 Dhbhaiyon Ka Naya Gaon        bundi dhabayonkanayagaon   0.918919                    0                    0            never
      40346          Daboli Meethi       nagaur       davolimeethi   0.916667                    0                    1             once
      34669                  Bohat        baran             bohath   0.909091                    0                    1             once
      36086                  Kotri      bikaner             kotari   0.909091                    1                    0             once
      3

In [50]:
# =============================================================================
# Add Stage G2 and rerun pipeline
# =============================================================================

gp_res_history_final_v7 = pd.concat([
    gp_res_history_final,
    stage_g2_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
        "reserved_women_2005","reserved_women_2010",
        "reservation_dose_n","reservation_dose","match_stage"
    ]]
], ignore_index=True)
assert gp_res_history_final_v7["gp_lgd_code"].duplicated().sum() == 0
print(f"GP history: {len(gp_res_history_final)} → {len(gp_res_history_final_v7)}")

# Rebuild cluster_gp_res
cluster_gp_res_v7 = mc_long.merge(
    gp_res_history_final_v7[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

# Recompute treat_probs
cluster_known_v7 = (
    cluster_gp_res_v7[cluster_gp_res_v7["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v7.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v7.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v7.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v7 = all_clusters.merge(cluster_known_v7, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v7[col] = treat_probs_v7[col].fillna(0.0)

treat_probs_v7["p_unknown_treatment_mc"] = (
    1 - treat_probs_v7["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v7["p_any_reserved_mc"] = (
    treat_probs_v7["p_once_mc"] + treat_probs_v7["p_twice_mc"]
)
treat_probs_v7["expected_dose_mc"] = (
    treat_probs_v7["p_once_mc"] + 2 * treat_probs_v7["p_twice_mc"]
)
treat_probs_v7["treatment_certainty_mc"] = treat_probs_v7[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_v7 = (
    cluster_gp_res_v7[cluster_gp_res_v7["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v7 = treat_probs_v7.merge(primary_dose_v7, on="DHSCLUST", how="left")
treat_probs_v7["primary_gp_norm"] = treat_probs_v7["primary_gp"].apply(normalize_lgd_code)
treat_probs_v7["primary_gp_has_history"] = treat_probs_v7["primary_gp_norm"].isin(
    set(gp_res_history_final_v7["gp_lgd_code"])
)

working_v7 = treat_probs_v7[treat_probs_v7["primary_gp_has_history"]].copy()

print(f"\nWorking clusters:       {len(working_v7)}")
print(f"Fully linked (>=99.9%): {(working_v7['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v7['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v7['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v7['p_known_treatment_mc'].mean():.3f}")

# Update working variables
gp_res_history_final = gp_res_history_final_v7
cluster_gp_res       = cluster_gp_res_v7
treat_probs_new      = treat_probs_v7
working_clusters_new = working_v7
final_treat_probs    = working_v7[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
print(f"\nSaved. Total GP histories: {len(gp_res_history_final)}")
print(f"\nMatch stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

GP history: 6469 → 6484

Working clusters:       730
Fully linked (>=99.9%): 119
>=90% linked:           273
>=75% linked:           470
Mean p_known:           0.807

Saved. Total GP histories: 6484

Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
F2_fuzzy_sarpanch_both_years_alias_fix    1733
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide                 185
D_district_only_unique_dose                177
E_fuzzy_gp_name_district                    70
G_fuzzy_transliteration_single_blocker      15
G2_fuzzy_single_blocker_alias_fix           15
H_manual_verified_transliteration            5
Name: count, dtype: int64


In [51]:
# =============================================================================
# Check newer GP codes (262xxx, 293xxx, 294xxx, 295xxx) 
# These may be splits/renames of older GPs — find parent GP via village overlap
# =============================================================================

# Identify newer GP codes among single-blocker GPs
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)
analysis_clusters = set(final_treat_probs["DHSCLUST"]) - isolated_clusters

near_complete = final_treat_probs[
    (final_treat_probs["p_known_treatment_mc"] >= 0.90) &
    (final_treat_probs["p_known_treatment_mc"] < 0.999)
]

blocking_gps_v3 = []
for _, cluster in near_complete.iterrows():
    dhsclust = cluster["DHSCLUST"]
    unmatched_rows = cluster_gp_res[
        (cluster_gp_res["DHSCLUST"] == dhsclust) &
        (cluster_gp_res["reservation_dose"].isna()) &
        (cluster_gp_res["gp_lgd_code"].notna())
    ]
    for _, gp_row in unmatched_rows.iterrows():
        blocking_gps_v3.append({
            "DHSCLUST":             dhsclust,
            "p_known_treatment_mc": cluster["p_known_treatment_mc"],
            "gp_lgd_code":          gp_row["gp_lgd_code"],
            "gp_prob":              gp_row["gp_prob"],
        })

blocking_df_v3 = pd.DataFrame(blocking_gps_v3)
blocking_df_v3 = blocking_df_v3.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]],
    on="gp_lgd_code", how="left"
)

n_per_cluster_v3 = blocking_df_v3.groupby("DHSCLUST")["gp_lgd_code"].nunique()
single_blocker_clusters_v3 = n_per_cluster_v3[n_per_cluster_v3 == 1].index
single_blockers_v3 = blocking_df_v3[
    blocking_df_v3["DHSCLUST"].isin(single_blocker_clusters_v3)
].drop_duplicates(subset=["gp_lgd_code"]).copy()

print(f"Remaining single-blocker GPs: {len(single_blockers_v3)}")

# Flag newer GP codes
newer_prefixes = ("262", "293", "294", "295")
newer_gps = single_blockers_v3[
    single_blockers_v3["gp_lgd_code"].astype(str).str.startswith(newer_prefixes)
].copy()
older_gps = single_blockers_v3[
    ~single_blockers_v3["gp_lgd_code"].astype(str).str.startswith(newer_prefixes)
].copy()

print(f"  Newer GP codes (262/293/294/295xxx): {len(newer_gps)}")
print(f"  Older GP codes:                      {len(older_gps)}")

print(f"\nNewer GP codes:")
print(newer_gps[[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "subdistrict_lgd","p_known_treatment_mc","gp_prob"
]].sort_values("p_known_treatment_mc", ascending=False).to_string(index=False))

# For newer GPs, find villages that belong to them in the LGD file
# Then check if those villages previously belonged to an older GP
# that IS in gp_res_history_final

print(f"\nLooking up parent GPs via village overlap...")
already_matched = set(gp_res_history_final["gp_lgd_code"])

parent_gp_results = []
for _, gp_row in newer_gps.iterrows():
    lgd = gp_row["gp_lgd_code"]
    dist = gp_row["district_lgd"]

    # Find villages in this GP from LGD file
    villages_in_gp = lgd_rj[
        lgd_rj["gp_lgd_code"] == lgd
    ]["Village Census 2011 Code"].dropna().tolist()

    if not villages_in_gp:
        parent_gp_results.append({
            "gp_lgd_code": lgd,
            "gp_name_lgd": gp_row["gp_name_lgd"],
            "district_lgd": dist,
            "n_villages": 0,
            "parent_gp_lgd_code": None,
            "parent_gp_name": None,
            "parent_matched": False,
            "parent_dose": None
        })
        continue

    # Check if any nearby GP in same district has all of these villages
    # (i.e. was the parent GP before split)
    # Approach: find GPs in same district that share village census codes
    # This requires looking at the raw village file

    # Simpler approach: find GPs in same district that are matched
    # and have similar names
    same_district_matched = gp_res_history_final[
        gp_res_history_final["district_lgd"] == dist
    ].copy()

    gp_n = normalize_name(gp_row["gp_name_lgd"])
    same_district_matched["sim"] = same_district_matched["gp_name_lgd"].apply(
        lambda x: similarity(gp_n, normalize_name(str(x)))
    )
    if len(same_district_matched) > 0:
        best = same_district_matched.nlargest(1,"sim").iloc[0]
    else:
        best = None

    parent_gp_results.append({
        "gp_lgd_code":       lgd,
        "gp_name_lgd":       gp_row["gp_name_lgd"],
        "district_lgd":      dist,
        "subdistrict_lgd":   gp_row["subdistrict_lgd"],
        "n_villages":        len(villages_in_gp),
        "parent_gp_lgd_code":best["gp_lgd_code"] if best is not None else None,
        "parent_gp_name":    best["gp_name_lgd"] if best is not None else None,
        "parent_sim":        best["sim"] if best is not None else 0,
        "parent_matched":    best is not None and best["sim"] >= 0.75,
        "parent_dose":       best["reservation_dose"] if best is not None else None,
        "p_known":           gp_row["p_known_treatment_mc"],
        "gp_prob":           gp_row["gp_prob"]
    })

parent_df = pd.DataFrame(parent_gp_results)
print(f"\nNewer GPs with plausible parent GP (sim >= 0.75):")
print(parent_df[parent_df["parent_matched"]][[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "parent_gp_name","parent_sim","parent_dose","gp_prob"
]].to_string(index=False))

print(f"\nNewer GPs with no plausible parent:")
print(parent_df[~parent_df["parent_matched"]][[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "parent_gp_name","parent_sim","gp_prob"
]].to_string(index=False))

Remaining single-blocker GPs: 49
  Newer GP codes (262/293/294/295xxx): 22
  Older GP codes:                      27

Newer GP codes:
gp_lgd_code       gp_name_lgd     district_lgd subdistrict_lgd  p_known_treatment_mc  gp_prob
     294914        Sonthli(R)        Jhunjhunu       Nawalgarh              0.997696 0.002304
     295032        Kushalgarh      Chittorgarh      Rawatbhata              0.997110 0.002890
     295432   Badhli Nathusar        Jaisalmer         Pokaran              0.994312 0.005688
     294609         Bhojrasar            Churu    Sardarshahar              0.994000 0.006000
     294332            Dhatri            Churu       Sujangarh              0.993859 0.006141
     294952           Modayat          Bikaner           Bajju              0.990000 0.010000
     294053   Cheela Kashmeer          Bikaner           Bajju              0.988000 0.012000
     294703           6-8 Llw      Hanumangarh     Hanumangarh              0.980769 0.019231
     294261         

In [52]:
# Verify parent GP candidates in sarpanch files
candidates_to_check = [
    ("294914", "Sonthli(R)",  "jhunjhunu", "sohli"),
    ("295030", "Kazi",        "jhunjhunu", "kari"),
    ("295054", "Sarsanda",    "nagaur",    "sirasana"),
    ("262140", "Pundalsar",   "bikaner",   "udasar"),
    ("294609", "Bhojrasar",   "churu",     "harasar"),
]

print("Checking parent GP candidates in 2005 and 2010 sarpanch files:\n")
for lgd, gp_name, dist, parent_name in candidates_to_check:
    print(f"{gp_name} (LGD: {lgd}, {dist}) → parent candidate: {parent_name}")
    
    # Check 2005
    matches_2005 = sp2005[
        sp2005["district_norm_hist"] == dist
    ].copy()
    matches_2005["sim"] = matches_2005["gp_norm"].apply(
        lambda x: similarity(parent_name, x)
    )
    best_2005 = matches_2005.nlargest(1,"sim").iloc[0]
    print(f"  2005: {best_2005['gp_norm']:<20} sim={best_2005['sim']:.3f}  "
          f"categ={best_2005['CATEG. OF POST OF SARPANCH']}  "
          f"reserved={best_2005['reserved_women_2005']}")

    # Check 2010
    matches_2010 = sp2010[
        sp2010["district_norm_hist"] == dist
    ].copy()
    matches_2010["sim"] = matches_2010["gp_norm"].apply(
        lambda x: similarity(parent_name, x)
    )
    best_2010 = matches_2010.nlargest(1,"sim").iloc[0]
    print(f"  2010: {best_2010['gp_norm']:<20} sim={best_2010['sim']:.3f}  "
          f"categ={best_2010['Ward category']}  "
          f"reserved={best_2010['reserved_women_2010']}")

    # Also try matching the GP's own name directly
    matches_2005["sim_direct"] = matches_2005["gp_norm"].apply(
        lambda x: similarity(normalize_name(gp_name), x)
    )
    best_direct_2005 = matches_2005.nlargest(1,"sim_direct").iloc[0]
    matches_2010["sim_direct"] = matches_2010["gp_norm"].apply(
        lambda x: similarity(normalize_name(gp_name), x)
    )
    best_direct_2010 = matches_2010.nlargest(1,"sim_direct").iloc[0]
    print(f"  Direct 2005: {best_direct_2005['gp_norm']:<20} "
          f"sim={best_direct_2005['sim_direct']:.3f}")
    print(f"  Direct 2010: {best_direct_2010['gp_norm']:<20} "
          f"sim={best_direct_2010['sim_direct']:.3f}")
    print()

Checking parent GP candidates in 2005 and 2010 sarpanch files:

Sonthli(R) (LGD: 294914, jhunjhunu) → parent candidate: sohli
  2005: soheli               sim=0.909  categ=GEN  reserved=0
  2010: sohli                sim=1.000  categ=GEN  reserved=0
  Direct 2005: soheli               sim=0.714
  Direct 2010: sohli                sim=0.769

Kazi (LGD: 295030, jhunjhunu) → parent candidate: kari
  2005: kari                 sim=1.000  categ=GEN  reserved=0
  2010: kari                 sim=1.000  categ=OBCW  reserved=1
  Direct 2005: kari                 sim=0.750
  Direct 2010: kari                 sim=0.750

Sarsanda (LGD: 295054, nagaur) → parent candidate: sirasana
  2005: sirasana             sim=1.000  categ=OBC  reserved=0
  2010: sirasna              sim=0.933  categ=GEN  reserved=0
  Direct 2005: sirasana             sim=0.750
  Direct 2010: sarsani              sim=0.800

Pundalsar (LGD: 262140, bikaner) → parent candidate: udasar
  2005: udasar               sim=1.000  categ=O

In [53]:
# =============================================================================
# Stage I — accept verified parent GP matches and direct matches
# =============================================================================

stage_i_entries = [
    {
        "gp_lgd_code":        "294914",
        "gp_name_lgd":        "Sonthli(R)",
        "district_lgd":       "Jhunjhunu",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="294914"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 0,
        "reserved_women_2010": 0,
        "reservation_dose_n":  0,
        "reservation_dose":    "never",
        "match_stage":        "I_parent_gp_inference",
        "notes":              "Rural split of Sohli; parent never reserved in 2005/2010"
    },
    {
        "gp_lgd_code":        "262140",
        "gp_name_lgd":        "Pundalsar",
        "district_lgd":       "Bikaner",
        "subdistrict_lgd":    gp_lookup[gp_lookup["gp_lgd_code"]=="262140"]["subdistrict_lgd"].iloc[0],
        "reserved_women_2005": 1,
        "reserved_women_2010": 1,
        "reservation_dose_n":  2,
        "reservation_dose":    "twice",
        "match_stage":        "I_parent_gp_inference",
        "notes":              "Punrasar in sarpanch records (sim=0.824 both years); twice reserved"
    },
]

stage_i_df = pd.DataFrame(stage_i_entries)

# Verify no overlap
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_i_df["gp_lgd_code"]) & already
print(f"Overlap with existing history (should be 0): {len(overlap)}")
print(f"\nStage I entries:")
print(stage_i_df[[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "reserved_women_2005","reserved_women_2010",
    "reservation_dose","notes"
]].to_string(index=False))

# Add to history and rerun
gp_res_history_final_v8 = pd.concat([
    gp_res_history_final,
    stage_i_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
        "reserved_women_2005","reserved_women_2010",
        "reservation_dose_n","reservation_dose","match_stage"
    ]]
], ignore_index=True)
assert gp_res_history_final_v8["gp_lgd_code"].duplicated().sum() == 0

# Rebuild
cluster_gp_res_v8 = mc_long.merge(
    gp_res_history_final_v8[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

cluster_known_v8 = (
    cluster_gp_res_v8[cluster_gp_res_v8["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v8.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v8.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v8.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v8 = all_clusters.merge(cluster_known_v8, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v8[col] = treat_probs_v8[col].fillna(0.0)
treat_probs_v8["p_unknown_treatment_mc"] = (
    1 - treat_probs_v8["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v8["p_any_reserved_mc"] = (
    treat_probs_v8["p_once_mc"] + treat_probs_v8["p_twice_mc"]
)
treat_probs_v8["expected_dose_mc"] = (
    treat_probs_v8["p_once_mc"] + 2 * treat_probs_v8["p_twice_mc"]
)
treat_probs_v8["treatment_certainty_mc"] = treat_probs_v8[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_v8 = (
    cluster_gp_res_v8[cluster_gp_res_v8["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v8 = treat_probs_v8.merge(primary_dose_v8, on="DHSCLUST", how="left")
treat_probs_v8["primary_gp_norm"] = treat_probs_v8["primary_gp"].apply(normalize_lgd_code)
treat_probs_v8["primary_gp_has_history"] = treat_probs_v8["primary_gp_norm"].isin(
    set(gp_res_history_final_v8["gp_lgd_code"])
)

working_v8 = treat_probs_v8[treat_probs_v8["primary_gp_has_history"]].copy()

print(f"\nWorking clusters:       {len(working_v8)}")
print(f"Fully linked (>=99.9%): {(working_v8['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v8['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v8['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v8['p_known_treatment_mc'].mean():.3f}")

# Update
gp_res_history_final = gp_res_history_final_v8
cluster_gp_res       = cluster_gp_res_v8
treat_probs_new      = treat_probs_v8
working_clusters_new = working_v8
final_treat_probs    = working_v8[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
print(f"\nSaved. Total GP histories: {len(gp_res_history_final)}")

Overlap with existing history (should be 0): 0

Stage I entries:
gp_lgd_code gp_name_lgd district_lgd  reserved_women_2005  reserved_women_2010 reservation_dose                                                               notes
     294914  Sonthli(R)    Jhunjhunu                    0                    0            never            Rural split of Sohli; parent never reserved in 2005/2010
     262140   Pundalsar      Bikaner                    1                    1            twice Punrasar in sarpanch records (sim=0.824 both years); twice reserved

Working clusters:       730
Fully linked (>=99.9%): 121
>=90% linked:           273
>=75% linked:           470
Mean p_known:           0.807

Saved. Total GP histories: 6486


In [54]:
# =============================================================================
# Remove Stage I entries and revert to 119 fully linked
# =============================================================================

# Remove the 2 Stage I GPs from history
gp_res_history_final = gp_res_history_final[
    gp_res_history_final["match_stage"] != "I_parent_gp_inference"
].copy()
print(f"GP histories after removing Stage I: {len(gp_res_history_final)}")

# Rebuild cluster_gp_res
cluster_gp_res = mc_long.merge(
    gp_res_history_final[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

# Recompute treat_probs
cluster_known = (
    cluster_gp_res[cluster_gp_res["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_final = all_clusters.merge(cluster_known, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_final[col] = treat_probs_final[col].fillna(0.0)
treat_probs_final["p_unknown_treatment_mc"] = (
    1 - treat_probs_final["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_final["p_any_reserved_mc"] = (
    treat_probs_final["p_once_mc"] + treat_probs_final["p_twice_mc"]
)
treat_probs_final["expected_dose_mc"] = (
    treat_probs_final["p_once_mc"] + 2 * treat_probs_final["p_twice_mc"]
)
treat_probs_final["treatment_certainty_mc"] = treat_probs_final[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_final = (
    cluster_gp_res[cluster_gp_res["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_final = treat_probs_final.merge(primary_dose_final, on="DHSCLUST", how="left")
treat_probs_final["primary_gp_norm"] = treat_probs_final["primary_gp"].apply(normalize_lgd_code)
treat_probs_final["primary_gp_has_history"] = treat_probs_final["primary_gp_norm"].isin(
    set(gp_res_history_final["gp_lgd_code"])
)

working_final = treat_probs_final[treat_probs_final["primary_gp_has_history"]].copy()
final_treat_probs = working_final[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

print(f"Working clusters:       {len(final_treat_probs)}")
print(f"Fully linked (>=99.9%): "
      f"{(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")

# =============================================================================
# Treatment dosage distribution for fully linked clusters
# =============================================================================

fully_linked = final_treat_probs[
    final_treat_probs["p_known_treatment_mc"] >= 0.999
].copy()

print(f"\n{'='*60}")
print(f"TREATMENT DOSE DISTRIBUTION — 119 FULLY LINKED CLUSTERS")
print(f"{'='*60}")

print(f"\nCluster-level dose distribution:")
print(f"  Never reserved (p_never=1.0):  "
      f"{(fully_linked['p_never_mc'] >= 0.999).sum()} clusters "
      f"({(fully_linked['p_never_mc'] >= 0.999).mean():.1%})")
print(f"  Once reserved  (p_once=1.0):   "
      f"{(fully_linked['p_once_mc'] >= 0.999).sum()} clusters "
      f"({(fully_linked['p_once_mc'] >= 0.999).mean():.1%})")
print(f"  Twice reserved (p_twice=1.0):  "
      f"{(fully_linked['p_twice_mc'] >= 0.999).sum()} clusters "
      f"({(fully_linked['p_twice_mc'] >= 0.999).mean():.1%})")

print(f"\nMean dose probabilities (fully linked):")
print(f"  Mean p_never_mc:   {fully_linked['p_never_mc'].mean():.3f}")
print(f"  Mean p_once_mc:    {fully_linked['p_once_mc'].mean():.3f}")
print(f"  Mean p_twice_mc:   {fully_linked['p_twice_mc'].mean():.3f}")
print(f"  Mean expected_dose:{fully_linked['expected_dose_mc'].mean():.3f}")

print(f"\nPrimary GP dose distribution:")
print(fully_linked["primary_gp_dose"].value_counts(dropna=False))

print(f"\nDistrict breakdown:")
district_dose = (
    fully_linked.merge(
        mc[["DHSCLUST","DHSREGNA"]].drop_duplicates(),
        on="DHSCLUST", how="left", suffixes=("","_mc")
    )
    .groupby("DHSREGNA")
    .agg(
        n_clusters  = ("DHSCLUST",    "count"),
        n_never     = ("p_never_mc",  lambda x: (x>=0.999).sum()),
        n_once      = ("p_once_mc",   lambda x: (x>=0.999).sum()),
        n_twice     = ("p_twice_mc",  lambda x: (x>=0.999).sum()),
        mean_p_known= ("p_known_treatment_mc","mean")
    )
    .sort_values("n_clusters", ascending=False)
)
print(district_dose.to_string())

# Compare with Beaman's dose distribution
print(f"\n{'='*60}")
print(f"COMPARISON WITH BEAMAN (165 village councils)")
print(f"{'='*60}")
print(f"  {'Dose':<15} {'Beaman N':>10} {'Beaman %':>10} "
      f"{'Ours N':>10} {'Ours %':>10}")
print(f"  {'-'*55}")
beaman = {"never": 222, "once": 213, "twice": 60}
beaman_total = sum(beaman.values())
ours = {
    "never": (fully_linked["p_never_mc"] >= 0.999).sum(),
    "once":  (fully_linked["p_once_mc"]  >= 0.999).sum(),
    "twice": (fully_linked["p_twice_mc"] >= 0.999).sum()
}
our_total = sum(ours.values())
for dose in ["never","once","twice"]:
    print(f"  {dose:<15} {beaman[dose]:>10} "
          f"{beaman[dose]/beaman_total:>10.1%} "
          f"{ours[dose]:>10} "
          f"{ours[dose]/our_total:>10.1%}")

# Save final outputs
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
print(f"\nSaved final outputs to: {OUTPUT_DIR}")

GP histories after removing Stage I: 6484
Working clusters:       730
Fully linked (>=99.9%): 119

TREATMENT DOSE DISTRIBUTION — 119 FULLY LINKED CLUSTERS

Cluster-level dose distribution:
  Never reserved (p_never=1.0):  3 clusters (2.5%)
  Once reserved  (p_once=1.0):   6 clusters (5.0%)
  Twice reserved (p_twice=1.0):  0 clusters (0.0%)

Mean dose probabilities (fully linked):
  Mean p_never_mc:   0.342
  Mean p_once_mc:    0.512
  Mean p_twice_mc:   0.145
  Mean expected_dose:0.803

Primary GP dose distribution:
primary_gp_dose
once     62
never    39
twice    18
Name: count, dtype: int64

District breakdown:
              n_clusters  n_never  n_once  n_twice  mean_p_known
DHSREGNA                                                        
Bhilwara              11        0       1        0        1.0000
Hanumangarh           10        0       0        0        0.9999
Ajmer                  8        0       0        0        1.0000
Sirohi                 8        0       0        0    

In [55]:
# =============================================================================
# Detailed dose distribution analysis
# =============================================================================

print("="*60)
print("FULLY LINKED CLUSTERS (119) — DETAILED DOSE ANALYSIS")
print("="*60)

# Distribution of treatment certainty
print(f"\nTreatment certainty distribution:")
print(f"  Mean:   {fully_linked['treatment_certainty_mc'].mean():.3f}")
print(f"  Median: {fully_linked['treatment_certainty_mc'].median():.3f}")
print(f"  >=0.90: {(fully_linked['treatment_certainty_mc'] >= 0.90).sum()}")
print(f"  >=0.75: {(fully_linked['treatment_certainty_mc'] >= 0.75).sum()}")
print(f"  >=0.50: {(fully_linked['treatment_certainty_mc'] >= 0.50).sum()}")

# Classify each cluster by its dominant dose
fully_linked["dominant_dose"] = fully_linked[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].idxmax(axis=1).str.replace("p_","").str.replace("_mc","")

print(f"\nDominant dose distribution (highest probability dose):")
print(fully_linked["dominant_dose"].value_counts())
print(f"\n  {'Dose':<10} {'N':>6} {'%':>8} {'Mean p':>10} {'Mean certainty':>16}")
print(f"  {'-'*52}")
for dose in ["never","once","twice"]:
    subset = fully_linked[fully_linked["dominant_dose"]==dose]
    col = f"p_{dose}_mc"
    print(f"  {dose:<10} {len(subset):>6} "
          f"{len(subset)/len(fully_linked):>8.1%} "
          f"{subset[col].mean():>10.3f} "
          f"{subset['treatment_certainty_mc'].mean():>16.3f}")

# Distribution of p_twice specifically (key treatment in BDPT)
print(f"\np_twice_mc distribution among fully linked clusters:")
bins = [0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.01]
labels = ["0-10%","10-25%","25-50%","50-75%","75-90%","90-100%"]
fully_linked["twice_bin"] = pd.cut(
    fully_linked["p_twice_mc"], bins=bins, labels=labels, right=False
)
print(fully_linked["twice_bin"].value_counts().sort_index())

# Effective sample sizes by dose
print(f"\nEffective sample for dose comparisons:")
print(f"  Never vs Once+Twice (any reserved):")
n_never_dom  = (fully_linked["dominant_dose"]=="never").sum()
n_once_dom   = (fully_linked["dominant_dose"]=="once").sum()
n_twice_dom  = (fully_linked["dominant_dose"]=="twice").sum()
print(f"    Never dominant:  {n_never_dom}")
print(f"    Once dominant:   {n_once_dom}")
print(f"    Twice dominant:  {n_twice_dom}")
print(f"\n  Compare with Beaman's 165 GPs:")
print(f"    Never:  222 (44.8%)")
print(f"    Once:   213 (43.0%)")
print(f"    Twice:   60 (12.1%)")
print(f"\n  Our 119 fully linked:")
print(f"    Never:  {n_never_dom} ({n_never_dom/119:.1%})")
print(f"    Once:   {n_once_dom} ({n_once_dom/119:.1%})")
print(f"    Twice:  {n_twice_dom} ({n_twice_dom/119:.1%})")

# Full working sample dose distribution for comparison
print(f"\n{'='*60}")
print(f"FULL WORKING SAMPLE (730 clusters) — DOSE DISTRIBUTION")
print(f"{'='*60}")
final_treat_probs["dominant_dose"] = final_treat_probs[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].idxmax(axis=1).str.replace("p_","").str.replace("_mc","")

print(f"\nDominant dose (full working sample):")
print(final_treat_probs["dominant_dose"].value_counts())
print(f"\nPrimary GP dose (full working sample):")
print(final_treat_probs["primary_gp_dose"].value_counts(dropna=False))
print(f"\nMean probabilities (full working sample):")
print(f"  p_never_mc:  {final_treat_probs['p_never_mc'].mean():.3f}")
print(f"  p_once_mc:   {final_treat_probs['p_once_mc'].mean():.3f}")
print(f"  p_twice_mc:  {final_treat_probs['p_twice_mc'].mean():.3f}")
print(f"  p_known_mc:  {final_treat_probs['p_known_treatment_mc'].mean():.3f}")
print(f"  certainty:   {final_treat_probs['treatment_certainty_mc'].mean():.3f}")

FULLY LINKED CLUSTERS (119) — DETAILED DOSE ANALYSIS

Treatment certainty distribution:
  Mean:   0.747
  Median: 0.729
  >=0.90: 30
  >=0.75: 52
  >=0.50: 113

Dominant dose distribution (highest probability dose):
dominant_dose
once     65
never    38
twice    16
Name: count, dtype: int64

  Dose            N        %     Mean p   Mean certainty
  ----------------------------------------------------
  never          38    31.9%      0.733            0.733
  once           65    54.6%      0.776            0.776
  twice          16    13.4%      0.663            0.663

p_twice_mc distribution among fully linked clusters:
twice_bin
0-10%      78
10-25%     14
25-50%     15
50-75%      8
75-90%      2
90-100%     2
Name: count, dtype: int64

Effective sample for dose comparisons:
  Never vs Once+Twice (any reserved):
    Never dominant:  38
    Once dominant:   65
    Twice dominant:  16

  Compare with Beaman's 165 GPs:
    Never:  222 (44.8%)
    Once:   213 (43.0%)
    Twice:   60 (1

In [56]:
# =============================================================================
# How many more single-blocker GPs needed to reach 150 fully linked?
# =============================================================================

# Current state
current_fully_linked = (final_treat_probs["p_known_treatment_mc"] >= 0.999).sum()
target = 150
gap = target - current_fully_linked
print(f"Current fully linked: {current_fully_linked}")
print(f"Target:               {target}")
print(f"Gap:                  {gap} clusters needed")

# Recompute current single-blocker landscape
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)
analysis_clusters = set(final_treat_probs["DHSCLUST"]) - isolated_clusters

near_complete = final_treat_probs[
    (final_treat_probs["p_known_treatment_mc"] >= 0.90) &
    (final_treat_probs["p_known_treatment_mc"] < 0.999)
].copy()

# For each near-complete cluster, count unmatched GPs
blocking = []
for _, cluster in near_complete.iterrows():
    unmatched_rows = cluster_gp_res[
        (cluster_gp_res["DHSCLUST"] == cluster["DHSCLUST"]) &
        (cluster_gp_res["reservation_dose"].isna()) &
        (cluster_gp_res["gp_lgd_code"].notna())
    ]
    blocking.append({
        "DHSCLUST":             cluster["DHSCLUST"],
        "p_known_treatment_mc": cluster["p_known_treatment_mc"],
        "n_unmatched_gps":      len(unmatched_rows),
        "unmatched_gps":        set(unmatched_rows["gp_lgd_code"].tolist())
    })

blocking_df = pd.DataFrame(blocking).sort_values(
    "p_known_treatment_mc", ascending=False
)

# Clusters with exactly 1 unmatched GP (single-blockers)
single = blocking_df[blocking_df["n_unmatched_gps"] == 1]
double = blocking_df[blocking_df["n_unmatched_gps"] == 2]
triple = blocking_df[blocking_df["n_unmatched_gps"] == 3]

print(f"\nNear-complete clusters (90-99% known mass): {len(near_complete)}")
print(f"  Single-blocker (1 unmatched GP):  {len(single)} clusters")
print(f"  Double-blocker (2 unmatched GPs): {len(double)} clusters")
print(f"  Triple-blocker (3 unmatched GPs): {len(triple)} clusters")
print(f"  4+ blockers:                      "
      f"{len(blocking_df) - len(single) - len(double) - len(triple)} clusters")

# How many unique GPs across all single-blockers?
single_blocker_gps = set()
for gps in single["unmatched_gps"]:
    single_blocker_gps.update(gps)
print(f"\nUnique single-blocker GPs: {len(single_blocker_gps)}")
print(f"  → Matching all {len(single_blocker_gps)} would add {len(single)} fully linked clusters")

# Double-blockers: unique GPs
double_blocker_gps = set()
for gps in double["unmatched_gps"]:
    double_blocker_gps.update(gps)
double_only = double_blocker_gps - single_blocker_gps
print(f"\nAdditional unique GPs in double-blockers: {len(double_only)}")
print(f"  → Matching all would add up to {len(double)} more clusters")

# Cumulative projection
print(f"\nCumulative projection:")
print(f"  Current fully linked:              {current_fully_linked}")
print(f"  + all single-blocker GPs ({len(single_blocker_gps)} GPs): "
      f"{current_fully_linked + len(single)} clusters")
print(f"  + all double-blocker GPs ({len(double_only)} more GPs): "
      f"{current_fully_linked + len(single) + len(double)} clusters")

# How many GPs to reach exactly 150?
print(f"\nTo reach {target} fully linked:")
print(f"  Need {gap} more clusters")
print(f"  Single-blockers alone give {len(single)} — "
      f"{'sufficient' if len(single) >= gap else f'still need {gap - len(single)} more from double-blockers'}")

# List single-blocker GPs with their names
single_blockers_named = []
for _, row in single.iterrows():
    gp_code = list(row["unmatched_gps"])[0]
    gp_info = gp_lookup[gp_lookup["gp_lgd_code"] == gp_code]
    gp_name = gp_info["gp_name_lgd"].iloc[0] if len(gp_info) > 0 else "Unknown"
    district = gp_info["district_lgd"].iloc[0] if len(gp_info) > 0 else "Unknown"
    samiti   = gp_info["subdistrict_lgd"].iloc[0] if len(gp_info) > 0 else "Unknown"
    single_blockers_named.append({
        "DHSCLUST":             row["DHSCLUST"],
        "p_known_treatment_mc": row["p_known_treatment_mc"],
        "gp_lgd_code":          gp_code,
        "gp_name":              gp_name,
        "district":             district,
        "subdistrict_samiti":   samiti
    })

single_blockers_named_df = pd.DataFrame(single_blockers_named).sort_values(
    "p_known_treatment_mc", ascending=False
)
print(f"\nSingle-blocker GPs ranked by cluster p_known:")
print(single_blockers_named_df[[
    "gp_lgd_code","gp_name","district",
    "subdistrict_samiti","p_known_treatment_mc"
]].to_string(index=False))

Current fully linked: 119
Target:               150
Gap:                  31 clusters needed

Near-complete clusters (90-99% known mass): 154
  Single-blocker (1 unmatched GP):  58 clusters
  Double-blocker (2 unmatched GPs): 63 clusters
  Triple-blocker (3 unmatched GPs): 21 clusters
  4+ blockers:                      12 clusters

Unique single-blocker GPs: 49
  → Matching all 49 would add 58 fully linked clusters

Additional unique GPs in double-blockers: 122
  → Matching all would add up to 63 more clusters

Cumulative projection:
  Current fully linked:              119
  + all single-blocker GPs (49 GPs): 177 clusters
  + all double-blocker GPs (122 more GPs): 240 clusters

To reach 150 fully linked:
  Need 31 more clusters
  Single-blockers alone give 58 — sufficient

Single-blocker GPs ranked by cluster p_known:
gp_lgd_code                gp_name         district subdistrict_samiti  p_known_treatment_mc
     295432        Badhli Nathusar        Jaisalmer            Pokaran     

In [57]:
# =============================================================================
# Fuzzy match top 31 single-blocker GPs against ALL districts in sarpanch files
# (not just their own district — checking for boundary changes)
# =============================================================================

# Get top 31 unique single-blocker GPs by p_known
top_31 = (
    single_blockers_named_df
    .drop_duplicates(subset=["gp_lgd_code"])
    .head(31)
    .copy()
)

print(f"Top 31 unique single-blocker GPs:")
print(top_31[["gp_lgd_code","gp_name","district","subdistrict_samiti",
              "p_known_treatment_mc"]].to_string(index=False))

# Fuzzy match each GP against ALL records in 2005 sarpanch (no district filter)
print(f"\nRunning cross-district fuzzy match against 2005 sarpanch...")
results_crossdistrict_2005 = []
for _, gp_row in top_31.iterrows():
    gp_n     = normalize_name(gp_row["gp_name"])
    lgd      = gp_row["gp_lgd_code"]
    own_dist = normalize_district_historical(gp_row["district"])

    sp2005["sim"] = sp2005["gp_norm"].apply(lambda x: similarity(gp_n, x))
    top_matches   = sp2005.nlargest(3, "sim")

    for _, match in top_matches.iterrows():
        results_crossdistrict_2005.append({
            "gp_lgd_code":    lgd,
            "gp_name":        gp_row["gp_name"],
            "own_district":   gp_row["district"],
            "match_gp":       match["gp_norm"],
            "match_district": match["district_norm_hist"],
            "match_samiti":   match["samiti_norm"],
            "similarity":     match["sim"],
            "categ_2005":     match["CATEG. OF POST OF SARPANCH"],
            "reserved_2005":  match["reserved_women_2005"],
            "same_district":  match["district_norm_hist"] == own_dist
        })

results_2005_cross = pd.DataFrame(results_crossdistrict_2005)

# Focus on high-similarity matches in DIFFERENT districts
cross_district_hits_2005 = results_2005_cross[
    (results_2005_cross["similarity"] >= 0.80) &
    (~results_2005_cross["same_district"])
].copy()

print(f"\nCross-district matches >= 0.80 in 2005:")
print(f"  Total hits: {len(cross_district_hits_2005)}")
print(f"  Unique GPs with cross-district hit: "
      f"{cross_district_hits_2005['gp_lgd_code'].nunique()}")

print(f"\nCross-district hits:")
print(cross_district_hits_2005[[
    "gp_lgd_code","gp_name","own_district",
    "match_gp","match_district","similarity","categ_2005"
]].sort_values("similarity", ascending=False).to_string(index=False))

# Repeat for 2010
print(f"\nRunning cross-district fuzzy match against 2010 sarpanch...")
results_crossdistrict_2010 = []
for _, gp_row in top_31.iterrows():
    gp_n     = normalize_name(gp_row["gp_name"])
    lgd      = gp_row["gp_lgd_code"]
    own_dist = normalize_district_historical(gp_row["district"])

    sp2010["sim"] = sp2010["gp_norm"].apply(lambda x: similarity(gp_n, x))
    top_matches   = sp2010.nlargest(3, "sim")

    for _, match in top_matches.iterrows():
        results_crossdistrict_2010.append({
            "gp_lgd_code":    lgd,
            "gp_name":        gp_row["gp_name"],
            "own_district":   gp_row["district"],
            "match_gp":       match["gp_norm"],
            "match_district": match["district_norm_hist"],
            "match_samiti":   match["samiti_norm"],
            "similarity":     match["sim"],
            "categ_2010":     match["Ward category"],
            "reserved_2010":  match["reserved_women_2010"],
            "same_district":  match["district_norm_hist"] == own_dist
        })

results_2010_cross = pd.DataFrame(results_crossdistrict_2010)

cross_district_hits_2010 = results_2010_cross[
    (results_2010_cross["similarity"] >= 0.80) &
    (~results_2010_cross["same_district"])
].copy()

print(f"\nCross-district matches >= 0.80 in 2010:")
print(f"  Total hits: {len(cross_district_hits_2010)}")
print(f"  Unique GPs with cross-district hit: "
      f"{cross_district_hits_2010['gp_lgd_code'].nunique()}")

print(f"\nCross-district hits:")
print(cross_district_hits_2010[[
    "gp_lgd_code","gp_name","own_district",
    "match_gp","match_district","similarity","categ_2010"
]].sort_values("similarity", ascending=False).to_string(index=False))

# Combined: GPs with cross-district hits in BOTH years
both_cross = set(cross_district_hits_2005["gp_lgd_code"]) & \
             set(cross_district_hits_2010["gp_lgd_code"])
print(f"\nGPs with cross-district hits in BOTH years: {len(both_cross)}")
for lgd in both_cross:
    hit_2005 = cross_district_hits_2005[
        cross_district_hits_2005["gp_lgd_code"]==lgd
    ].iloc[0]
    hit_2010 = cross_district_hits_2010[
        cross_district_hits_2010["gp_lgd_code"]==lgd
    ].iloc[0]
    print(f"\n  {hit_2005['gp_name']} (LGD: {lgd}, own district: {hit_2005['own_district']})")
    print(f"    2005: {hit_2005['match_gp']} in {hit_2005['match_district']} "
          f"(sim={hit_2005['similarity']:.3f}, categ={hit_2005['categ_2005']})")
    print(f"    2010: {hit_2010['match_gp']} in {hit_2010['match_district']} "
          f"(sim={hit_2010['similarity']:.3f}, categ={hit_2010['categ_2010']})")

Top 31 unique single-blocker GPs:
gp_lgd_code                gp_name         district subdistrict_samiti  p_known_treatment_mc
     295432        Badhli Nathusar        Jaisalmer            Pokaran              0.997936
     294914             Sonthli(R)        Jhunjhunu          Nawalgarh              0.997696
     295032             Kushalgarh      Chittorgarh         Rawatbhata              0.997110
      34833    Laxmipura Khankaraa            Baran         Kishanganj              0.997000
      40260                 Kolana             Kota            Ladpura              0.996917
      42040                  Awada             Tonk            Malpura              0.995227
      38034 Chak Jawala Singh Wala      Hanumangarh        Hanumangarh              0.995000
      36361             Jawti Kala            Bundi              Bundi              0.995000
      38102            Peerkamaria      Hanumangarh              Tibbi              0.994813
     294609              Bhojrasar  

In [58]:
# =============================================================================
# Key finding: Kekri was part of Ajmer in 2005/2010
# Add alias and recheck affected GPs
# =============================================================================

# Update DISTRICT_ALIASES
new_aliases = {
    "kekri":          "ajmer",    # Kekri carved from Ajmer in 2023
    "beawar":         "ajmer",    # Beawar carved from Ajmer in 2023
    # Already have: neemkathana -> sikar
}

print("Checking which of our unmatched GPs are in Kekri or Beawar:")
for dist in ["kekri","beawar"]:
    n_unmatched = (
        gp_lookup[
            ~gp_lookup["gp_lgd_code"].isin(set(gp_res_history_final["gp_lgd_code"]))
        ]["district_norm"] == dist
    ).sum()
    n_single_blocker = sum(
        1 for _, r in single_blockers_named_df.iterrows()
        if normalize_name(r["district"]) == dist
    )
    print(f"  {dist}: {n_unmatched} unmatched GPs total, "
          f"{n_single_blocker} are single-blockers")

# Check Gopalpura specifically under Ajmer
print("\nGopalpura (294500) under Ajmer in sarpanch files:")
gp_n = normalize_name("Gopalpura")

matches_2005_ajmer = sp2005[sp2005["district_norm_hist"]=="ajmer"].copy()
matches_2005_ajmer["sim"] = matches_2005_ajmer["gp_norm"].apply(
    lambda x: similarity(gp_n, x)
)
best_2005 = matches_2005_ajmer.nlargest(3,"sim")
print("2005 Ajmer top matches:")
print(best_2005[["gp_norm","samiti_norm","sim",
                 "CATEG. OF POST OF SARPANCH","reserved_women_2005"]].to_string(index=False))

matches_2010_ajmer = sp2010[sp2010["district_norm_hist"]=="ajmer"].copy()
matches_2010_ajmer["sim"] = matches_2010_ajmer["gp_norm"].apply(
    lambda x: similarity(gp_n, x)
)
best_2010 = matches_2010_ajmer.nlargest(3,"sim")
print("\n2010 Ajmer top matches:")
print(best_2010[["gp_norm","samiti_norm","sim",
                 "Ward category","reserved_women_2010"]].to_string(index=False))

# Also check Hanootiya under Ajmer (Beawar -> Ajmer)
print("\n\nHanootiya (33758) under Ajmer in sarpanch files:")
gp_n2 = normalize_name("Hanootiya")
matches_2005_ajmer["sim"] = matches_2005_ajmer["gp_norm"].apply(
    lambda x: similarity(gp_n2, x)
)
print("2005 Ajmer top matches:")
print(matches_2005_ajmer.nlargest(3,"sim")[
    ["gp_norm","samiti_norm","sim","CATEG. OF POST OF SARPANCH","reserved_women_2005"]
].to_string(index=False))

matches_2010_ajmer["sim"] = matches_2010_ajmer["gp_norm"].apply(
    lambda x: similarity(gp_n2, x)
)
print("\n2010 Ajmer top matches:")
print(matches_2010_ajmer.nlargest(3,"sim")[
    ["gp_norm","samiti_norm","sim","Ward category","reserved_women_2010"]
].to_string(index=False))

Checking which of our unmatched GPs are in Kekri or Beawar:
  kekri: 45 unmatched GPs total, 1 are single-blockers
  beawar: 103 unmatched GPs total, 1 are single-blockers

Gopalpura (294500) under Ajmer in sarpanch files:
2005 Ajmer top matches:
 gp_norm samiti_norm      sim CATEG. OF POST OF SARPANCH  reserved_women_2005
 malpura      jawaja 0.750000                        GEN                    0
   palra   shrinagar 0.714286                        GEN                    0
somalpur    pisangan 0.705882                        GEN                    0

2010 Ajmer top matches:
gp_norm samiti_norm      sim Ward category  reserved_women_2010
malpura      jawaja 0.750000           GEN                    0
  palra    srinagar 0.714286           GEN                    0
 goyala      bhinay 0.666667          GENW                    1


Hanootiya (33758) under Ajmer in sarpanch files:
2005 Ajmer top matches:
  gp_norm samiti_norm      sim CATEG. OF POST OF SARPANCH  reserved_women_2005
 hanut

In [59]:
# =============================================================================
# Update district aliases for Kekri and Beawar → Ajmer
# and rerun Stage F2 to capture newly matchable GPs
# =============================================================================

# Update aliases
DISTRICT_ALIASES["kekri"]  = "ajmer"
DISTRICT_ALIASES["beawar"] = "ajmer"

print(f"Updated DISTRICT_ALIASES: kekri→ajmer, beawar→ajmer")
print(f"Total aliases: {len(DISTRICT_ALIASES)}")

# Re-normalize all district columns
for df_name, df in [("sp2005", sp2005), ("sp2010", sp2010),
                     ("res_clean", res_clean), ("gp_lookup", gp_lookup)]:
    if df_name == "gp_lookup":
        df["district_norm_hist"] = df["district_lgd"].apply(
            normalize_district_historical
        )
    else:
        df["district_norm_hist"] = df["district_norm"].apply(
            lambda x: DISTRICT_ALIASES.get(x, x)
        )

# Verify Hanootiya now finds match
print("\nHanootiya (33758, Beawar→Ajmer) after alias fix:")
gp_n = normalize_name("Hanootiya")
dist = "ajmer"  # after alias

m2005 = sp2005[sp2005["district_norm_hist"]==dist].copy()
m2005["sim"] = m2005["gp_norm"].apply(lambda x: similarity(gp_n, x))
best_2005 = m2005.nlargest(1,"sim").iloc[0]
print(f"  2005: {best_2005['gp_norm']} (sim={best_2005['sim']:.3f}, "
      f"categ={best_2005['CATEG. OF POST OF SARPANCH']}, "
      f"reserved={best_2005['reserved_women_2005']})")

m2010 = sp2010[sp2010["district_norm_hist"]==dist].copy()
m2010["sim"] = m2010["gp_norm"].apply(lambda x: similarity(gp_n, x))
best_2010 = m2010.nlargest(1,"sim").iloc[0]
print(f"  2010: {best_2010['gp_norm']} (sim={best_2010['sim']:.3f}, "
      f"categ={best_2010['Ward category']}, "
      f"reserved={best_2010['reserved_women_2010']})")

# Rebuild sarpanch lookups with updated aliases
sp2005_by_district = (
    sp2005.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_rows              = ("gp_norm",             "size"),
        n_categ_values      = ("CATEG. OF POST OF SARPANCH","nunique"),
        categ_values        = ("CATEG. OF POST OF SARPANCH",
                               lambda x: " | ".join(sorted(set(x.dropna())))),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        samiti              = ("samiti_norm",         "first")
    ).reset_index()
)

sp2010_by_district = (
    sp2010.groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_rows              = ("gp_norm",             "size"),
        n_categ_values      = ("Ward category",       "nunique"),
        categ_values        = ("Ward category",
                               lambda x: " | ".join(sorted(set(
                                   str(v).strip() for v in x.dropna())))),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        samiti              = ("samiti_norm",         "first")
    ).reset_index()
)

# Rerun fuzzy match for all currently unmatched GPs
already_matched = set(gp_res_history_final["gp_lgd_code"])
unmatched_gps_v3 = gp_lookup[
    ~gp_lookup["gp_lgd_code"].isin(already_matched)
][["gp_lgd_code","gp_name_lgd","district_lgd",
   "subdistrict_lgd","gp_norm","district_norm_hist"]].copy()
unmatched_gps_v3 = unmatched_gps_v3.rename(columns={
    "gp_name_lgd":"gp_name","district_lgd":"district",
    "subdistrict_lgd":"subdistrict_samiti"
})

print(f"\nUnmatched GPs for rerun: {len(unmatched_gps_v3)}")
print(f"In kekri→ajmer: {(unmatched_gps_v3['district_norm_hist']=='ajmer').sum()} "
      f"(includes kekri and beawar)")

# 2005 fuzzy match
print(f"\nRunning 2005 fuzzy match...")
results_2005_v3 = []
for _, gp_row in unmatched_gps_v3.iterrows():
    dist = gp_row["district_norm_hist"]
    gp_n = gp_row["gp_norm"]
    lgd  = gp_row["gp_lgd_code"]
    cands = sp2005_by_district[
        sp2005_by_district["district_norm_hist"]==dist
    ].copy()
    if cands.empty:
        results_2005_v3.append({
            "gp_lgd_code":lgd,"best_match_gp":None,
            "similarity":0.0,"reserved_women_2005":None,
            "n_rows":0,"n_categ_values":0,"match_found":False
        })
        continue
    cands["sim"] = cands["gp_norm"].apply(lambda x: similarity(gp_n,x))
    best = cands.nlargest(1,"sim").iloc[0]
    mf = best["sim"] >= 0.85
    results_2005_v3.append({
        "gp_lgd_code":        lgd,
        "best_match_gp":      best["gp_norm"],
        "similarity":         best["sim"],
        "reserved_women_2005":best["reserved_women_2005"] if mf else None,
        "n_rows":             best["n_rows"] if mf else 0,
        "n_categ_values":     best["n_categ_values"] if mf else 0,
        "match_found":        mf
    })

r2005_v3 = pd.DataFrame(results_2005_v3)
safe_2005_v3 = r2005_v3[r2005_v3["match_found"] & (r2005_v3["n_categ_values"]<=1)]
print(f"2005 safe matches: {len(safe_2005_v3)}")

# 2010 fuzzy match
print(f"Running 2010 fuzzy match...")
results_2010_v3 = []
for _, gp_row in unmatched_gps_v3.iterrows():
    dist = gp_row["district_norm_hist"]
    gp_n = gp_row["gp_norm"]
    lgd  = gp_row["gp_lgd_code"]
    cands = sp2010_by_district[
        sp2010_by_district["district_norm_hist"]==dist
    ].copy()
    if cands.empty:
        results_2010_v3.append({
            "gp_lgd_code":lgd,"best_match_gp":None,
            "similarity":0.0,"reserved_women_2010":None,
            "n_rows":0,"n_categ_values":0,"match_found":False
        })
        continue
    cands["sim"] = cands["gp_norm"].apply(lambda x: similarity(gp_n,x))
    best = cands.nlargest(1,"sim").iloc[0]
    mf = best["sim"] >= 0.85
    results_2010_v3.append({
        "gp_lgd_code":        lgd,
        "best_match_gp":      best["gp_norm"],
        "similarity":         best["sim"],
        "reserved_women_2010":best["reserved_women_2010"] if mf else None,
        "n_rows":             best["n_rows"] if mf else 0,
        "n_categ_values":     best["n_categ_values"] if mf else 0,
        "match_found":        mf
    })

r2010_v3 = pd.DataFrame(results_2010_v3)
safe_2010_v3 = r2010_v3[r2010_v3["match_found"] & (r2010_v3["n_categ_values"]<=1)]
print(f"2010 safe matches: {len(safe_2010_v3)}")

# Both years
safe_2005_v3_df = safe_2005_v3[["gp_lgd_code","reserved_women_2005","similarity"]].rename(
    columns={"similarity":"sim_2005"}
)
safe_2010_v3_df = safe_2010_v3[["gp_lgd_code","reserved_women_2010","similarity"]].rename(
    columns={"similarity":"sim_2010"}
)

combined_v3 = unmatched_gps_v3[["gp_lgd_code","gp_name","district","subdistrict_samiti"]].merge(
    safe_2005_v3_df, on="gp_lgd_code", how="left"
).merge(
    safe_2010_v3_df, on="gp_lgd_code", how="left"
)

both_v3 = combined_v3[
    combined_v3["reserved_women_2005"].notna() &
    combined_v3["reserved_women_2010"].notna()
].copy()
both_v3["reserved_women_2005"] = both_v3["reserved_women_2005"].astype(int)
both_v3["reserved_women_2010"] = both_v3["reserved_women_2010"].astype(int)
both_v3["reservation_dose_n"]  = both_v3["reserved_women_2005"] + both_v3["reserved_women_2010"]
both_v3["reservation_dose"]    = both_v3["reservation_dose_n"].map(DOSE_LABEL_MAP)

print(f"\nBoth-year matches (Stage F3): {len(both_v3)}")
print(f"\nDistrict breakdown of new matches:")
print(both_v3.merge(
    unmatched_gps_v3[["gp_lgd_code","district_norm_hist"]],
    on="gp_lgd_code"
)["district_norm_hist"].value_counts().head(10))

Updated DISTRICT_ALIASES: kekri→ajmer, beawar→ajmer
Total aliases: 23

Hanootiya (33758, Beawar→Ajmer) after alias fix:
  2005: hanutiya (sim=0.824, categ=sew, reserved=1)
  2010: hanutiya (sim=0.824, categ=GENW, reserved=1)

Unmatched GPs for rerun: 4715
In kekri→ajmer: 189 (includes kekri and beawar)

Running 2005 fuzzy match...
2005 safe matches: 835
Running 2010 fuzzy match...
2010 safe matches: 542

Both-year matches (Stage F3): 0

District breakdown of new matches:
Series([], Name: count, dtype: int64)


In [60]:
# Revert kekri and beawar aliases — didn't produce new matches
del DISTRICT_ALIASES["kekri"]
del DISTRICT_ALIASES["beawar"]

# Re-normalize back
for df_name, df in [("sp2005", sp2005), ("sp2010", sp2010),
                     ("res_clean", res_clean), ("gp_lookup", gp_lookup)]:
    if df_name == "gp_lookup":
        df["district_norm_hist"] = df["district_lgd"].apply(
            normalize_district_historical
        )
    else:
        df["district_norm_hist"] = df["district_norm"].apply(
            lambda x: DISTRICT_ALIASES.get(x, x)
        )

print(f"Reverted. Total aliases: {len(DISTRICT_ALIASES)}")
print(f"Current aliases: {DISTRICT_ALIASES}")

Reverted. Total aliases: 21
Current aliases: {'chittaurgarh': 'chittorgarh', 'dhaulpur': 'dholpur', 'jalor': 'jalore', 'jhunjhunun': 'jhunjhunu', 'anupgarh': 'ganganagar', 'balotra': 'barmer', 'deeg': 'bharatpur', 'didwanakuchaman': 'nagaur', 'dudu': 'jaipur', 'gangapurcity': 'sawaimadhopur', 'jaipurgramin': 'jaipur', 'jodhpurgramin': 'jodhpur', 'khairthaltijara': 'alwar', 'kotputlibehror': 'jaipur', 'neemkathana': 'sikar', 'phalodi': 'jodhpur', 'salumbar': 'udaipur', 'sanchore': 'jalore', 'shahpura': 'bhilwara', 'dungerpur': 'dungarpur', 'smadhopur': 'sawaimadhopur'}


In [61]:
# =============================================================================
# Export clean manual lookup workbook for top 31 single-blocker GPs
# =============================================================================

from openpyxl.styles import PatternFill, Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# Get top 31 unique single-blocker GPs
top_31_lookup = (
    single_blockers_named_df
    .drop_duplicates(subset=["gp_lgd_code"])
    .head(31)
    .copy()
)

# Add number of clusters each GP unlocks
gp_cluster_counts = (
    single_blockers_named_df
    .groupby("gp_lgd_code")["DHSCLUST"]
    .nunique()
    .reset_index()
    .rename(columns={"DHSCLUST":"n_clusters_unlocked"})
)
top_31_lookup = top_31_lookup.merge(gp_cluster_counts, on="gp_lgd_code", how="left")

# Add best fuzzy match from res_clean for reference
stage_g2_scores_lookup = pd.DataFrame(stage_g2_scores).rename(
    columns={"best_sim":"best_auto_sim","best_match":"best_auto_match",
             "dose":"best_auto_dose"}
)
top_31_lookup = top_31_lookup.merge(
    stage_g2_scores_lookup[["gp_lgd_code","best_auto_sim",
                             "best_auto_match","best_auto_dose"]],
    on="gp_lgd_code", how="left"
)

# Build export dataframe
export_manual = top_31_lookup[[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "p_known_treatment_mc","n_clusters_unlocked",
    "best_auto_sim","best_auto_match","best_auto_dose"
]].copy()

export_manual = export_manual.rename(columns={
    "gp_lgd_code":          "LGD Code",
    "gp_name":              "GP Name (LGD)",
    "district":             "District",
    "subdistrict_samiti":   "Panchayat Samiti",
    "p_known_treatment_mc": "Cluster p_known",
    "n_clusters_unlocked":  "Clusters Unlocked",
    "best_auto_sim":        "Best Auto Match Score",
    "best_auto_match":      "Best Auto Match Name",
    "best_auto_dose":       "Best Auto Match Dose"
})

# Add manual entry columns
export_manual["reserved_women_2005"] = ""
export_manual["reserved_women_2010"] = ""
export_manual["reservation_dose"]    = ""
export_manual["source_notes"]        = ""

# Add search instructions
export_manual["SEC Search URL"] = export_manual.apply(
    lambda r: f"https://sec.rajasthan.gov.in/grampanchayatdetails.aspx",
    axis=1
)

print(f"Manual lookup export: {len(export_manual)} GPs")
print(f"  → If all matched: {119 + export_manual['Clusters Unlocked'].sum()} fully linked")
print(f"\nPreview:")
print(export_manual[[
    "LGD Code","GP Name (LGD)","District","Panchayat Samiti",
    "Clusters Unlocked","Best Auto Match Score","Best Auto Match Dose"
]].to_string(index=False))

# Export to Excel with formatting
OUTPUT_PATH = OUTPUT_DIR / "manual_lookup_top31_single_blocker_gps.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    export_manual.to_excel(writer, index=False, sheet_name="Manual Lookup")
    ws = writer.sheets["Manual Lookup"]

    # Freeze header
    ws.freeze_panes = "A2"

    # Column widths
    col_widths = {
        "A": 12,  # LGD Code
        "B": 25,  # GP Name
        "C": 18,  # District
        "D": 20,  # Samiti
        "E": 14,  # p_known
        "F": 12,  # Clusters
        "G": 14,  # Auto sim
        "H": 22,  # Auto match name
        "I": 18,  # Auto match dose
        "J": 10,  # reserved 2005
        "K": 10,  # reserved 2010
        "L": 14,  # dose
        "M": 30,  # notes
        "N": 45,  # URL
    }
    for col, width in col_widths.items():
        ws.column_dimensions[col].width = width

    # Header formatting
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(bold=True, color="FFFFFF")
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", wrap_text=True)

    # Highlight manual entry columns (J, K, L, M) in yellow
    entry_fill = PatternFill("solid", fgColor="FFFF99")
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row,
                             min_col=10, max_col=13):
        for cell in row:
            cell.fill = entry_fill

    # Color-code rows by clusters unlocked
    green_fill  = PatternFill("solid", fgColor="C6EFCE")
    yellow_fill = PatternFill("solid", fgColor="FFEB9C")
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        n_unlock = row[5].value  # Clusters Unlocked column
        if n_unlock and int(n_unlock) >= 2:
            for cell in row[:9]:  # Only color non-entry columns
                cell.fill = green_fill
        else:
            for cell in row[:9]:
                cell.fill = yellow_fill

    # Add instructions sheet
    ws_info = writer.book.create_sheet("Instructions")
    instructions = [
        ["MANUAL LOOKUP INSTRUCTIONS"],
        [""],
        ["Goal: Find the 2005 and 2010 reservation category for each GP"],
        [""],
        ["Step 1: Go to https://sec.rajasthan.gov.in/grampanchayatdetails.aspx"],
        ["Step 2: Select District → Panchayat Samiti → Gram Panchayat"],
        ["Step 3: Note the reservation category for 2005 and 2010 elections"],
        ["Step 4: Enter in columns J and K:"],
        ["         0 = NOT reserved for women (GEN, OBC, SC, ST)"],
        ["         1 = Reserved for women (any category ending in W: GENW, OBCW, SCW, STW)"],
        ["Step 5: Column L (reservation_dose) will be:"],
        ["         never  = both 0,0"],
        ["         once   = one year reserved (0,1 or 1,0)"],
        ["         twice  = both reserved (1,1)"],
        [""],
        ["PRIORITY: Green rows unlock 2+ clusters each — do these first"],
        [""],
        ["NOTE: 'Best Auto Match' columns show the closest automated match found."],
        ["       If the auto match name looks correct, verify its dose on SEC website."],
        ["       If GP not found under its own district, try searching by GP name only."],
        [""],
        ["After filling, share the file and we will run the merge-back pipeline."],
    ]
    for i, row in enumerate(instructions, 1):
        ws_info.cell(row=i, column=1, value=row[0] if row else "")
    ws_info.column_dimensions["A"].width = 80
    ws_info["A1"].font = Font(bold=True, size=14)

print(f"\nSaved to: {OUTPUT_PATH}")

Manual lookup export: 31 GPs
  → If all matched: 157 fully linked

Preview:
LGD Code          GP Name (LGD)         District Panchayat Samiti  Clusters Unlocked  Best Auto Match Score Best Auto Match Dose
  295432        Badhli Nathusar        Jaisalmer          Pokaran                  2               0.571429                never
  294914             Sonthli(R)        Jhunjhunu        Nawalgarh                  1               0.769231                never
  295032             Kushalgarh      Chittorgarh       Rawatbhata                  1               0.631579                twice
   34833    Laxmipura Khankaraa            Baran       Kishanganj                  1               0.562500                never
   40260                 Kolana             Kota          Ladpura                  2               0.666667                 once
   42040                  Awada             Tonk          Malpura                  1               0.833333                 once
   38034 Chak Jawala 

In [62]:
# =============================================================================
# Thorough fuzzy search for all 31 single-blocker GPs
# against FULL 2005 and 2010 sarpanch files (no district filter)
# =============================================================================

print("Running thorough cross-district fuzzy search for all 31 GPs...\n")

thorough_results = []

for _, gp_row in top_31_lookup.iterrows():
    gp_n     = normalize_name(gp_row["gp_name"])
    lgd      = gp_row["gp_lgd_code"]
    own_dist = normalize_district_historical(gp_row["district"])

    # Search all of 2005
    sp2005["sim"] = sp2005["gp_norm"].apply(lambda x: similarity(gp_n, x))
    best_2005 = sp2005.nlargest(1, "sim").iloc[0]

    # Search all of 2010
    sp2010["sim"] = sp2010["gp_norm"].apply(lambda x: similarity(gp_n, x))
    best_2010 = sp2010.nlargest(1, "sim").iloc[0]

    thorough_results.append({
        "gp_lgd_code":       lgd,
        "gp_name":           gp_row["gp_name"],
        "district":          gp_row["district"],
        "subdistrict_samiti":gp_row["subdistrict_samiti"],
        "n_clusters":        gp_row["n_clusters_unlocked"],
        # 2005 best match
        "best_2005_gp":      best_2005["gp_norm"],
        "best_2005_dist":    best_2005["district_norm_hist"],
        "best_2005_samiti":  best_2005["samiti_norm"],
        "best_2005_sim":     best_2005["sim"],
        "best_2005_categ":   best_2005["CATEG. OF POST OF SARPANCH"],
        "best_2005_reserved":best_2005["reserved_women_2005"],
        "same_dist_2005":    best_2005["district_norm_hist"] == own_dist,
        # 2010 best match
        "best_2010_gp":      best_2010["gp_norm"],
        "best_2010_dist":    best_2010["district_norm_hist"],
        "best_2010_samiti":  best_2010["samiti_norm"],
        "best_2010_sim":     best_2010["sim"],
        "best_2010_categ":   best_2010["Ward category"],
        "best_2010_reserved":best_2010["reserved_women_2010"],
        "same_dist_2010":    best_2010["district_norm_hist"] == own_dist,
    })

thorough_df = pd.DataFrame(thorough_results)

print("Full results — best match in entire sarpanch file:")
print(thorough_df[[
    "gp_lgd_code","gp_name","district",
    "best_2005_gp","best_2005_dist","best_2005_sim","best_2005_categ",
    "best_2010_gp","best_2010_dist","best_2010_sim","best_2010_categ"
]].to_string(index=False))

# Flag GPs where best match is >= 0.80 in both years
both_good = thorough_df[
    (thorough_df["best_2005_sim"] >= 0.80) &
    (thorough_df["best_2010_sim"] >= 0.80)
].copy()

print(f"\nGPs with sim >= 0.80 in both years: {len(both_good)}")
print(both_good[[
    "gp_lgd_code","gp_name","district",
    "best_2005_gp","best_2005_dist","best_2005_sim","best_2005_categ",
    "best_2010_gp","best_2010_dist","best_2010_sim","best_2010_categ"
]].to_string(index=False))

# Flag cases where best match is in a DIFFERENT district in both years
# (potential boundary change)
cross_both = both_good[
    (~both_good["same_dist_2005"]) |
    (~both_good["same_dist_2010"])
].copy()
print(f"\nGPs where best match is cross-district in at least one year: {len(cross_both)}")
print(cross_both[[
    "gp_lgd_code","gp_name","district",
    "best_2005_gp","best_2005_dist","best_2005_sim",
    "best_2010_gp","best_2010_dist","best_2010_sim"
]].to_string(index=False))

Running thorough cross-district fuzzy search for all 31 GPs...

Full results — best match in entire sarpanch file:
gp_lgd_code                gp_name         district      best_2005_gp best_2005_dist  best_2005_sim best_2005_categ       best_2010_gp best_2010_dist  best_2010_sim best_2010_categ
     295432        Badhli Nathusar        Jaisalmer          nathusar          alwar       0.727273            ST W           nathusar          alwar       0.727273             OBC
     294914             Sonthli(R)        Jhunjhunu          sainthli          alwar       0.750000           OBC W              sohli      jhunjhunu       0.769231             GEN
     295032             Kushalgarh      Chittorgarh        kishangarh       bhilwara       0.800000             GEN         kishangarh       bhilwara       0.800000             STW
      34833    Laxmipura Khankaraa            Baran      saipurpakhar          alwar       0.666667              SC laxmipurakathawala         jaipur       0.777

In [63]:
# =============================================================================
# Verify the strongest cross-district consistent matches
# and accept safe ones
# =============================================================================

# GPs where best match is in SAME district both years AND sim >= 0.85
# These are the most reliable
consistent_same_dist = thorough_df[
    (thorough_df["best_2005_sim"] >= 0.85) &
    (thorough_df["best_2010_sim"] >= 0.85) &
    (thorough_df["best_2005_dist"] == thorough_df["best_2010_dist"])
].copy()

print("GPs with consistent match in same district both years (sim>=0.85):")
print(consistent_same_dist[[
    "gp_lgd_code","gp_name","district",
    "best_2005_gp","best_2005_dist","best_2005_sim","best_2005_categ",
    "best_2010_gp","best_2010_dist","best_2010_sim","best_2010_categ"
]].to_string(index=False))

# For each, check dose consistency
print("\nDose analysis:")
for _, row in consistent_same_dist.iterrows():
    r2005 = int(row["best_2005_reserved"])
    r2010 = int(row["best_2010_reserved"])
    dose_n = r2005 + r2010
    dose = DOSE_LABEL_MAP[dose_n]
    same_dist_2005 = row["same_dist_2005"]
    same_dist_2010 = row["same_dist_2010"]
    print(f"\n  {row['gp_name']} (LGD: {row['gp_lgd_code']}, own: {row['district']})")
    print(f"    2005: {row['best_2005_gp']} in {row['best_2005_dist']} "
          f"(sim={row['best_2005_sim']:.3f}, categ={row['best_2005_categ']}, "
          f"reserved={r2005}, own_dist={same_dist_2005})")
    print(f"    2010: {row['best_2010_gp']} in {row['best_2010_dist']} "
          f"(sim={row['best_2010_sim']:.3f}, categ={row['best_2010_categ']}, "
          f"reserved={r2010}, own_dist={same_dist_2010})")
    print(f"    → reservation_dose = {dose} ({r2005},{r2010})")
    if same_dist_2005 and same_dist_2010:
        print(f"    ✓ SAME DISTRICT — high confidence")
    elif row["best_2005_dist"] == row["best_2010_dist"]:
        print(f"    ~ CROSS-DISTRICT but consistent — possible boundary change")
    else:
        print(f"    ✗ INCONSISTENT DISTRICTS — skip")

GPs with consistent match in same district both years (sim>=0.85):
gp_lgd_code   gp_name  district best_2005_gp best_2005_dist  best_2005_sim best_2005_categ best_2010_gp best_2010_dist  best_2010_sim best_2010_categ
      40260    Kolana      Kota       kolana          dausa       1.000000              ST       kolana          dausa       1.000000             sew
      42040     Awada      Tonk       jawada    chittorgarh       0.909091           GEN W       jawada    chittorgarh       0.909091              SC
     294609 Bhojrasar     Churu     bhojasar    hanumangarh       0.941176             OBC     bhojasar    hanumangarh       0.941176            GENW
      40323 Madanpura      Kota    madanpura           kota       1.000000             GEN    madanpura           kota       1.000000              SC
      39419  Shyopura Jhunjhunu     shyopura          churu       1.000000             GEN     shyopura          churu       1.000000            GENW
     294500 Gopalpura     Kekri  

In [64]:
# =============================================================================
# Verify the strongest cross-district consistent matches
# and accept safe ones
# =============================================================================

# GPs where best match is in SAME district both years AND sim >= 0.85
# These are the most reliable
consistent_same_dist = thorough_df[
    (thorough_df["best_2005_sim"] >= 0.85) &
    (thorough_df["best_2010_sim"] >= 0.85) &
    (thorough_df["best_2005_dist"] == thorough_df["best_2010_dist"])
].copy()

print("GPs with consistent match in same district both years (sim>=0.85):")
print(consistent_same_dist[[
    "gp_lgd_code","gp_name","district",
    "best_2005_gp","best_2005_dist","best_2005_sim","best_2005_categ",
    "best_2010_gp","best_2010_dist","best_2010_sim","best_2010_categ"
]].to_string(index=False))

# For each, check dose consistency
print("\nDose analysis:")
for _, row in consistent_same_dist.iterrows():
    r2005 = int(row["best_2005_reserved"])
    r2010 = int(row["best_2010_reserved"])
    dose_n = r2005 + r2010
    dose = DOSE_LABEL_MAP[dose_n]
    same_dist_2005 = row["same_dist_2005"]
    same_dist_2010 = row["same_dist_2010"]
    print(f"\n  {row['gp_name']} (LGD: {row['gp_lgd_code']}, own: {row['district']})")
    print(f"    2005: {row['best_2005_gp']} in {row['best_2005_dist']} "
          f"(sim={row['best_2005_sim']:.3f}, categ={row['best_2005_categ']}, "
          f"reserved={r2005}, own_dist={same_dist_2005})")
    print(f"    2010: {row['best_2010_gp']} in {row['best_2010_dist']} "
          f"(sim={row['best_2010_sim']:.3f}, categ={row['best_2010_categ']}, "
          f"reserved={r2010}, own_dist={same_dist_2010})")
    print(f"    → reservation_dose = {dose} ({r2005},{r2010})")
    if same_dist_2005 and same_dist_2010:
        print(f"    ✓ SAME DISTRICT — high confidence")
    elif row["best_2005_dist"] == row["best_2010_dist"]:
        print(f"    ~ CROSS-DISTRICT but consistent — possible boundary change")
    else:
        print(f"    ✗ INCONSISTENT DISTRICTS — skip")

GPs with consistent match in same district both years (sim>=0.85):
gp_lgd_code   gp_name  district best_2005_gp best_2005_dist  best_2005_sim best_2005_categ best_2010_gp best_2010_dist  best_2010_sim best_2010_categ
      40260    Kolana      Kota       kolana          dausa       1.000000              ST       kolana          dausa       1.000000             sew
      42040     Awada      Tonk       jawada    chittorgarh       0.909091           GEN W       jawada    chittorgarh       0.909091              SC
     294609 Bhojrasar     Churu     bhojasar    hanumangarh       0.941176             OBC     bhojasar    hanumangarh       0.941176            GENW
      40323 Madanpura      Kota    madanpura           kota       1.000000             GEN    madanpura           kota       1.000000              SC
      39419  Shyopura Jhunjhunu     shyopura          churu       1.000000             GEN     shyopura          churu       1.000000            GENW
     294500 Gopalpura     Kekri  

In [65]:
# Recreate stage_j_df
stage_j_entries = []

def get_subdistrict(lgd):
    r = gp_lookup[gp_lookup["gp_lgd_code"]==lgd]
    return r["subdistrict_lgd"].iloc[0] if len(r)>0 else None

stage_j_entries.append({
    "gp_lgd_code": "40323", "gp_name_lgd": "Madanpura",
    "district_lgd": "Kota", "subdistrict_lgd": get_subdistrict("40323"),
    "reserved_women_2005": 0, "reserved_women_2010": 0,
    "reservation_dose_n": 0, "reservation_dose": "never",
    "match_stage": "J_thorough_cross_district_search",
})
stage_j_entries.append({
    "gp_lgd_code": "38034", "gp_name_lgd": "Chak Jawala Singh Wala",
    "district_lgd": "Hanumangarh", "subdistrict_lgd": get_subdistrict("38034"),
    "reserved_women_2005": 1, "reserved_women_2010": 1,
    "reservation_dose_n": 2, "reservation_dose": "twice",
    "match_stage": "J_thorough_cross_district_search",
})
stage_j_entries.append({
    "gp_lgd_code": "36361", "gp_name_lgd": "Jawti Kala",
    "district_lgd": "Bundi", "subdistrict_lgd": get_subdistrict("36361"),
    "reserved_women_2005": 0, "reserved_women_2010": 0,
    "reservation_dose_n": 0, "reservation_dose": "never",
    "match_stage": "J_thorough_cross_district_search",
})
stage_j_entries.append({
    "gp_lgd_code": "40555", "gp_name_lgd": "Meethri",
    "district_lgd": "Didwana-Kuchaman", "subdistrict_lgd": get_subdistrict("40555"),
    "reserved_women_2005": 0, "reserved_women_2010": 1,
    "reservation_dose_n": 1, "reservation_dose": "once",
    "match_stage": "J_thorough_cross_district_search",
})
stage_j_entries.append({
    "gp_lgd_code": "33758", "gp_name_lgd": "Hanootiya",
    "district_lgd": "Beawar", "subdistrict_lgd": get_subdistrict("33758"),
    "reserved_women_2005": 1, "reserved_women_2010": 1,
    "reservation_dose_n": 2, "reservation_dose": "twice",
    "match_stage": "J_thorough_cross_district_search",
})

stage_j_df = pd.DataFrame(stage_j_entries)

# Verify
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_j_df["gp_lgd_code"]) & already
print(f"Overlap (should be 0): {len(overlap)}")
print(f"Stage J entries: {len(stage_j_df)}")
print(stage_j_df[["gp_lgd_code","gp_name_lgd","district_lgd",
                   "reserved_women_2005","reserved_women_2010",
                   "reservation_dose"]].to_string(index=False))

Overlap (should be 0): 0
Stage J entries: 5
gp_lgd_code            gp_name_lgd     district_lgd  reserved_women_2005  reserved_women_2010 reservation_dose
      40323              Madanpura             Kota                    0                    0            never
      38034 Chak Jawala Singh Wala      Hanumangarh                    1                    1            twice
      36361             Jawti Kala            Bundi                    0                    0            never
      40555                Meethri Didwana-Kuchaman                    0                    1             once
      33758              Hanootiya           Beawar                    1                    1            twice


In [66]:
# =============================================================================
# Add Stage J and rerun pipeline
# =============================================================================

gp_res_history_final_v9 = pd.concat([
    gp_res_history_final,
    stage_j_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
        "reserved_women_2005","reserved_women_2010",
        "reservation_dose_n","reservation_dose","match_stage"
    ]]
], ignore_index=True)
assert gp_res_history_final_v9["gp_lgd_code"].duplicated().sum() == 0
print(f"GP history: {len(gp_res_history_final)} → {len(gp_res_history_final_v9)}")

cluster_gp_res_v9 = mc_long.merge(
    gp_res_history_final_v9[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

cluster_known_v9 = (
    cluster_gp_res_v9[cluster_gp_res_v9["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v9.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v9.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v9.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v9 = all_clusters.merge(cluster_known_v9, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v9[col] = treat_probs_v9[col].fillna(0.0)
treat_probs_v9["p_unknown_treatment_mc"] = (
    1 - treat_probs_v9["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v9["p_any_reserved_mc"] = (
    treat_probs_v9["p_once_mc"] + treat_probs_v9["p_twice_mc"]
)
treat_probs_v9["expected_dose_mc"] = (
    treat_probs_v9["p_once_mc"] + 2 * treat_probs_v9["p_twice_mc"]
)
treat_probs_v9["treatment_certainty_mc"] = treat_probs_v9[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_v9 = (
    cluster_gp_res_v9[cluster_gp_res_v9["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v9 = treat_probs_v9.merge(primary_dose_v9, on="DHSCLUST", how="left")
treat_probs_v9["primary_gp_norm"] = treat_probs_v9["primary_gp"].apply(normalize_lgd_code)
treat_probs_v9["primary_gp_has_history"] = treat_probs_v9["primary_gp_norm"].isin(
    set(gp_res_history_final_v9["gp_lgd_code"])
)

working_v9 = treat_probs_v9[treat_probs_v9["primary_gp_has_history"]].copy()

print(f"\nWorking clusters:       {len(working_v9)}")
print(f"Fully linked (>=99.9%): {(working_v9['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v9['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v9['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v9['p_known_treatment_mc'].mean():.3f}")

# Update working variables
gp_res_history_final = gp_res_history_final_v9
cluster_gp_res       = cluster_gp_res_v9
treat_probs_new      = treat_probs_v9
working_clusters_new = working_v9
final_treat_probs    = working_v9[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
print(f"\nSaved. Total GP histories: {len(gp_res_history_final)}")
print(f"\nMatch stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

GP history: 6484 → 6489

Working clusters:       731
Fully linked (>=99.9%): 124
>=90% linked:           275
>=75% linked:           471
Mean p_known:           0.808

Saved. Total GP histories: 6489

Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
F2_fuzzy_sarpanch_both_years_alias_fix    1733
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide                 185
D_district_only_unique_dose                177
E_fuzzy_gp_name_district                    70
G_fuzzy_transliteration_single_blocker      15
G2_fuzzy_single_blocker_alias_fix           15
H_manual_verified_transliteration            5
J_thorough_cross_district_search             5
Name: count, dtype: int64


In [67]:
# =============================================================================
# Update manual lookup workbook — remove Stage J matches, update projections
# =============================================================================

# GPs already matched in Stage J
stage_j_lgd_codes = set(stage_j_df["gp_lgd_code"].astype(str))

# Recompute single-blocker list with updated pipeline
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)
analysis_clusters = set(final_treat_probs["DHSCLUST"]) - isolated_clusters

near_complete = final_treat_probs[
    (final_treat_probs["p_known_treatment_mc"] >= 0.90) &
    (final_treat_probs["p_known_treatment_mc"] < 0.999)
]

blocking = []
for _, cluster in near_complete.iterrows():
    unmatched_rows = cluster_gp_res[
        (cluster_gp_res["DHSCLUST"] == cluster["DHSCLUST"]) &
        (cluster_gp_res["reservation_dose"].isna()) &
        (cluster_gp_res["gp_lgd_code"].notna())
    ]
    for _, gp_row in unmatched_rows.iterrows():
        blocking.append({
            "DHSCLUST":             cluster["DHSCLUST"],
            "p_known_treatment_mc": cluster["p_known_treatment_mc"],
            "n_unmatched_gps":      len(unmatched_rows),
            "gp_lgd_code":          gp_row["gp_lgd_code"],
            "gp_prob":              gp_row["gp_prob"],
        })

blocking_df = pd.DataFrame(blocking)
blocking_df = blocking_df.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd"]],
    on="gp_lgd_code", how="left"
)

n_per_cluster = blocking_df.groupby("DHSCLUST")["gp_lgd_code"].nunique()
single_blocker_clusters = n_per_cluster[n_per_cluster == 1].index
single_blockers_updated = blocking_df[
    blocking_df["DHSCLUST"].isin(single_blocker_clusters)
].drop_duplicates(subset=["gp_lgd_code"]).copy()

# Cluster count per GP
gp_cluster_counts_updated = (
    blocking_df[blocking_df["DHSCLUST"].isin(single_blocker_clusters)]
    .groupby("gp_lgd_code")["DHSCLUST"].nunique()
    .reset_index().rename(columns={"DHSCLUST":"n_clusters_unlocked"})
)

single_blockers_updated = single_blockers_updated.merge(
    gp_cluster_counts_updated, on="gp_lgd_code", how="left"
).sort_values("p_known_treatment_mc", ascending=False)

# Get top 31 unique
top_31_updated = single_blockers_updated.drop_duplicates(
    subset=["gp_lgd_code"]
).head(31)

print(f"Updated single-blocker GPs: {len(single_blockers_updated)}")
print(f"Top 31 unique:")
print(top_31_updated[[
    "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
    "n_clusters_unlocked","p_known_treatment_mc"
]].to_string(index=False))

n_if_all_matched = 124 + top_31_updated["n_clusters_unlocked"].sum()
print(f"\nCurrently fully linked: 124")
print(f"If all top 31 matched:  {n_if_all_matched}")

# Save updated manual lookup
from openpyxl.styles import PatternFill, Font, Alignment

export_manual_v2 = top_31_updated[[
    "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
    "p_known_treatment_mc","n_clusters_unlocked"
]].copy().rename(columns={
    "gp_lgd_code":          "LGD Code",
    "gp_name_lgd":          "GP Name (LGD)",
    "district_lgd":         "District",
    "subdistrict_lgd":      "Panchayat Samiti",
    "p_known_treatment_mc": "Cluster p_known",
    "n_clusters_unlocked":  "Clusters Unlocked"
})
export_manual_v2["reserved_women_2005"] = ""
export_manual_v2["reserved_women_2010"] = ""
export_manual_v2["reservation_dose"]    = ""
export_manual_v2["notes"]               = ""

OUTPUT_PATH_V2 = OUTPUT_DIR / "manual_lookup_top31_updated.xlsx"
with pd.ExcelWriter(OUTPUT_PATH_V2, engine="openpyxl") as writer:
    export_manual_v2.to_excel(writer, index=False, sheet_name="Manual Lookup")
    ws = writer.sheets["Manual Lookup"]
    ws.freeze_panes = "A2"
    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 40)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(bold=True, color="FFFFFF")
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
    entry_fill = PatternFill("solid", fgColor="FFFF99")
    green_fill  = PatternFill("solid", fgColor="C6EFCE")
    yellow_fill = PatternFill("solid", fgColor="FFEB9C")
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        n_unlock = row[5].value
        base_fill = green_fill if (n_unlock and int(n_unlock) >= 2) else yellow_fill
        for i, cell in enumerate(row):
            if i >= 6:  # entry columns
                cell.fill = entry_fill
            else:
                cell.fill = base_fill

print(f"\nSaved: {OUTPUT_PATH_V2}")

# Final outputs
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
print(f"All outputs saved to: {OUTPUT_DIR}")

Updated single-blocker GPs: 44
Top 31 unique:
gp_lgd_code         gp_name_lgd     district_lgd subdistrict_lgd  n_clusters_unlocked  p_known_treatment_mc
     294914          Sonthli(R)        Jhunjhunu       Nawalgarh                    1              0.997696
     295032          Kushalgarh      Chittorgarh      Rawatbhata                    1              0.997110
      34833 Laxmipura Khankaraa            Baran      Kishanganj                    1              0.997000
      40260              Kolana             Kota         Ladpura                    2              0.996917
      42040               Awada             Tonk         Malpura                    1              0.995227
     295432     Badhli Nathusar        Jaisalmer         Pokaran                    2              0.994312
     294609           Bhojrasar            Churu    Sardarshahar                    1              0.994000
     294332              Dhatri            Churu       Sujangarh                    1     

In [68]:
# =============================================================================
# Lower fuzzy threshold sweep — within district matches only
# Find all unmatched GPs with similarity 0.70-0.85 against res_clean
# =============================================================================

already_matched = set(gp_res_history_final["gp_lgd_code"])

# All unmatched GPs still in the MC sample
unmatched_in_sample = gp_lookup[
    gp_lookup["gp_lgd_code"].isin(
        set(mc_long["gp_lgd_code"].dropna())
    ) &
    ~gp_lookup["gp_lgd_code"].isin(already_matched)
].copy()

print(f"Unmatched GPs in MC sample: {len(unmatched_in_sample)}")

# Rebuild res_by_district with current aliases
res_by_district_final = (
    res_clean
    .groupby(["district_norm_hist","gp_norm"])
    .agg(
        n_dose_values       = ("reservation_dose",    "nunique"),
        reservation_dose    = ("reservation_dose",    "first"),
        reserved_women_2005 = ("reserved_women_2005", "first"),
        reserved_women_2010 = ("reserved_women_2010", "first"),
        reservation_dose_n  = ("reservation_dose_n",  "first")
    )
    .reset_index()
)

# Fuzzy match at lower thresholds — within district only
low_thresh_results = []

for _, gp_row in unmatched_in_sample.iterrows():
    dist   = gp_row["district_norm_hist"]
    gp_n   = gp_row["gp_norm"]
    lgd    = gp_row["gp_lgd_code"]

    cands = res_by_district_final[
        res_by_district_final["district_norm_hist"] == dist
    ].copy()

    if cands.empty:
        continue

    cands["sim"] = cands["gp_norm"].apply(lambda x: similarity(gp_n, x))
    best = cands.nlargest(1, "sim").iloc[0]

    # Only collect matches in 0.70-0.85 range (above already accepted)
    if 0.70 <= best["sim"] < 0.85:
        low_thresh_results.append({
            "gp_lgd_code":        lgd,
            "gp_name":            gp_row["gp_name_lgd"],
            "district":           gp_row["district_lgd"],
            "subdistrict_samiti": gp_row["subdistrict_lgd"],
            "best_sim":           best["sim"],
            "best_match":         best["gp_norm"],
            "dose":               best["reservation_dose"],
            "n_dose_values":      best["n_dose_values"],
            "reserved_2005":      best["reserved_women_2005"],
            "reserved_2010":      best["reserved_women_2010"],
        })

low_thresh_df = pd.DataFrame(low_thresh_results).sort_values(
    "best_sim", ascending=False
).reset_index(drop=True)

print(f"\nMatches in 0.70-0.85 range: {len(low_thresh_df)}")
print(f"With unique dose:            "
      f"{(low_thresh_df['n_dose_values']==1).sum()}")

# Focus on unique dose matches — these are the actionable ones
unique_dose = low_thresh_df[low_thresh_df["n_dose_values"]==1].copy()

print(f"\nUnique dose matches by similarity band:")
for lo, hi in [(0.80, 0.85), (0.75, 0.80), (0.70, 0.75)]:
    n = ((unique_dose["best_sim"] >= lo) & 
         (unique_dose["best_sim"] < hi)).sum()
    print(f"  {lo}-{hi}: {n} matches")

print(f"\nAll unique-dose matches (0.70-0.85), sorted by similarity:")
print(unique_dose[[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "best_sim","best_match","dose","reserved_2005","reserved_2010"
]].to_string(index=False))

Unmatched GPs in MC sample: 2488

Matches in 0.70-0.85 range: 1292
With unique dose:            1192

Unique dose matches by similarity band:
  0.8-0.85: 473 matches
  0.75-0.8: 352 matches
  0.7-0.75: 367 matches

All unique-dose matches (0.70-0.85), sorted by similarity:
gp_lgd_code               gp_name         district subdistrict_samiti  best_sim         best_match  dose  reserved_2005  reserved_2010
     294790           Narayanpura  Jaipur (Gramin)       Madhorajpura  0.846154    harinarayanpura  once              1              0
      37775        Shivpur Fatuhi       Ganganagar         Ganganagar  0.846154      shivpurphatui never              0              0
      35853      Motoron Ka Khera         Bhilwara         Mandalgarh  0.846154       motrokakheda  once              0              1
      41579        Jheegar Chhoti            Sikar       Sikar Gramin  0.846154      jhingarchhoti never              0              0
      42531        Masaro Ki Ovri          Udaipur 

In [69]:
# =============================================================================
# Filter low-threshold matches to only single-blocker GPs
# These are the ones that actually push clusters to fully linked
# =============================================================================

# Current single-blocker GPs (blocking 90-99% clusters)
current_single_blockers = set(
    single_blockers_updated["gp_lgd_code"].astype(str)
)
print(f"Current single-blocker GPs: {len(current_single_blockers)}")

# Filter low-threshold matches to single-blockers only
low_thresh_single = low_thresh_df[
    low_thresh_df["gp_lgd_code"].astype(str).isin(current_single_blockers) &
    (low_thresh_df["n_dose_values"] == 1)
].copy()

print(f"Single-blocker GPs with low-threshold match: {len(low_thresh_single)}")
print(f"\nFull list for manual review:")
print(low_thresh_single[[
    "gp_lgd_code","gp_name","district","subdistrict_samiti",
    "best_sim","best_match","dose","reserved_2005","reserved_2010"
]].sort_values("best_sim", ascending=False).to_string(index=False))

Current single-blocker GPs: 44
Single-blocker GPs with low-threshold match: 17

Full list for manual review:
gp_lgd_code        gp_name    district subdistrict_samiti  best_sim    best_match  dose  reserved_2005  reserved_2010
      36253      Vijaigarh       Bundi            Hindoli  0.842105    vijayagarh  once              0              1
      42040          Awada        Tonk            Malpura  0.833333       banwada  once              1              0
      39419       Shyopura   Jhunjhunu            Chirawa  0.777778    kishorpura  once              0              1
      41788        Gungara       Sikar              Sikar  0.769231        gurara never              0              0
     294264         Morani   Jaisalmer            Pokaran  0.769231       modradi twice              1              1
     294914     Sonthli(R)   Jhunjhunu          Nawalgarh  0.769231         sohli never              0              0
      35954 Phooliya Khurd    Shahpura     Phooliya Kalan  0.7692

In [70]:
# Verify uncertain candidates against 2005 and 2010 sarpanch files
candidates_to_verify = [
    ("36253",  "Vijaigarh",    "bundi",      "vijayagarh"),
    ("35815",  "Pithas",       "bhilwara",   "peethas"),
    ("36617",  "Doongla",      "chittorgarh","dungala"),
    ("36362",  "Jhak Mund",    "bundi",       "jamkhmund"),
    ("41788",  "Gungara",      "sikar",       "gurara"),
    ("295030", "Kazi",         "jhunjhunu",   "kari"),
    ("42186",  "Palara",       "tonk",        "palai"),
    ("294867", "Ghana",        "sikar",       "ganoda"),
    ("294264", "Morani",       "jaisalmer",   "modradi"),
    ("38370",  "Mahalaan",     "jaipur",      "mathasula"),
]

print("Verifying candidates against 2005 and 2010 sarpanch files:\n")
for lgd, gp_name, dist, best_match in candidates_to_verify:
    dist_norm = normalize_district_historical(dist)
    gp_n = normalize_name(gp_name)

    # 2005
    m2005 = sp2005[sp2005["district_norm_hist"]==dist_norm].copy()
    m2005["sim"] = m2005["gp_norm"].apply(lambda x: similarity(gp_n, x))
    b2005 = m2005.nlargest(1,"sim").iloc[0]

    # 2010
    m2010 = sp2010[sp2010["district_norm_hist"]==dist_norm].copy()
    m2010["sim"] = m2010["gp_norm"].apply(lambda x: similarity(gp_n, x))
    b2010 = m2010.nlargest(1,"sim").iloc[0]

    print(f"{gp_name} (LGD: {lgd}, {dist})")
    print(f"  2005: {b2005['gp_norm']:<22} sim={b2005['sim']:.3f}  "
          f"categ={b2005['CATEG. OF POST OF SARPANCH']}  "
          f"reserved={b2005['reserved_women_2005']}")
    print(f"  2010: {b2010['gp_norm']:<22} sim={b2010['sim']:.3f}  "
          f"categ={b2010['Ward category']}  "
          f"reserved={b2010['reserved_women_2010']}")
    print()

Verifying candidates against 2005 and 2010 sarpanch files:

Vijaigarh (LGD: 36253, bundi)
  2005: vijayagarh             sim=0.842  categ=SC  reserved=0
  2010: vijayagarh             sim=0.842  categ=GENW  reserved=1

Pithas (LGD: 35815, bhilwara)
  2005: pithas                 sim=1.000  categ=SC  reserved=0
  2010: peethas                sim=0.769  categ=GEN  reserved=0

Doongla (LGD: 36617, chittorgarh)
  2005: dungla                 sim=0.769  categ=SC  reserved=0
  2010: dungala                sim=0.714  categ=GENW  reserved=1

Jhak Mund (LGD: 36362, bundi)
  2005: jhakhmund              sim=0.941  categ=ST  reserved=0
  2010: jamkhmund              sim=0.706  categ=GEN  reserved=0

Gungara (LGD: 41788, sikar)
  2005: gungara                sim=1.000  categ=GEN  reserved=0
  2010: gurara                 sim=0.769  categ=SC  reserved=0

Kazi (LGD: 295030, jhunjhunu)
  2005: kari                   sim=0.750  categ=GEN  reserved=0
  2010: kari                   sim=0.750  categ=OBCW

In [71]:
# Revise Stage K to 5 confirmed matches only
stage_k_entries = []

# 1. Vijaigarh — once (0,1)
stage_k_entries.append({
    "gp_lgd_code": "36253", "gp_name_lgd": "Vijaigarh",
    "district_lgd": "Bundi", "subdistrict_lgd": get_subdistrict("36253"),
    "reserved_women_2005": 0, "reserved_women_2010": 1,
    "reservation_dose_n": 1, "reservation_dose": "once",
    "match_stage": "K_low_threshold_single_blocker",
})

# 2. Pithas — never (0,0)
stage_k_entries.append({
    "gp_lgd_code": "35815", "gp_name_lgd": "Pithas",
    "district_lgd": "Bhilwara", "subdistrict_lgd": get_subdistrict("35815"),
    "reserved_women_2005": 0, "reserved_women_2010": 0,
    "reservation_dose_n": 0, "reservation_dose": "never",
    "match_stage": "K_low_threshold_single_blocker",
})

# 3. Jhak Mund — never (0,0)
stage_k_entries.append({
    "gp_lgd_code": "36362", "gp_name_lgd": "Jhak Mund",
    "district_lgd": "Bundi", "subdistrict_lgd": get_subdistrict("36362"),
    "reserved_women_2005": 0, "reserved_women_2010": 0,
    "reservation_dose_n": 0, "reservation_dose": "never",
    "match_stage": "K_low_threshold_single_blocker",
})

# 4. Gungara — never (0,0)
stage_k_entries.append({
    "gp_lgd_code": "41788", "gp_name_lgd": "Gungara",
    "district_lgd": "Sikar", "subdistrict_lgd": get_subdistrict("41788"),
    "reserved_women_2005": 0, "reserved_women_2010": 0,
    "reservation_dose_n": 0, "reservation_dose": "never",
    "match_stage": "K_low_threshold_single_blocker",
})

# 5. Doongla — once (0,1)
stage_k_entries.append({
    "gp_lgd_code": "36617", "gp_name_lgd": "Doongla",
    "district_lgd": "Chittorgarh", "subdistrict_lgd": get_subdistrict("36617"),
    "reserved_women_2005": 0, "reserved_women_2010": 1,
    "reservation_dose_n": 1, "reservation_dose": "once",
    "match_stage": "K_low_threshold_single_blocker",
})

stage_k_df = pd.DataFrame(stage_k_entries)

# Verify no overlap
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_k_df["gp_lgd_code"]) & already
print(f"Overlap (should be 0): {len(overlap)}")
print(f"\nStage K entries: {len(stage_k_df)}")
print(stage_k_df[[
    "gp_lgd_code","gp_name_lgd","district_lgd",
    "reserved_women_2005","reserved_women_2010","reservation_dose"
]].to_string(index=False))

# Add to history and rerun
gp_res_history_final_v10 = pd.concat([
    gp_res_history_final,
    stage_k_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
        "reserved_women_2005","reserved_women_2010",
        "reservation_dose_n","reservation_dose","match_stage"
    ]]
], ignore_index=True)
assert gp_res_history_final_v10["gp_lgd_code"].duplicated().sum() == 0
print(f"\nGP history: {len(gp_res_history_final)} → {len(gp_res_history_final_v10)}")

# Rebuild cluster_gp_res
cluster_gp_res_v10 = mc_long.merge(
    gp_res_history_final_v10[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

# Recompute treat_probs
cluster_known_v10 = (
    cluster_gp_res_v10[cluster_gp_res_v10["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v10.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v10.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v10.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v10 = all_clusters.merge(cluster_known_v10, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v10[col] = treat_probs_v10[col].fillna(0.0)
treat_probs_v10["p_unknown_treatment_mc"] = (
    1 - treat_probs_v10["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v10["p_any_reserved_mc"] = (
    treat_probs_v10["p_once_mc"] + treat_probs_v10["p_twice_mc"]
)
treat_probs_v10["expected_dose_mc"] = (
    treat_probs_v10["p_once_mc"] + 2 * treat_probs_v10["p_twice_mc"]
)
treat_probs_v10["treatment_certainty_mc"] = treat_probs_v10[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_v10 = (
    cluster_gp_res_v10[cluster_gp_res_v10["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v10 = treat_probs_v10.merge(primary_dose_v10, on="DHSCLUST", how="left")
treat_probs_v10["primary_gp_norm"] = treat_probs_v10["primary_gp"].apply(normalize_lgd_code)
treat_probs_v10["primary_gp_has_history"] = treat_probs_v10["primary_gp_norm"].isin(
    set(gp_res_history_final_v10["gp_lgd_code"])
)

working_v10 = treat_probs_v10[treat_probs_v10["primary_gp_has_history"]].copy()

print(f"\nWorking clusters:       {len(working_v10)}")
print(f"Fully linked (>=99.9%): {(working_v10['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=90% linked:           {(working_v10['p_known_treatment_mc'] >= 0.90).sum()}")
print(f">=75% linked:           {(working_v10['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v10['p_known_treatment_mc'].mean():.3f}")

# Update working variables
gp_res_history_final = gp_res_history_final_v10
cluster_gp_res       = cluster_gp_res_v10
treat_probs_new      = treat_probs_v10
working_clusters_new = working_v10
final_treat_probs    = working_v10[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

# Save
gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)
print(f"\nSaved. Total GP histories: {len(gp_res_history_final)}")
print(f"\nMatch stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

Overlap (should be 0): 0

Stage K entries: 5
gp_lgd_code gp_name_lgd district_lgd  reserved_women_2005  reserved_women_2010 reservation_dose
      36253   Vijaigarh        Bundi                    0                    1             once
      35815      Pithas     Bhilwara                    0                    0            never
      36362   Jhak Mund        Bundi                    0                    0            never
      41788     Gungara        Sikar                    0                    0            never
      36617     Doongla  Chittorgarh                    0                    1             once

GP history: 6489 → 6494

Working clusters:       732
Fully linked (>=99.9%): 131
>=90% linked:           276
>=75% linked:           474
Mean p_known:           0.809

Saved. Total GP histories: 6494

Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
F2_fuzzy_sarpanch_both_years_alias_fix    1733
B_exact_gp_district                       1514
F

In [72]:
# =============================================================================
# Identify single-blocker GPs with one election year matched but not the other
# =============================================================================

# Current single-blocker GPs
isolated_clusters = set(
    cluster_gp_res[
        cluster_gp_res["DHSCLUST"].isin(set(final_treat_probs["DHSCLUST"])) &
        cluster_gp_res["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST")
    .filter(lambda x: len(x) == 1)
    ["DHSCLUST"].unique()
)

near_complete = final_treat_probs[
    (final_treat_probs["p_known_treatment_mc"] >= 0.90) &
    (final_treat_probs["p_known_treatment_mc"] < 0.999)
]

blocking = []
for _, cluster in near_complete.iterrows():
    unmatched_rows = cluster_gp_res[
        (cluster_gp_res["DHSCLUST"] == cluster["DHSCLUST"]) &
        (cluster_gp_res["reservation_dose"].isna()) &
        (cluster_gp_res["gp_lgd_code"].notna())
    ]
    n_unmatched = len(unmatched_rows)
    for _, gp_row in unmatched_rows.iterrows():
        blocking.append({
            "DHSCLUST":             cluster["DHSCLUST"],
            "p_known_treatment_mc": cluster["p_known_treatment_mc"],
            "n_unmatched_gps":      n_unmatched,
            "gp_lgd_code":          gp_row["gp_lgd_code"],
            "gp_prob":              gp_row["gp_prob"],
        })

blocking_df = pd.DataFrame(blocking)
blocking_df = blocking_df.merge(
    gp_lookup[["gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
               "gp_norm","district_norm_hist"]],
    on="gp_lgd_code", how="left"
)

n_per_cluster = blocking_df.groupby("DHSCLUST")["gp_lgd_code"].nunique()
single_blocker_clusters = n_per_cluster[n_per_cluster == 1].index
single_blockers_final = blocking_df[
    blocking_df["DHSCLUST"].isin(single_blocker_clusters)
].drop_duplicates(subset=["gp_lgd_code"]).copy()

print(f"Current single-blocker GPs: {len(single_blockers_final)}")

# For each single-blocker GP, check if it appears in EITHER sarpanch file
# (i.e. one year matched, one missing)
partial_match_results = []

for _, gp_row in single_blockers_final.iterrows():
    gp_n   = gp_row["gp_norm"]
    dist   = gp_row["district_norm_hist"]
    lgd    = gp_row["gp_lgd_code"]

    # Check 2005
    m2005 = sp2005[sp2005["district_norm_hist"]==dist].copy()
    if len(m2005) > 0:
        m2005["sim"] = m2005["gp_norm"].apply(lambda x: similarity(gp_n, x))
        b2005 = m2005.nlargest(1,"sim").iloc[0]
        sim_2005    = b2005["sim"]
        match_2005  = b2005["gp_norm"]
        categ_2005  = b2005["CATEG. OF POST OF SARPANCH"]
        res_2005    = b2005["reserved_women_2005"]
    else:
        sim_2005 = 0; match_2005 = None; categ_2005 = None; res_2005 = None

    # Check 2010
    m2010 = sp2010[sp2010["district_norm_hist"]==dist].copy()
    if len(m2010) > 0:
        m2010["sim"] = m2010["gp_norm"].apply(lambda x: similarity(gp_n, x))
        b2010 = m2010.nlargest(1,"sim").iloc[0]
        sim_2010    = b2010["sim"]
        match_2010  = b2010["gp_norm"]
        categ_2010  = b2010["Ward category"]
        res_2010    = b2010["reserved_women_2010"]
    else:
        sim_2010 = 0; match_2010 = None; categ_2010 = None; res_2010 = None

    # Classify
    has_2005 = sim_2005 >= 0.85
    has_2010 = sim_2010 >= 0.85

    partial_match_results.append({
        "gp_lgd_code":   lgd,
        "gp_name":       gp_row["gp_name_lgd"],
        "district":      gp_row["district_lgd"],
        "subdistrict":   gp_row["subdistrict_lgd"],
        "p_known":       gp_row["p_known_treatment_mc"],
        "gp_prob":       gp_row["gp_prob"],
        "sim_2005":      sim_2005,
        "match_2005":    match_2005,
        "categ_2005":    categ_2005,
        "res_2005":      res_2005,
        "sim_2010":      sim_2010,
        "match_2010":    match_2010,
        "categ_2010":    categ_2010,
        "res_2010":      res_2010,
        "has_2005":      has_2005,
        "has_2010":      has_2010,
        "status":        (
            "both_matched"    if has_2005 and has_2010 else
            "only_2005"       if has_2005 and not has_2010 else
            "only_2010"       if not has_2005 and has_2010 else
            "neither_matched"
        )
    })

partial_df = pd.DataFrame(partial_match_results).sort_values(
    "p_known", ascending=False
)

print(f"\nMatch status breakdown:")
print(partial_df["status"].value_counts())

print(f"\nGPs with only 2005 matched (missing 2010):")
only_2005 = partial_df[partial_df["status"]=="only_2005"]
print(only_2005[[
    "gp_lgd_code","gp_name","district","subdistrict",
    "sim_2005","match_2005","categ_2005",
    "sim_2010","match_2010","p_known"
]].to_string(index=False))

print(f"\nGPs with only 2010 matched (missing 2005):")
only_2010 = partial_df[partial_df["status"]=="only_2010"]
print(only_2010[[
    "gp_lgd_code","gp_name","district","subdistrict",
    "sim_2010","match_2010","categ_2010",
    "sim_2005","match_2005","p_known"
]].to_string(index=False))

Current single-blocker GPs: 39

Match status breakdown:
status
neither_matched    32
only_2005           7
Name: count, dtype: int64

GPs with only 2005 matched (missing 2010):
gp_lgd_code        gp_name    district    subdistrict  sim_2005    match_2005 categ_2005  sim_2010    match_2010  p_known
      40277           Doti        Kota         Kanwas  0.888889         dhoti        GEN  0.666667         tohti 0.992754
      35526        Marauli   Bharatpur        Rudawal  0.857143       barauli        OBC  0.666667         barai 0.991098
      39419       Shyopura   Jhunjhunu        Chirawa  1.000000      shyopura        GEN  0.777778    kishorpura 0.986216
      39412   Gothra Lamba   Jhunjhunu        Chirawa  0.869565  gothadalamba         ST  0.705882        gothra 0.975699
      35891 Pitha Ka Khera    Bhilwara         Raipur  0.880000 peethakakhera        sew  0.695652   gegakakhera 0.969377
      35954 Phooliya Khurd    Shahpura Phooliya Kalan  0.880000  phuliyakhurd         SC  0

In [73]:
# Check 2010 matches for the 7 "only_2005" GPs at a lower threshold
print("Detailed 2010 search for 7 only-2005 GPs:\n")

for _, row in only_2005.iterrows():
    gp_n = normalize_name(row["gp_name"])
    dist = normalize_district_historical(row["district"])

    m2010 = sp2010[sp2010["district_norm_hist"]==dist].copy()
    if len(m2010) == 0:
        print(f"{row['gp_name']} ({row['district']}): NO 2010 RECORDS IN DISTRICT")
        continue

    m2010["sim"] = m2010["gp_norm"].apply(lambda x: similarity(gp_n, x))
    top3 = m2010.nlargest(3,"sim")

    print(f"{row['gp_name']} (LGD: {row['gp_lgd_code']}, {row['district']}/{row['subdistrict']})")
    print(f"  2005 match: {row['match_2005']} (sim={row['sim_2005']:.3f}, "
          f"categ={row['categ_2005']}, reserved={int(row['res_2005'])})")
    print(f"  2010 top 3 matches:")
    for _, m in top3.iterrows():
        print(f"    {m['gp_norm']:<25} sim={m['sim']:.3f}  "
              f"categ={m['Ward category']}  "
              f"reserved={m['reserved_women_2010']}")
    print()

Detailed 2010 search for 7 only-2005 GPs:

Doti (LGD: 40277, Kota/Kanwas)
  2005 match: dhoti (sim=0.889, categ=GEN, reserved=0)
  2010 top 3 matches:
    tohti                     sim=0.667  categ=OBCW  reserved=1
    dolya                     sim=0.444  categ=SC  reserved=0
    danta                     sim=0.444  categ=OBC  reserved=0

Marauli (LGD: 35526, Bharatpur/Rudawal)
  2005 match: barauli (sim=0.857, categ=OBC, reserved=0)
  2010 top 3 matches:
    barai                     sim=0.667  categ=GENW  reserved=1
    mawal                     sim=0.667  categ=OBCW  reserved=1
    pathrali                  sim=0.667  categ=OBC  reserved=0

Shyopura (LGD: 39419, Jhunjhunu/Chirawa)
  2005 match: shyopura (sim=1.000, categ=GEN, reserved=0)
  2010 top 3 matches:
    kishorpura                sim=0.778  categ=GENW  reserved=1
    shahpura                  sim=0.750  categ=OBCW  reserved=1
    kishorepura               sim=0.737  categ=GENW  reserved=1

Gothra Lamba (LGD: 39412, Jhunjhun

In [74]:
# =============================================================================
# Stage L — accept Phooliya Khurd (one-year partial match, both never)
# =============================================================================

stage_l_df = pd.DataFrame([{
    "gp_lgd_code":        "35954",
    "gp_name_lgd":        "Phooliya Khurd",
    "district_lgd":       "Shahpura",
    "subdistrict_lgd":    get_subdistrict("35954"),
    "reserved_women_2005": 0,
    "reserved_women_2010": 0,
    "reservation_dose_n":  0,
    "reservation_dose":    "never",
    "match_stage":        "L_partial_match_both_never",
}])

# Verify
already = set(gp_res_history_final["gp_lgd_code"])
overlap = set(stage_l_df["gp_lgd_code"]) & already
print(f"Overlap (should be 0): {len(overlap)}")

gp_res_history_final_v11 = pd.concat([
    gp_res_history_final, stage_l_df[[
        "gp_lgd_code","gp_name_lgd","district_lgd","subdistrict_lgd",
        "reserved_women_2005","reserved_women_2010",
        "reservation_dose_n","reservation_dose","match_stage"
    ]]
], ignore_index=True)
assert gp_res_history_final_v11["gp_lgd_code"].duplicated().sum() == 0

# Rebuild
cluster_gp_res_v11 = mc_long.merge(
    gp_res_history_final_v11[[
        "gp_lgd_code","reservation_dose","reservation_dose_n",
        "reserved_women_2005","reserved_women_2010",
        "gp_name_lgd","district_lgd","subdistrict_lgd","match_stage"
    ]],
    on="gp_lgd_code", how="left"
)

cluster_known_v11 = (
    cluster_gp_res_v11[cluster_gp_res_v11["reservation_dose"].notna()]
    .groupby("DHSCLUST", as_index=False)
    .agg(
        p_known_treatment_mc = ("gp_prob", "sum"),
        p_never_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v11.loc[x.index,"reservation_dose"]=="never"].sum()),
        p_once_mc            = ("gp_prob", lambda x:
            x[cluster_gp_res_v11.loc[x.index,"reservation_dose"]=="once"].sum()),
        p_twice_mc           = ("gp_prob", lambda x:
            x[cluster_gp_res_v11.loc[x.index,"reservation_dose"]=="twice"].sum()),
    )
)

treat_probs_v11 = all_clusters.merge(cluster_known_v11, on="DHSCLUST", how="left")
for col in ["p_known_treatment_mc","p_never_mc","p_once_mc","p_twice_mc"]:
    treat_probs_v11[col] = treat_probs_v11[col].fillna(0.0)
treat_probs_v11["p_unknown_treatment_mc"] = (
    1 - treat_probs_v11["p_known_treatment_mc"]
).clip(lower=0)
treat_probs_v11["p_any_reserved_mc"] = (
    treat_probs_v11["p_once_mc"] + treat_probs_v11["p_twice_mc"]
)
treat_probs_v11["expected_dose_mc"] = (
    treat_probs_v11["p_once_mc"] + 2 * treat_probs_v11["p_twice_mc"]
)
treat_probs_v11["treatment_certainty_mc"] = treat_probs_v11[[
    "p_never_mc","p_once_mc","p_twice_mc"
]].max(axis=1)

primary_dose_v11 = (
    cluster_gp_res_v11[cluster_gp_res_v11["reservation_dose"].notna()]
    .sort_values(["DHSCLUST","gp_prob"], ascending=[True,False])
    .groupby("DHSCLUST", as_index=False).first()
    [["DHSCLUST","gp_lgd_code","reservation_dose"]]
    .rename(columns={
        "gp_lgd_code":      "primary_gp_matched",
        "reservation_dose": "primary_gp_dose"
    })
)
treat_probs_v11 = treat_probs_v11.merge(primary_dose_v11, on="DHSCLUST", how="left")
treat_probs_v11["primary_gp_norm"] = treat_probs_v11["primary_gp"].apply(normalize_lgd_code)
treat_probs_v11["primary_gp_has_history"] = treat_probs_v11["primary_gp_norm"].isin(
    set(gp_res_history_final_v11["gp_lgd_code"])
)

working_v11 = treat_probs_v11[treat_probs_v11["primary_gp_has_history"]].copy()
final_treat_probs = working_v11[[
    "DHSCLUST","DHSREGNA","p_never_mc","p_once_mc","p_twice_mc",
    "p_any_reserved_mc","expected_dose_mc","p_known_treatment_mc",
    "p_unknown_treatment_mc","treatment_certainty_mc",
    "primary_gp","primary_gp_prob","primary_gp_dose","n_gps_hit"
]].copy()

print(f"Working clusters:       {len(working_v11)}")
print(f"Fully linked (>=99.9%): {(working_v11['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=75% linked:           {(working_v11['p_known_treatment_mc'] >= 0.75).sum()}")
print(f"Mean p_known:           {working_v11['p_known_treatment_mc'].mean():.3f}")

# Update and save
gp_res_history_final = gp_res_history_final_v11
cluster_gp_res       = cluster_gp_res_v11
treat_probs_new      = treat_probs_v11
working_clusters_new = working_v11

gp_res_history_final.to_csv(OUTPUT_DIR / "gp_reservation_history.csv", index=False)
cluster_gp_res.to_csv(OUTPUT_DIR / "cluster_gp_res_long.csv", index=False)
final_treat_probs.to_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv", index=False)

print(f"\nSaved. Total GP histories: {len(gp_res_history_final)}")
print(f"\nFINAL PIPELINE SUMMARY")
print(f"{'='*50}")
print(f"Match stage breakdown:")
print(gp_res_history_final["match_stage"].value_counts())

Overlap (should be 0): 0
Working clusters:       732
Fully linked (>=99.9%): 132
>=75% linked:           474
Mean p_known:           0.809

Saved. Total GP histories: 6495

FINAL PIPELINE SUMMARY
Match stage breakdown:
match_stage
A_exact_gp_district_samiti                2277
F2_fuzzy_sarpanch_both_years_alias_fix    1733
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide                 185
D_district_only_unique_dose                177
E_fuzzy_gp_name_district                    70
G_fuzzy_transliteration_single_blocker      15
G2_fuzzy_single_blocker_alias_fix           15
H_manual_verified_transliteration            5
J_thorough_cross_district_search             5
K_low_threshold_single_blocker               5
L_partial_match_both_never                   1
Name: count, dtype: int64


In [75]:
# Quick state check
print(f"GP reservation histories: {len(gp_res_history_final)}")
print(f"Working clusters:         {len(final_treat_probs)}")
print(f"Fully linked (>=99.9%):   {(final_treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")
print(f">=75% linked:             {(final_treat_probs['p_known_treatment_mc'] >= 0.75).sum()}")

# Check if files on disk match current in-memory state
import pandas as pd
from pathlib import Path
OUTPUT_DIR = Path("../outputs/final_rj_sample")

treat_on_disk = pd.read_csv(OUTPUT_DIR / "cluster_treatment_probs_rj.csv")
print(f"\nOn-disk cluster_treatment_probs_rj.csv:")
print(f"  Rows:                   {len(treat_on_disk)}")
print(f"  Fully linked (>=99.9%): {(treat_on_disk['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  >=75% linked:           {(treat_on_disk['p_known_treatment_mc'] >= 0.75).sum()}")

gp_hist_on_disk = pd.read_csv(OUTPUT_DIR / "gp_reservation_history.csv")
print(f"\nOn-disk gp_reservation_history.csv:")
print(f"  GP histories:           {len(gp_hist_on_disk)}")
print(f"  Match stages:")
print(gp_hist_on_disk["match_stage"].value_counts())

GP reservation histories: 6495
Working clusters:         732
Fully linked (>=99.9%):   132
>=75% linked:             474

On-disk cluster_treatment_probs_rj.csv:
  Rows:                   732
  Fully linked (>=99.9%): 132
  >=75% linked:           474

On-disk gp_reservation_history.csv:
  GP histories:           6495
  Match stages:
match_stage
A_exact_gp_district_samiti                2277
F2_fuzzy_sarpanch_both_years_alias_fix    1733
B_exact_gp_district                       1514
F_fuzzy_sarpanch_both_years                493
C_unique_gp_name_statewide                 185
D_district_only_unique_dose                177
E_fuzzy_gp_name_district                    70
G_fuzzy_transliteration_single_blocker      15
G2_fuzzy_single_blocker_alias_fix           15
H_manual_verified_transliteration            5
J_thorough_cross_district_search             5
K_low_threshold_single_blocker               5
L_partial_match_both_never                   1
Name: count, dtype: int64
